# Supply Chain System — Checkpoint Summary

## What We Built

A logistics layer for a hex-grid strategy game where **food is the fundamental constraint**. Pieces don't just fight — they eat, starve, and die. Every military action requires a working supply chain behind it.

### Core Mechanics

- **Pawns** harvest food from terrain tiles (yield = `harvest_strength × food_tier`)
- **Bishops** carry food along routes between settlements and consumers
- **Queens, Knights, Rooks** consume food each turn — they starve without supply lines
- **Food tiers** are terrain-driven: coastal plains are fertile, mountains are barren
- Pieces have `diet`, `food_capacity`, and `harvest_strength` configured via `FoodProfile`

### The Five Rules

1. Every piece eats `diet` food per turn
2. Shortfall × 5 = starvation damage per turn
3. Pawns harvest `harvest_strength × tile_tier` food when executing HARVEST
4. GIVE transfers surplus food to visible friendly pieces
5. Bishops move along planned routes; their effective delivery = `capacity - (2 × travel_turns × diet)`

### Key Classes

**`FoodProfile`** — per-piece-type food configuration (diet, capacity, harvest_strength).
Applied via `FoodProfile.apply(piece)` in `Piece.__post_init__`.

**`FoodSimulator`** — runs the turn loop (eat → starve → harvest → give → move),
records events per piece, renders `show(sim)` as alive/health/food line charts.

**`SupplyPlanner`** — the math engine for supply logistics:
- `net_delivery(path_cost)` — food remaining after a bishop eats its own cargo
- `throughput(cost, n_bishops)` — sustained food/turn at a given distance
- `bishops_needed(cost, diet)` — minimum couriers to sustain a consumer
- `max_range()` — furthest a bishop can go and still deliver food
- `supply_range_map()` — Dijkstra reachability from settlement
- `find_target()` — locate elevated, reachable positions for consumers
- `create_supply_line()` — place and route bishops automatically
- `provision_consumer()` — set starting food based on delivery wait time
- `show(planner)` — throughput curves + bishops-needed heatmap

**`ColonyOptimizer`** — MILP-based resource allocation:
- Given N consumers at various distances with priorities (queen=10, knight=3, etc.)
- Allocates limited bishops and harvest budget to maximize total priority fed
- Constraints: total bishops, throughput ≥ diet, harvest ≥ total demand
- `show(optimizer)` — bishop allocation bars, throughput vs diet, harvest budget pie

### Key Insight: The Bishop Fuel Problem

A bishop carrying food eats its own cargo in transit. The **net delivery** formula:



### Making the game
This is a great design question. The answer I'd push toward: the player decides where and what priority, the system shows them what it costs, and they confirm.

The decision flow:

1. Explore — Player sees a supply range heatmap overlaid on the board. Color encodes "how expensive is it to sustain a piece here" — green (1 bishop), yellow (2), orange (3), red (unreachable). This is supply_range_map() rendered on the hex grid. The player immediately sees the shape of their logistical reach.

2. Designate — Player clicks hexes to mark them as expansion targets. Maybe they want a knight outpost on that hilltop, or the queen pushed forward to a defensible position. Each click adds a pin to the map.

3. Preview — System runs ColonyOptimizer and shows the result before committing. "You want 3 targets? Here's the plan: queen gets 2 bishops, knight gets 1, rook is ❌ too far — we'd need 2 more pawns." The player sees the harvest budget pie and the tradeoffs before any pieces move.

4. Commit or adjust — Player can reprioritize (drag the rook closer, drop a target, add pawns). Each change re-runs the optimizer and updates the preview. When they're happy, they commit and the bishops get their marching orders.

5. Micromanage (optional) — The supply lines run on autopilot via patrol instructions. But the player can manually reroute a bishop, override a patrol, pull a supply line to react to threats. This is the "cool and unique" part — you don't have to touch it, but you can.

For the AI opponent, the same code runs but steps 2-4 collapse into:
Copied!

# AI picks targets using its own strategy (threat assessment, territory, etc.)
targets = ai.pick_expansion_targets(board, planner)

# Same optimizer the player uses
consumers = opt.analyze_consumers(targets)
result = opt.optimize(consumers)

# Execute directly
for a in result['assignments']:
    if a['fed']:
        planner.create_supply_line(a['consumer'].hex_idx, board,
                                   n_bishops=a['bishops'])

The AI uses the exact same logistics constraints — no cheating on food. It just skips the UI.

What the player doesn't decide (and shouldn't have to):

    Exact number of bishops per route (optimizer)
    Patrol route instructions (pathfinder)
    Starting food amounts (provisioner)
    When to harvest vs give (patrol loop)

What the player does decide (the fun part):

    Where to expand
    What to prioritize when resources are scarce
    When to cut losses and abandon a position
    Whether to micromanage a critical supply line

The heatmap is the key design insight — it turns complex math into spatial intuition. A player looks at it and thinks "I can reach there cheaply but there is expensive" without ever seeing a number. That's good game design.

### Visualizing where to go

```python
# Just the supply heatmap on cream
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement),
    terrain=myTerr,
)

# With rivers too
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement, field='bishops_needed'),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)

# Compare different fields
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement, field='net_delivery'),
    terrain=myTerr,
)

# Reuse an existing planner (avoids recomputing)
planner = SupplyPlanner(myTerr, settlement=best_settlement)
planner.compute_supply_fields()

TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement, planner=planner),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)
```


Good thinking. Here's where I'd draw the line between "build now" and "build later":

### Flesh out now (supply-only)

**1. `_reoptimize` implementation** — this is pure supply math, no combat needed:

```python
@patch
def _reoptimize(self: BattlePlan):
    """Re-run optimizer respecting player overrides."""
    opt = self.optimizer
    
    # Filter out vetoed consumers
    active = [a['consumer'] for a in self.assignments
              if a['consumer'].hex_idx not in self.vetoed]
    
    # Apply locked constraints — force f_i = 1 for locked consumers
    # Apply boosted — set minimum bishops per route
    result = opt.optimize(active,
                          locked=self.locked,
                          min_bishops=self.boosted)
    
    self.assignments = result['assignments']
    self.harvest_budget = result['harvest_available']
    return self
```

**2. `__ft__` for BattlePlan** — the interactive preview card. This is what makes it fun:

```python
@patch
def __ft__(self: BattlePlan):
    """Interactive plan preview with veto/lock/boost controls."""
    rows = []
    for a in sorted(self.assignments,
                    key=lambda x: -x['consumer'].priority):
        c = a['consumer']
        name = getattr(c.piece, 'name', f'hex {c.hex_idx}')
        status = "✅" if a['fed'] else "❌"
        vetoed = c.hex_idx in self.vetoed
        locked = c.hex_idx in self.locked
        
        icon = "🚫" if vetoed else ("🔒" if locked else status)
        
        rows.append(Tr(
            Td(icon),
            Td(Strong(name)),
            Td(f"{c.piece.piece_type.name}"),
            Td(f"{c.diet:.1f}"),
            Td(f"{c.path_cost:.1f}"),
            Td(f"{a['bishops']}"),
            Td(f"{a['throughput']:.2f}"),
        ))
    
    table = Table(
        Thead(Tr(*[Th(h) for h in
            ["", "Name", "Type", "Diet", "Cost", "Bishops", "Throughput"]])),
        Tbody(*rows),
        cls="text-sm"
    )
    
    fed = sum(1 for a in self.assignments if a['fed'])
    total = len(self.assignments)
    
    return Div(
        P(f"🗺 Battle Plan — {fed}/{total} fed, "
          f"{sum(a['bishops'] for a in self.assignments)} bishops committed",
          cls="font-bold"),
        table,
        P(f"🌾 Harvest budget: {self.harvest_budget:.0f} food/turn",
          cls="text-xs opacity-60"),
        cls="space-y-2"
    )
```

**3. King protection priority** — this is just a number tweak, no combat:

```python
# King gets infinite-ish priority — optimizer will NEVER cut the king's supply
PieceType.KING: 20   # already highest, but could be 100 to guarantee it
```

### Think about now, build later

The piece roles you described create a beautiful **rock-paper-scissors** with supply:

| Piece | Speed | Diet | Combat Role | Supply Role |
|-------|-------|------|-------------|-------------|
| **King** | Slow | 4.0 | Must survive | Consumes heavily |
| **Queen** | Medium | 3.0 | Power piece | Consumes heavily |
| **Rook** | Slow | 2.0 | Defends supply lines | Static guard |
| **Knight** | Fast | 1.5 | Raids enemy supply | Light footprint |
| **Bishop** | Medium | 1.5 | None | Courier |
| **Pawn** | Slow | 0.5 | None | Harvester |

The strategic loop becomes:
- **Knights raid enemy supply lines** (fast, cheap to sustain, can hit and run)
- **Rooks defend your supply lines** (slow but tough, stationed along routes)
- **Queen projects power** but needs 2-3 bishops dedicated to her
- **King stays near the settlement** (expensive to supply far away, losing him = game over)

The fun tension: **every rook you place on defense is a rook not attacking, and every knight raiding is a knight not defending.** The supply chain makes this tradeoff real — you can't afford to deploy everything everywhere.


```python

# King gets infinite-ish priority — optimizer will NEVER cut the king's supply
PieceType.KING: 20   # already highest, but could be 100 to guarantee it


# Full battle view: terrain + supply range + routes + piece movements
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement),
    BattlePlanOverlay(battle_plan),
    BattlePieceOverlay(battle_plan, num_turns=10),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)

# Just the piece movements without supply heatmap
TerrainDisplay(
    CreamOverlay(),
    BattlePieceOverlay(battle_plan, num_turns=8, show_pawns=True),
    terrain=myTerr,
)

# Bishops only — see the supply runs
TerrainDisplay(
    CreamOverlay(),
    BattlePieceOverlay(battle_plan, show_consumers=False, show_pawns=False),
    terrain=myTerr,
)
```


# Maneuvers & Formations — API Reference

## Core Primitives

### TroopPath

A resolved hex route with cached metadata. Returned by all pathfinding functions.

```python
path = piece.pathfind(target_hex, grid, elevations, countries=None)

path.hexes       # list[int] — ordered hex indices
path.cost        # float — total movement cost
path.start       # first hex
path.end         # last hex
path.as_set      # set[int] — cached for fast membership checks
path.to_rules()  # list[int] — convert to Instruction values

bool(path)       # False if empty or single-hex (unreachable)
len(path)        # number of hexes in route
```

**Movement costs:** flat/downhill = 1.0, uphill = 2.0. Water and invalid hexes are impassable.

### Pathfinding Variants

```python
# Basic A→B shortest path (Dijkstra)
path = piece.pathfind(target_hex, grid, elevations, countries)

# Nearest hex with food_tier >= threshold
path = piece.pathfind_to_food(grid, elevations, food_tiers, min_tier=3, countries)

# Path AWAY from a threat — prefers high ground in the opposite hemisphere
path = retreat_path(piece, threat_hex, grid, elevations,
                    distance=5, countries=None, elevation_weight=0.5)

# Convenience: pathfind + convert to InstructionList in one call
il = piece.pathfind_to_rules(target, grid, elevations, countries)
```

### InstructionList

The rule program for a piece. Each rule is an `Instruction` enum value.

```python
il = InstructionList(rules=[...], cursor=0, patrol=False)

il.insert_rules(new_rules)   # insert at cursor, advance cursor
il.delete_at(idx)             # remove one rule, clamp cursor
il.set_cursor(idx)            # move cursor
il.toggle_patrol()            # flip loop mode
il.est_turns(speed)           # lower-bound turn count for sync
il.compact(piece_id)          # FastHTML badge strip
```

**Patrol mode:** when `patrol=True`, cursor wraps to 0 at end of rules.
When `patrol=False`, piece stops after the last instruction.

### Instruction Budget Rules

Per turn, each piece gets `move_strength` budget points:

| Instruction | Cost | Notes |
|-------------|------|-------|
| FORWARD | 1 (flat/downhill) or 2 (uphill) | Into water/edge: auto-rotate left free, up to 5×. Into occupied hex: forced PAUSE cost 1. |
| ROT_L, ROT_R | 0 (free) | |
| PAUSE | 1 | |
| DEFEND | 1 | |
| HARVEST | 1 | |
| GIVE | 1 | |
| SETTLE | 1 | |

When budget runs out mid-instruction, the turn ends. If the last instruction
was non-movement (PAUSE/DEFEND/HARVEST/GIVE/SETTLE), remaining budget is
idled out as implicit PAUSEs.

---

## Formations

### FormationType

```python
class FormationType(Enum):
    LINE   = "line"    # row abreast, one step behind leader
    WEDGE  = "wedge"   # V-shape, leader at point, wings fan back
    BOX    = "box"     # ring-1 around leader (overflow to ring-2 if >6)
    COLUMN = "column"  # single file directly behind leader
```

**Choosing a formation:**
- **COLUMN** — narrow passages, rivers, mountain paths
- **LINE** — broad front for harvesting or sight coverage
- **WEDGE** — aggressive advance, concentrates force at point of contact
- **BOX** — protect a VIP (queen, settler), all-around defense

### form_up

Assemble followers around the leader's **current position**.

```python
orders = squad.form_up(
    FormationType.WEDGE, grid, elevations,
    leader=None,          # auto-picks highest rank if omitted
    countries=None,
    food_tiers=None,      # enables harvest padding during sync
    sync=True,            # pad so all arrive same turn
)
# orders: {piece_id: InstructionList} — leader NOT included
```

**Important:** `form_up` does not move the leader. It positions followers
relative to where the leader already is. Compose with `march_to` or
`pathfind_to_rules` for the leader's own movement.

### march_to

Move entire squad to a destination, arriving in formation.

```python
orders = squad.march_to(
    destination_hex,
    FormationType.LINE, grid, elevations,
    leader=None,          # auto-picks highest rank
    countries=None,
    food_tiers=None,
    sync=True,
)
# orders: {piece_id: InstructionList} — leader IS included
```

Formation offsets are computed relative to **destination + leader's arrival
facing**, so the formation assembles correctly on arrival. Uses Hungarian
matching to optimally assign followers to slots.

---

## Tactical Maneuvers

### surround

Position squad members around a target hex in a ring.

```python
orders = squad.surround(
    target_hex, grid, elevations,
    mode=SurroundMode.GUARD,  # GUARD = face outward, FEED = face inward
    max_ring=3,
    countries=None,
    elevation_mult=0.005,     # sight bonus per elevation unit
)
```

**Modes:**
- `GUARD` — defensive perimeter; sight cones scan outward for threats
- `FEED` — supply hub; GIVE cones converge on center piece

Uses Hungarian matching. Respects `effective_sight` — pieces won't be placed
farther than they can see.

### retreat

Scatter squad away from a threat. Each piece picks its own best retreat hex
independently (no formation maintained during withdrawal).

```python
orders = squad.retreat(
    threat_hex, grid, elevations,
    distance=5,               # max rings to search
    countries=None,
    food_tiers=None,
    sync=True,
    elevation_weight=0.5,     # preference for high ground
)
```

All pieces face **away** from the threat on arrival. Typical follow-up:

```python
# After executing retreat orders...
regroup = squad.form_up(FormationType.LINE, grid, elevations)
```

### retreat_path (standalone)

Single-piece retreat. Returns a `TroopPath`.

```python
path = retreat_path(piece, threat_hex, grid, elevations,
                    distance=5, countries=None, elevation_weight=0.5)
```

Runs one Dijkstra flood-fill within `distance` rings. Scores candidates by:
```
score = elevation × elevation_weight − path_cost
```
Only considers hexes in the **away hemisphere** (cube dot product < 0 with
the toward-direction). Falls back to cheapest reachable hex if cornered.

---

## Synchronization

### sync_paths

Pad instruction lists so all pieces finish on the same turn.

```python
synced = sync_paths(
    pieces,                   # all pieces referenced by orders
    orders,                   # {piece_id: InstructionList}
    end_hexes=None,           # {piece_id: final_hex} for harvest check
    food_tiers=None,          # enables productive padding
    harvest_threshold=3,      # min food tier for harvest padding
    min_turns=0,              # floor (use leader's turns as floor)
)
```

**Padding priority:**
1. If `food_tiers[end_hex] >= threshold` → `HARVEST, GIVE, HARVEST, GIVE...`
2. Otherwise → `PAUSE, PAUSE, PAUSE...`

Called automatically by `form_up`, `march_to`, `surround`, and `retreat`
when `sync=True`.

---

## Helper Functions

```python
# Direction from one hex toward another (returns 0–5)
facing = _facing_toward(grid, from_idx, to_idx)

# Opposite direction (180°)
facing = _facing_away(grid, from_idx, to_idx)

# Effective sight with elevation bonus
sight = piece.effective_sight(elevations, elevation_mult=0.005)

# Formation slot offsets in cube coordinates
offsets = _formation_offsets(FormationType.WEDGE, n_followers, leader_facing)
```

---

## Recommended Combos

### Escort a Queen to a Settlement Site

```python
# Queen paths to destination
queen.instructions = queen.pathfind_to_rules(site, grid, elevations)

# Escort forms up around queen, then marches together
orders = escort_squad.march_to(
    site, FormationType.BOX, grid, elevations,
    leader=queen, food_tiers=food_tiers,
)
for p in escort_squad.alive:
    if p.id in orders:
        p.instructions = orders[p.id]
```

### Defend a Settlement

```python
orders = garrison.surround(
    settlement.location, grid, elevations,
    mode=SurroundMode.GUARD, max_ring=2,
)
```

### Supply Chain: Harvest + Deliver

```python
# Pawn finds food, paths there, harvests, returns to queen
food_path = pawn.pathfind_to_food(grid, elevations, food_tiers)
return_path = pawn.pathfind(queen.location, grid, elevations,
                            source=food_path.end)

rules = (food_path.to_rules(grid, pawn.facing)
         + [Instruction.HARVEST.value] * 3
         + return_path.to_rules(grid, _facing_toward(grid, food_path.end, queen.location))
         + [Instruction.GIVE.value])

pawn.instructions = InstructionList(rules, patrol=True)
```

### Retreat + Regroup

```python
# Phase 1: scatter away from threat
retreat_orders = squad.retreat(enemy_hex, grid, elevations, distance=6)
for p in squad.alive:
    if p.id in retreat_orders:
        p.instructions = retreat_orders[p.id]

# Phase 2 (after retreat executes): regroup in defensive line
regroup = squad.form_up(FormationType.LINE, grid, elevations)
```

### Pincer: Two Squads from Opposite Sides

```python
# Left wing approaches from the west
left_orders = left_squad.march_to(
    west_of_target, FormationType.LINE, grid, elevations)

# Right wing approaches from the east
right_orders = right_squad.march_to(
    east_of_target, FormationType.LINE, grid, elevations)

# Sync both wings to arrive simultaneously
all_pieces = left_squad.alive + right_squad.alive
all_orders = {**left_orders, **right_orders}
synced = sync_paths(all_pieces, all_orders)
```

### Scout Patrol (Loop)

```python
# Knight patrols a triangle: forward, turn, forward, turn...
knight.instructions = InstructionList([
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
], patrol=True)  # loops forever
```

---

## Design Notes

**Hungarian matching** (`scipy.optimize.linear_sum_assignment`) is used
everywhere pieces need optimal assignment to positions: surround, form_up,
march_to. Cost matrix is `pathfind_cost[piece, slot]`. Complexity O(n³)
but n is squad size (typically 3–8), so it's instant.

**Occupied set is deliberately omitted** from plan overlays. Two pieces
can path through the same hex in preview. This is intentional — plans are
pre-execution previews, and collision resolution happens during `simulate()`.

**`form_up` vs `march_to`:**
- `form_up` — positions followers around leader's CURRENT hex. Leader not included in orders.
- `march_to` — positions everyone around a DESTINATION hex. Leader IS included in orders.

**`sync_paths` is composable.** Any function that returns
`dict[str, InstructionList]` can be synced. You can also sync orders from
different sources by merging the dicts before calling `sync_paths`.

**`retreat` breaks formation intentionally.** Each piece finds its own
best escape hex. Regroup afterward with `form_up`. Trying to maintain
formation during retreat is expensive and usually tactically wrong.

**Elevation matters in two places:**
1. Movement cost: uphill = 2×
2. Sight range: `effective_sight = base_sight + elevation × 0.005`

**Food integration:** All sync functions accept `food_tiers`. When a
piece's end hex has good food, slack turns become HARVEST+GIVE cycles
instead of idle PAUSEs. This means early-arriving units productively
farm while waiting for slower friends.



In [ ]:
#| default_exp game/food

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import FlowOverlay, CreamOverlay, RiverOverlay, ClimateOverlay, OverlayContext
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns
from HexMagic.plot.cube import HexPosition


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc

import httpx
import random
import pandas as pd
import threading
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import math
from dataclasses import dataclass, field
from functools import cached_property
#| export
import heapq
import io
from monsterui.all import *
from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds

In [ ]:
#| export

from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging

from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad
from HexMagic.game.data import RANK_ORDER 

from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from HexMagic.game.piece import SquadPlanOverlay, PieceList, FormationType, SurroundMode, TroopPath, _facing_toward
from HexMagic.game.piece import GroupedPieceList, SquadPlanOverlay, piece_plan_overlay, SquadSymbolOverlay #, SquadVisionOverlay
from HexMagic.game.piece import FormationType


from HexMagic.game.piece import pieces_center, StatBar

from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds



from enum import Enum

In [ ]:
#| export
STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

In [ ]:
#| export
@patch
def __ft__(self: Squad, grid: HexGrid = None, list_pieces: bool = False):
    """Compact squad dashboard — header with icon + name, box plots by piece type."""
    alive = self.alive
    all_pieces = self.pieces

    # ── Header: icon (left) + text (center) + flag (right) ──
    icon_html = None
    flag_builder = None
    if self.flag is not None:
        animal = self.animal or self.flag.animal_name()
        icon_svg = self.flag.animal_svg(size='large', animal=animal,
                                         piece_id=f"squad_icon_{self.id}")
        r = int(22.5 * 3.5 * 1.3) + 5
        svg_size = r * 2 + 8
        icon_html = NotStr(
            f'<svg xmlns="http://www.w3.org/2000/svg" '
            f'width="{svg_size}" height="{svg_size}" '
            f'viewBox="0 0 {svg_size} {svg_size}">{icon_svg}</svg>'
        )
        flag_builder = self.flag.flag_banner(width=90, height=60, wavy=True, pole=True)

    squad_label = self.name or f"Squad #{self.id}"
    total_alive = len(alive)
    total_all = len(all_pieces)

    text_parts = [Strong(squad_label, style="font-size:1.2em")]
    if self.countryName is not None:
        text_parts.append(P(self.countryName, cls="text-sm opacity-70"))
    text_parts.append(P(f"🗡 {total_alive}/{total_all} alive", cls="text-sm"))

    header_parts = []
    if icon_html:     header_parts.append(Div(icon_html, cls="shrink-0"))
    header_parts.append(Div(*text_parts, cls="space-y-1 flex-1 text-center"))
    if flag_builder:  header_parts.append(Div(NotStr(flag_builder.xml()), cls="shrink-0"))

    header = Div(*header_parts, cls="flex items-center gap-4 mb-3")

    # ── Early out if no pieces — still show header ──
    if not all_pieces:
        return Div(header, P("No pieces assigned", cls="text-sm opacity-50 ml-2"),
                   id=f"squad-{self.id}", cls="space-y-2")

    # ── Group by piece type (ranked order) ──
    types_present = sorted(set(p.piece_type for p in all_pieces),
                           key=lambda t: RANK_ORDER.get(t, 99))
    type_labels = [t.name.title() for t in types_present]

    health_data, food_data, turns_data, distance_data = [], [], [], []
    alive_counts, total_counts = [], []

    centroid = None
    if grid is not None:
        locs = [p.location for p in alive
                if p.location is not None and p.location >= 0]
        if len(locs) >= 2:
            positions = [grid.index_to_hexposition(loc) for loc in locs]
            cq = sum(p.q for p in positions) / len(positions)
            cr = sum(p.r for p in positions) / len(positions)
            cs = sum(p.s for p in positions) / len(positions)
            centroid = (cq, cr, cs)

    for pt in types_present:
        members = [p for p in all_pieces if p.piece_type == pt]
        alive_m = [p for p in members if p.health > 0]
        alive_counts.append(len(alive_m))
        total_counts.append(len(members))
        health_data.append([p.health for p in members])
        food_data.append([p.food for p in members])
        turns_data.append([p.food / max(p.diet, 0.01) for p in members])

        if centroid is not None:
            dists = []
            for p in members:
                if p.location is not None and p.location >= 0:
                    pos = grid.index_to_hexposition(p.location)
                    d = (abs(pos.q - centroid[0])
                       + abs(pos.r - centroid[1])
                       + abs(pos.s - centroid[2])) / 2
                    dists.append(d)
                else:
                    dists.append(float('nan'))
            distance_data.append(dists)

    # ── Build figure ──
    has_dist = centroid is not None and distance_data
    n_panels = 4 if has_dist else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(3.2 * n_panels, 3.0))
    if n_panels == 1: axes = [axes]

    panels = [
        ("Health",            health_data,   "#e74c3c"),
        ("Food",              food_data,     "#27ae60"),
        ("Food (turns left)", turns_data,    "#f39c12"),
    ]
    if has_dist:
        panels.append(("Distance from center", distance_data, "#3498db"))

    for ax, (title, data, color) in zip(axes, panels):
        if not data or all(len(d) == 0 for d in data):
            ax.set_visible(False)
            continue
        bp = ax.boxplot(data, tick_labels=type_labels, patch_artist=True,
                        medianprops=dict(color='#222', linewidth=1.5),
                        whiskerprops=dict(color='#888'),
                        capprops=dict(color='#888'))
        for patch in bp['boxes']:
            patch.set_facecolor(color); patch.set_alpha(0.55)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', rotation=35, labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
        ax.grid(axis='y', alpha=0.3, linewidth=0.5)
        ymax = ax.get_ylim()[1]
        for i, (a, t) in enumerate(zip(alive_counts, total_counts)):
            ax.text(i + 1, ymax * 1.02, f"{a}/{t}",
                    ha='center', va='bottom', fontsize=7, color='#555',
                    fontstyle='italic')

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    avg_hp = np.mean([p.health for p in alive]) if alive else 0
    min_hp = min((p.health for p in alive), default=0)
    avg_turns = np.mean([p.food / max(p.diet, 0.01) for p in alive]) if alive else 0

    summary = P(
        f"❤ avg {avg_hp:.0f} (min {min_hp:.0f}) · "
        f"🌾 {avg_turns:.1f} avg food turns",
        cls="text-xs opacity-60 text-center mt-1"
    )

    parts = [header, NotStr(buf.getvalue()), summary]
    if list_pieces:
        parts.append(Hr())
        parts.append(PieceList(all_pieces))

    return Div(*parts, id=f"squad-{self.id}", cls="space-y-2")


In [ ]:
@patch
def food_gauge(piece:Piece, w=220, h=32, bar_h=14, bar_y=4):
    """Inline SVG food-turns-remaining gauge for a Piece card."""
    turns_left = piece.food / piece.diet if piece.diet > 0 else float('inf')
    max_turns = piece.food_capacity / piece.diet if piece.diet > 0 else 1
    reserve = FOOD_RESERVE_TURNS

    frac = min(turns_left / max_turns, 1.0) if max_turns > 0 else 1.0
    reserve_frac = min(reserve / max_turns, 1.0) if max_turns > 0 else 0

    # ── Styles ──
    canvas = SVGBuilder()
    canvas.width, canvas.height = w, h

    track = StyleCSS("fg_track", fill="#eee", stroke="none", stroke_width=0)

    if turns_left >= reserve:
        bar_color = StyleCSS.lerp_color('#e74c3c', '#27ae60', min(frac, 1.0))
    elif turns_left >= reserve * 0.5:
        bar_color = '#f39c12'
    else:
        bar_color = '#e74c3c'

    fill = StyleCSS("fg_fill", fill=bar_color, stroke="none", opacity=0.8)
    tick = StyleCSS("fg_tick", stroke="#e67e22", fill="none",
                    stroke_width=2, stroke_dasharray="2,1")
    lbl  = StyleCSS("fg_lbl", fill="#555", stroke="none", stroke_width=0)
    rlbl = StyleCSS("fg_rlbl", fill="#e67e22", stroke="none", stroke_width=0)

    for s in (track, fill, tick, lbl, rlbl):
        canvas.add_style(s)

    # ── Track + fill bar ──
    body  = f'<rect x="0" y="{bar_y}" width="{w}" height="{bar_h}" rx="3" class="{track.name}"/>'
    body += f'<rect x="0" y="{bar_y}" width="{w * frac:.1f}" height="{bar_h}" rx="3" class="{fill.name}"/>'

    # ── Reserve tick ──
    body += MapPath([
        MapCord(w * reserve_frac, bar_y - 1),
        MapCord(w * reserve_frac, bar_y + bar_h + 1),
    ], tick).drawPolygon()

    # ── Labels ──
    txt_y = bar_y + bar_h + 14
    tl_str = f"{turns_left:.1f}" if turns_left < 1000 else "∞"
    body += (f'<text x="4" y="{txt_y}" font-size="9" '
             f'font-family="sans-serif" class="{lbl.name}">'
             f'🌾 {tl_str} turns left</text>')
    body += (f'<text x="{w - 4}" y="{txt_y}" font-size="9" '
             f'font-family="sans-serif" text-anchor="end" class="{rlbl.name}">'
             f'reserve={reserve}</text>')

    canvas.adjust("gauge", body)
    return NotStr(canvas.xml())


In [ ]:
#| export
@patch
def __ft__(self: Piece):
    """Narrow vertical layout for w-64 sidebar panel."""
    ptype = self.piece_type.name.title()
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # ── Flag-patterned piece icon (80px) ──
    icon_size = 80
    icon_el = P(self.piece_type.icon, cls="text-5xl")

    if self.flag:
        builder = SVGBuilder()
        builder.width = icon_size
        builder.height = icon_size
        center = MapCord(icon_size / 2, icon_size / 2)
        scale = (icon_size * 0.55) / 22.5
        svg = self.flag.draw_piece(
            self.piece_type, center, builder,
            scale=scale, size='large',
            piece_id=f"piece_icon_{self.id[:8]}",
            layer="piece",
        )
        if svg:
            icon_el = NotStr(builder.xml())

    # ── Name / type / birth ──
    display_name = self.name if self.name else ptype
    subtitle = ptype
    if self.name:
        subtitle = f"{ptype} • b.{self.birth_year}" if self.birth_year else ptype

    # ── Stats row ──
    stats = [("⚔", self.attack_strength, "Atk"),
             ("🏃", self.move_strength,   "Spd"),
             ("🌾", self.harvest_strength, "Harv"),
             ("👁", self.sight,            "Sight")]
    stat_row = DivFullySpaced(
        *[Div(
            P(f"{ic} {v}", cls="text-sm font-bold text-center"),
            P(label, cls="text-[10px] text-center opacity-50"),
          ) for ic, v, label in stats],
    )

    # ── Rules — compact badge strip ──
    rules_el = self.instructions.compact(self.flag, self.id)

    return Div(
        DivCentered(icon_el, cls="pt-2"),
        DivCentered(
            Strong(display_name),
            P(subtitle, cls=TextPresets.muted_sm),
        ),
        P(f"Facing {facing_arrow}", cls="text-center text-xs opacity-60"),

        StatBar("Health", self.health, self.max_health),
        StatBar("Food",   self.food,   self.food_capacity),
        DivCentered(self.food_gauge()),

        stat_row,

        Divider(),
        rules_el,

        Divider(),
        DivVStacked(
            Button("Edit Rules",
                   hx_get=f"/piece_rules/{self.id}",
                   hx_target="#right-panel", hx_swap="innerHTML",
                   cls=ButtonT.primary + " " + ButtonT.sm + " w-full"),
            Button("Show Plan",
                   hx_get=f"/piece_plan/{self.id}",
                   hx_target="#map", hx_swap="innerHTML",
                   cls=ButtonT.sm + " w-full"),
            cls="space-y-1",
        ),

        id=f"piece-card-{self.id[:8]}",
        cls="space-y-2 p-3",
    )


In [ ]:
@patch
def place_squad_spiral(settlement:Settlement,squad,  terrain, max_ring=10):
    """Place squad pieces in spiral around settlement, skipping water and occupied hexes.
    
    Args:
        squad: Squad with pieces to place
        settlement: Settlement to spiral around  
        terrain: Terrain object (has hexGrid, elevations)
        max_ring: Maximum spiral radius to search
        
    Returns:
        Number of pieces successfully placed
    """
    grid = terrain.hexGrid
    center = settlement.location
    
    # Collect occupied hexes: settlement location + all citizen locations
    occupied = {center}
    for citizen in settlement.citizens:
        if citizen.location is not None:
            occupied.add(citizen.location)
    
    # Generate spiral outward from center
    spiral = HexPosition.origin().spiral(max_ring)
    
    placed = 0
    for hp in spiral[1:]:  # skip center (settlement itself)
        if placed >= len(squad.pieces):
            break
        
        idx = grid.hexposition_to_index(hp, origin_index=center)
        if idx < 0:
            continue
        
        # Skip water (elevation <= 0)
        if terrain.elevations[idx] <= 0:
            continue
        
        # Skip occupied by settlement citizens
        if idx in occupied:
            continue
        
        squad.pieces[placed].location = idx
        occupied.add(idx)  # prevent stacking within the squad too
        placed += 1
    
    return placed


def SquadVisionOverlay(squads, num_turns=5, facing_only=True,
                       opacity=0.30, color_attr="darkPrimary",
                       elevation_mult=0.005, **kw) -> OverlaySpec:
    """Vision cones + facing bars at each piece's simulated final position.

    Renders underneath SquadPlanOverlay (priority 65 < 70):
      - Dotted hex fill for every hex in each piece's FOV
      - White-on-dark facing bar showing which way each piece ends up looking

    Args:
        squads:         list of Squad objects
        num_turns:      turns to simulate forward
        facing_only:    True = 120° cone, False = full radius
        opacity:        fill opacity for vision hexes
        color_attr:     flag attribute for dot color ("darkPrimary", "primary", "comp")
        elevation_mult: sight bonus per elevation unit

    Context used:
        ctx.grid / ctx.builder  — render grid (may be zoomed)
        ctx.board               — coarse grid for simulation (optional)
        ctx.c2f                 — coarse→fine mapping for zoomed views (optional)
    """

    def render(ctx) -> str:
        grid = ctx.grid
        c2f  = getattr(ctx, 'c2f', None)

        board = getattr(ctx, 'board', None)
        if board is not None:
            sim_grid = board.terrain.hexGrid
            sim_elev = board.terrain.elevations
            sim_ctry = board.terrain.fields.get("country")
        else:
            sim_grid = grid
            sim_elev = ctx.terrain.elevations
            sim_ctry = ctx.terrain.fields.get("country")

        # ── Simulate to collect final positions ──
        color_to_hexes: dict[str, set[int]] = {}
        bars = []  # (render_hex_idx, facing, flag)

        for squad in squads:
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            for piece in pieces:
                if not piece.instructions or not piece.instructions.rules:
                    continue

                steps, _ = piece.plan_region(
                    sim_grid, sim_elev,
                    num_turns=num_turns, padding=0,
                    countries=sim_ctry,
                )
                if not steps:
                    continue

                last = steps[-1]
                color = (getattr(piece.flag, color_attr, "#888")
                         if piece.flag else "#888")

                # Vision from final position (computed on coarse/sim grid)
                elev = max(0, sim_elev[last.hex_idx])
                eff_sight = int(piece.sight + elev * elevation_mult)

                if facing_only:
                    facing_dir = HexPosition.directions()[last.facing % 6]
                    fov = HexPosition.origin().field_of_view(facing_dir, eff_sight)
                    visible = [idx for hp in fov
                               if (idx := sim_grid.hexposition_to_index(hp, last.hex_idx)) >= 0]
                else:
                    visible = sim_grid.indices_in_range(last.hex_idx, eff_sight)

                # Map coarse → fine for zoomed views
                if c2f is not None:
                    visible = [fi for v in visible if (fi := _map_point(v, c2f)) >= 0]
                    render_idx = _map_point(last.hex_idx, c2f)
                else:
                    render_idx = last.hex_idx

                color_to_hexes.setdefault(color, set()).update(visible)

                if 0 <= render_idx < len(grid.hexes):
                    bars.append((render_idx, last.facing, piece.flag))

        # ── Render vision hex fill ──
        parts = []
        r = grid.radius if hasattr(grid, 'radius') else 20
        dot_r   = max(2.5, r * 0.18)
        spacing = max(5.0, r * 0.32)

        for color, hex_indices in color_to_hexes.items():
            pat_id = f"vision_{color.replace('#', '')}"
            pat_svg = (
                f'<pattern id="{pat_id}" x="0" y="0" '
                f'width="{spacing:.1f}" height="{spacing:.1f}" '
                f'patternUnits="userSpaceOnUse">'
                f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
                f'r="{dot_r:.1f}" fill="{color}"/>'
                f'</pattern>')
            ctx.builder.add_definition(SVGDef("", pat_id, pat_svg, raw=True))

            for idx in sorted(hex_indices):
                if idx < 0 or idx >= len(grid.hexes):
                    continue
                h = grid.hexes[idx]
                pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
                parts.append(
                    f'<polygon points="{pts}" '
                    f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                    f'stroke="{color}" stroke-width="0.6" '
                    f'stroke-dasharray="3,2" stroke-opacity="0.4"/>')

        # ── Render facing bars ──
        for render_idx, facing, flag in bars:
            c = grid.hexes[render_idx].center
            g_out = DiagramGlyphs(color="#111111", size=r * 0.7,
                                  stroke_width=5.0, opacity=0.85)
            g_bar = DiagramGlyphs(color="#FFFFFF", size=r * 0.7,
                                  stroke_width=2.8, opacity=1.0)
            parts.append(g_out.facing_bar(c.x, c.y, facing))
            parts.append(g_bar.facing_bar(c.x, c.y, facing))

        return '\n'.join(parts)

    return OverlaySpec("squad_vision", render, priority=65)


In [ ]:
myTerr = TerraDemo().japan_korea_map()#.shrinkWeather()
myGrid = myTerr.hexGrid
myBuilder = myGrid.builder
def clearTerr():
    myBuilder.layers = []
myGrid.adjustRadius(10)
rivers = myTerr.carve_to_ocean(num_lakes=0)

In [ ]:
myTerr.compute_climate()

basins = DrainageBasins(myTerr)

In [ ]:
myBoard = GameBoard(myTerr,2) # palette_name = "Spectral")

In [ ]:

myCountry = myBoard.kingdoms[0]
myHome = myCountry.settlements[0]
mySquads = Squad.squads(6,myCountry.flag)
for squad in mySquads:
    squad.countryName = myCountry.countryName
queenGuard = mySquads[0]
for pType in [PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.QUEEN]:
    piece = myHome.recruit(pType,flag=queenGuard.flag)
    #FoodProfile.apply(piece)
    queenGuard.pieces.append(piece)

myHome.place_squad_spiral(queenGuard,myTerr)
myQueen = queenGuard.by_rank()[0]
show(myQueen)

GroupedPieceList(queenGuard.pieces,grid=myGrid)




In [ ]:
queenRegion = HexRegion(hexes=set(myTerr.ring(myQueen.location, 6)), hexGrid=myTerr.hexGrid)
def clearTerr():
    myBuilder.layers = []
    queenRegion = HexRegion(hexes=set(myTerr.ring(myQueen.location, 6)), hexGrid=myTerr.hexGrid)


## FoodYield

In [ ]:
#| export


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve: dict = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")


In [ ]:
#| export
@patch
def __ft__(self: FoodYield):
    """Distribution charts for food yield tuning."""
    t = self.terrain
    land = t.elevations > 0

    tiers     = self.tiers[land]
    elevs     = t.elevations[land]
    climate   = t.fields.get('climate', t.compute_climate())[land]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

    # ── 1. Tier distribution bar chart ──
    ax = axes[0]
    counts = [np.sum(tiers == i) for i in range(self.n_tiers)]
    colors = plt.cm.YlGn(np.linspace(0.2, 0.9, self.n_tiers))
    bars = ax.bar(range(self.n_tiers), counts, color=colors, edgecolor='#555', linewidth=0.5)
    ax.set_title('Tier Distribution', fontweight='bold', fontsize=10)
    ax.set_xlabel('Tier'); ax.set_ylabel('Land hexes')
    ax.set_xticks(range(self.n_tiers))
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    str(count), ha='center', va='bottom', fontsize=7, color='#444')
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 2. Tier by elevation band ──
    ax = axes[1]
    elev_max = np.percentile(elevs, 98)
    n_bands = 6
    band_edges = np.linspace(0, elev_max, n_bands + 1)
    band_data  = []
    band_labels = []
    for lo, hi in zip(band_edges, band_edges[1:]):
        mask = (elevs >= lo) & (elevs < hi)
        band_data.append(tiers[mask].tolist() if mask.any() else [0])
        band_labels.append(f"{lo:.0f}–{hi:.0f}")

    bp = ax.boxplot(band_data, tick_labels=band_labels, patch_artist=True,
                    medianprops=dict(color='#222', linewidth=1.5),
                    whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch in bp['boxes']:
        patch.set_facecolor('#8D6E63'); patch.set_alpha(0.55)
    ax.set_title('Tier by Elevation Band', fontweight='bold', fontsize=10)
    ax.set_xlabel('Elevation'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 3. Tier by climate zone ──
    ax = axes[2]
    climate_names = {0:'Ocean/Ice', 1:'Tundra', 2:'Boreal',
                     3:'Temperate', 4:'Subtropical', 5:'Tropical'}
    zones_present = sorted(set(int(c) for c in climate))
    zone_data   = [tiers[climate == z].tolist() for z in zones_present]
    zone_labels = [climate_names.get(z, str(z)) for z in zones_present]
    zone_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(zones_present)))

    bp2 = ax.boxplot(zone_data, tick_labels=zone_labels, patch_artist=True,
                     medianprops=dict(color='#222', linewidth=1.5),
                     whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch, color in zip(bp2['boxes'], zone_colors):
        patch.set_facecolor(color); patch.set_alpha(0.65)
    ax.set_title('Tier by Climate Zone', fontweight='bold', fontsize=10)
    ax.set_xlabel('Climate'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    land_count = int(np.sum(land))
    fig.suptitle(
        f"FoodYield — {land_count} land hexes, {self.n_tiers} tiers",
        fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"temp_curve · soil_mult · {self.n_tiers} tiers",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


In [ ]:
# Make sure soil exists

soil = SoilSystem.from_plates(myTerr, [])

fy = FoodYield(myTerr, basins)
fy.compute()
#show(fy)
fy.summary()

In [ ]:
#| export


def food_overlay(fy: FoodYield, color: str = "#558B2F") -> str:
    """Build dotted overlay from computed food yield tiers."""
    terrain = fy.terrain
    grid = terrain.hexGrid
    
    patGen = TerrainPatterns(terrain)
    patterns = patGen.ballDensity(
        levels=fy.n_tiers,
        fills=[color],
        prefix="food_yield"
    )
    
    overlay = ""
    used = set()
    
    for i in range(len(terrain.elevations)):
        if terrain.elevations[i] <= 0:
            continue
        tier = int(fy.tiers[i])
        if tier == 0:
            continue  # Skip barren — let base map show through
        
        used.add(tier)
        pat_name = patterns[tier].attributes['id']
        hex_obj = grid.hexes[i]
        
        pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
        overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
    
    for tier in sorted(used):
        grid.builder.add_definition(patterns[tier])
    
    return overlay


In [ ]:
svg = food_overlay(fy)
myTerr.hexGrid.builder.layers = []
myTerr.terrainCream()
myTerr.hexGrid.builder.adjust("food", svg)
show(myTerr.hexGrid.builder)


## FoodOverlay

In [ ]:
#| export
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


TerrainDisplay(
    CreamOverlay(),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)


In [ ]:
#!cat ../../HexMagic/game/piece.py

## Food Profile

Great balancing question. Let's work through the math.

**The steady-state formula** for a pawn on a `[HARVEST, GIVE]` patrol (assuming move_strength=1, so each takes a full turn):

```
delivery_per_turn = (h × T  −  2 × d_p) / 2
```

where `h` = harvest_strength, `T` = food tier, `d_p` = pawn diet. The `2 × d_p` accounts for eating on *both* the harvest and the give turn.

**Current values** (h=3, d_p=0.5, d_q=3.0):

| Tier | 1 pawn/turn | 2 pawns/turn | Feeds queen? |
|------|------------|-------------|-------------|
| 2 | 2.5 | 5.0 | ✅ 1 pawn |
| 3 | 4.0 | 8.0 | ✅ 1 pawn |
| 5 | 7.0 | 14.0 | ✅ 1 pawn |

One pawn already covers the queen at tier 2. Way too easy.

**Recommended tweak:** drop pawn `harvest_strength` 3→2, bump queen `diet` 3→5:

```python
FOOD_PROFILES = {
    PieceType.QUEEN:  FoodProfile(diet=5.0, food_capacity=12.0, harvest_strength=2),
    PieceType.PAWN:   FoodProfile(diet=0.5, food_capacity=8.0,  harvest_strength=2),
    # ... rest unchanged
}
```

| Tier | 1 pawn/turn | 2 pawns/turn | Queen needs 5.0 |
|------|------------|-------------|-----------------|
| 1 | 0.5 | 1.0 | ❌ need 3+ |
| 2 | 1.5 | 3.0 | ❌ need 2+ pawns + queen self-harvest |
| 3 | 2.5 | 5.0 | ✅ **exactly 2 pawns** |
| 4 | 3.5 | 7.0 | ✅ 2 pawns (surplus) |
| 5 | 4.5 | 9.0 | ✅ 2 pawns |
| 6 | 5.5 | 11.0 | ✅ **1 pawn!** |
| 7 | 6.5 | 13.0 | ✅ 1 pawn |

This gives you the exact dynamic you want:
- **Tiers 3-5** (typical land): 2 pawns needed ← the normal case
- **Tiers 6-7** (very fertile, top ~17% of your land): 1 pawn suffices
- **Tiers 1-2** (barren): even 2 pawns struggle — queen needs to self-harvest or get a 3rd pawn

And when the queen **parks on** a tier 6+ hex, she self-harvests `2×6 = 12` food/turn vs eating 5/turn — fully self-sufficient without any pawns. That's the "fertile hex" escape valve.

**One caveat:** if your pawns have `move_strength ≥ 2`, they execute HARVEST+GIVE in a *single* turn, and the formula becomes `h×T − d_p` (no division by 2) — roughly doubling delivery. If that's the case, you'd want to push queen diet even higher (~8) or drop harvest_strength to 1. What's the pawn's move_strength?

In [ ]:
#| export
STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

In [ ]:
#| export
@dataclass
class FoodProfile:
    """Per-piece-type food configuration."""
    diet:             float
    food_capacity:    float
    harvest_strength: int

    @staticmethod
    def apply(piece: 'Piece'):
        profile = FOOD_PROFILES.get(piece.piece_type)
        if profile and piece.diet == 1.0 and piece.food_capacity == 10.0:
            piece.diet             = profile.diet
            piece.food_capacity    = profile.food_capacity
            piece.harvest_strength = profile.harvest_strength
            piece.food             = profile.diet * FOOD_RESERVE_TURNS


FOOD_PROFILES: dict[PieceType, FoodProfile] = {
    PieceType.KING:   FoodProfile(diet=4.0, food_capacity=15.0, harvest_strength=1),
    PieceType.QUEEN:  FoodProfile(diet=6.0, food_capacity=12.0, harvest_strength=1),
    PieceType.ROOK:   FoodProfile(diet=4.0, food_capacity=12.0, harvest_strength=1),
    PieceType.BISHOP: FoodProfile(diet=1.5, food_capacity=75.0, harvest_strength=1),
    PieceType.KNIGHT: FoodProfile(diet=4.5, food_capacity=10.0, harvest_strength=1),
    PieceType.PAWN:   FoodProfile(diet=0.5, food_capacity=8.0,  harvest_strength=2),
}


In [ ]:
for person in queenGuard:
    FoodProfile.apply(person)

In [ ]:
#!cat ../../HexMagic/plot/*.py

## Mecanics

In [ ]:
#| export


def harvest(piece: Piece, grid: HexGrid, food_tiers: np.ndarray):
    """HARVEST action: gather food from current tile."""
    if piece.location is None or piece.location < 0:
        return
    tier = int(food_tiers[piece.location])
    gained = piece.harvest_strength * tier
    piece.food = min(piece.food + gained, piece.food_capacity)



def eat(piece: Piece) -> bool:
    """Consume food; apply starvation damage. Returns True if alive."""
    shortfall = max(0.0, piece.diet - piece.food)
    piece.food = max(0.0, piece.food - piece.diet)
    if shortfall > 0:
        piece.health -= shortfall * STARVE_MULT
    return piece.health > 0


def piece_food_step(piece: Piece, instruction: 'Instruction',
                    grid: HexGrid, food_tiers: np.ndarray,
                    all_pieces: list[Piece]) -> bool:
    """Run one turn's food cycle for a single piece.
    
    Call AFTER movement has been resolved.
    Returns True if piece is still alive.
    """
    if instruction == Instruction.HARVEST:
        harvest(piece, grid, food_tiers)
    elif instruction == Instruction.GIVE:
        give(piece, grid, all_pieces)

    return eat(piece)


#| export
@patch
def pieces_in_sight(piece: Piece, grid: HexGrid, elevations: np.ndarray,
                    squads: list[Squad],
                    elevation_mult: float = 0.005,
                    facing_only: bool = False,
                    allied_only: bool = False) -> list[Squad]:
    """Find pieces visible from piece's location, grouped by squad.
    
    Accepts squads, returns squads containing only visible members.
    Empty squads (no visible members) are dropped.
    """
    if piece.location is None or piece.location < 0:
        return []

    elev = max(0, elevations[piece.location])
    effective_sight = int(piece.sight + elev * elevation_mult)
    origin = grid.index_to_hexposition(piece.location)
    facing_dir = HexPosition.directions()[piece.facing % 6]

    if facing_only:
        fov_set = set(field_of_view(HexPosition.origin(), facing_dir, effective_sight))

    result = []
    for squad in squads:
        visible_members = []
        for other in squad.alive:
            if other is piece or other.location is None:
                continue
            if allied_only and other.owner_id != piece.owner_id:
                continue

            relative = grid.index_to_hexposition(other.location, piece.location)

            if facing_only:
                if relative not in fov_set:
                    continue
            else:
                if abs(relative) > effective_sight:
                    continue

            visible_members.append(other)

        if visible_members:
            result.append(Squad(id=squad.id, name=squad.name, pieces=visible_members))

    return result


can pieces_in_ sight take and give back squads

In [ ]:
#| export
@patch
def give(piece: Piece, grid: HexGrid, elevations: np.ndarray,
         battlefield: list[Piece]):
    """GIVE action: distribute surplus food to allies in facing cone.
    
    battlefield is ALL pieces (across all squads) — give targets
    any allied piece in sight, not just squad-mates.
    """
    reserve = FOOD_RESERVE_TURNS * piece.diet
    surplus = max(0.0, piece.food - reserve)
    if surplus <= 0:
        return

    receivers = [
        p for p in piece.pieces_in_sight(
            grid, elevations, battlefield,
            facing_only=True, allied_only=True
        )
        if p.food < p.food_capacity
    ]

    receivers.sort(key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

    for receiver in receivers:
        if surplus <= 0:
            break
        amount = min(surplus, receiver.food_capacity - receiver.food)
        receiver.food += amount
        piece.food -= amount
        surplus -= amount


## Squad extensions

#| export
@patch
def __ft__(self: Squad, grid: HexGrid = None, list_pieces: bool = False):
    """Compact squad dashboard — header with icon + name, box plots by piece type."""
    alive = self.alive
    all_pieces = self.pieces

    # ── Header: icon (left) + text (center) + flag (right) ──
    icon_html = None
    flag_builder = None
    if self.flag is not None:
        animal = self.animal or self.flag.animal_name()
        icon_svg = self.flag.animal_svg(size='large', animal=animal,
                                         piece_id=f"squad_icon_{self.id}")
        r = int(22.5 * 3.5 * 1.3) + 5
        svg_size = r * 2 + 8
        icon_html = NotStr(
            f'<svg xmlns="http://www.w3.org/2000/svg" '
            f'width="{svg_size}" height="{svg_size}" '
            f'viewBox="0 0 {svg_size} {svg_size}">{icon_svg}</svg>'
        )
        flag_builder = self.flag.flag_banner(width=90, height=60, wavy=True, pole=True)

    squad_label = self.name or f"Squad #{self.id}"
    total_alive = len(alive)
    total_all = len(all_pieces)

    text_parts = [Strong(squad_label, style="font-size:1.2em")]
    if self.countryName is not None:
        text_parts.append(P(self.countryName, cls="text-sm opacity-70"))
    text_parts.append(P(f"🗡 {total_alive}/{total_all} alive", cls="text-sm"))

    header_parts = []
    if icon_html:     header_parts.append(Div(icon_html, cls="shrink-0"))
    header_parts.append(Div(*text_parts, cls="space-y-1 flex-1 text-center"))
    if flag_builder:  header_parts.append(Div(NotStr(flag_builder.xml()), cls="shrink-0"))

    header = Div(*header_parts, cls="flex items-center gap-4 mb-3")

    # ── Early out if no pieces — still show header ──
    if not all_pieces:
        return Div(header, P("No pieces assigned", cls="text-sm opacity-50 ml-2"),
                   id=f"squad-{self.id}", cls="space-y-2")

    # ── Group by piece type (ranked order) ──
    types_present = sorted(set(p.piece_type for p in all_pieces),
                           key=lambda t: RANK_ORDER.get(t, 99))
    type_labels = [t.name.title() for t in types_present]

    health_data, food_data, turns_data, distance_data = [], [], [], []
    alive_counts, total_counts = [], []

    centroid = None
    if grid is not None:
        locs = [p.location for p in alive
                if p.location is not None and p.location >= 0]
        if len(locs) >= 2:
            positions = [grid.index_to_hexposition(loc) for loc in locs]
            cq = sum(p.q for p in positions) / len(positions)
            cr = sum(p.r for p in positions) / len(positions)
            cs = sum(p.s for p in positions) / len(positions)
            centroid = (cq, cr, cs)

    for pt in types_present:
        members = [p for p in all_pieces if p.piece_type == pt]
        alive_m = [p for p in members if p.health > 0]
        alive_counts.append(len(alive_m))
        total_counts.append(len(members))
        health_data.append([p.health for p in members])
        food_data.append([p.food for p in members])
        turns_data.append([p.food / max(p.diet, 0.01) for p in members])

        if centroid is not None:
            dists = []
            for p in members:
                if p.location is not None and p.location >= 0:
                    pos = grid.index_to_hexposition(p.location)
                    d = (abs(pos.q - centroid[0])
                       + abs(pos.r - centroid[1])
                       + abs(pos.s - centroid[2])) / 2
                    dists.append(d)
                else:
                    dists.append(float('nan'))
            distance_data.append(dists)

    # ── Build figure ──
    has_dist = centroid is not None and distance_data
    n_panels = 4 if has_dist else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(3.2 * n_panels, 3.0))
    if n_panels == 1: axes = [axes]

    panels = [
        ("Health",            health_data,   "#e74c3c"),
        ("Food",              food_data,     "#27ae60"),
        ("Food (turns left)", turns_data,    "#f39c12"),
    ]
    if has_dist:
        panels.append(("Distance from center", distance_data, "#3498db"))

    for ax, (title, data, color) in zip(axes, panels):
        if not data or all(len(d) == 0 for d in data):
            ax.set_visible(False)
            continue
        bp = ax.boxplot(data, tick_labels=type_labels, patch_artist=True,
                        medianprops=dict(color='#222', linewidth=1.5),
                        whiskerprops=dict(color='#888'),
                        capprops=dict(color='#888'))
        for patch in bp['boxes']:
            patch.set_facecolor(color); patch.set_alpha(0.55)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', rotation=35, labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
        ax.grid(axis='y', alpha=0.3, linewidth=0.5)
        ymax = ax.get_ylim()[1]
        for i, (a, t) in enumerate(zip(alive_counts, total_counts)):
            ax.text(i + 1, ymax * 1.02, f"{a}/{t}",
                    ha='center', va='bottom', fontsize=7, color='#555',
                    fontstyle='italic')

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    avg_hp = np.mean([p.health for p in alive]) if alive else 0
    min_hp = min((p.health for p in alive), default=0)
    avg_turns = np.mean([p.food / max(p.diet, 0.01) for p in alive]) if alive else 0

    summary = P(
        f"❤ avg {avg_hp:.0f} (min {min_hp:.0f}) · "
        f"🌾 {avg_turns:.1f} avg food turns",
        cls="text-xs opacity-60 text-center mt-1"
    )

    parts = [header, NotStr(buf.getvalue()), summary]
    if list_pieces:
        parts.append(Hr())
        parts.append(PieceList(all_pieces))

    return Div(*parts, id=f"squad-{self.id}", cls="space-y-2")


## TroopPath

In [ ]:
#| export
@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> TroopPath:
    """Nearest reachable tile with food >= min_tier. Returns TroopPath (falsy if none)."""
    start = self.location
    if start is None or start < 0:
        return TroopPath([], 0.0)

    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    best = {start: 0.0}

    while pq:
        cost, current = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)

        if current != start and int(food_tiers[current]) >= min_tier:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return TroopPath(path, cost)

        for nb in grid.neighborsOf(current):
            if nb in visited or nb in grid.invalidRegion:
                continue
            if elevations[nb] <= 0:
                continue
            if countries is not None and countries[nb] < 0:
                continue
            new_cost = cost + _move_cost(elevations, current, nb)
            if new_cost < best.get(nb, float('inf')):
                best[nb] = new_cost
                parent[nb] = current
                heapq.heappush(pq, (new_cost, nb))

    return TroopPath([], 0.0)



#| export
I want to start to build out how a squad can feed itself. It is a scheduling issue and movement issue. pieces need to be able to take turns feeding and sharing and possibly moving to better hexes. we also want to figure out if a squad needs outside help

Lets do SquadFoodAudit — static analysis: "given where everyone is right now, can this squad sustain itself?" Just math, no movement.

we defintiely are going to want this patched on a squad

should we factor in the yeild of what they can harvest based upon their hex?

We are going to want an overlay that shows this

Perfect. I think we can start to develop

```python
def SquadSymbolOverlay(squads, size='board', font='Cinzel',
                       label_offset=22, show_label=True, **kw) -> OverlaySpec:
    """Draw each squad's animal symbol at its centroid with a name label beneath.

    Args:
        squads:       list of Squad objects
        size:         animal_svg size — 'board', 'list', or 'large'
        font:         font family for labels (loaded via add_font)
        label_offset: vertical px offset for label below symbol center
        show_label:   whether to render the text label

    Context used:
        ctx.grid / ctx.builder  — render grid
        ctx.board               — coarse grid for centroid calc (optional)
        ctx.c2f                 — coarse→fine mapping for zoomed views (optional)
    """

    def render(ctx) -> str:
        grid    = ctx.grid
        c2f     = getattr(ctx, 'c2f', None)
        board   = getattr(ctx, 'board', None)
        sim_grid = board.terrain.hexGrid if board else grid

        ctx.builder.add_font(font)

        parts = []
        for i, squad in enumerate(squads):
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            if not pieces:
                continue

            center_idx, _ = pieces_center(pieces, sim_grid)
            if center_idx < 0:
                continue

            # Map coarse → fine for zoomed views
            render_idx = _map_point(center_idx, c2f)
            if render_idx < 0 or render_idx >= len(grid.hexes):
                continue

            center = grid.hexes[render_idx].center
            flag   = squad.flag
            if flag is None:
                continue

            # ── Animal symbol ──
            sid = f"sq_{squad.id}_{i}"
            parts.append(flag.animal_svg(size=size, center=center, piece_id=sid))

            # ── Label ──
            if show_label:
                label = squad.name or squad.animal or ""
                if not label:
                    continue
                lbl_style = flag.labelStyle(f"sql_{squad.id}_{i}")
                ctx.builder.add_style(lbl_style)
                parts.append(
                    f'<text x="{center.x}" y="{center.y + label_offset}" '
                    f'text-anchor="middle" font-size="11" '
                    f"font-family=\"'{font}', serif\" "
                    f'dominant-baseline="hanging" '
                    f'class="{lbl_style.name}">{label}</text>'
                )

        return '\n'.join(parts)

    return OverlaySpec("squad_symbols", render, priority=75)
```
this a little more to show diet levels. And separate overlay for the troops.

Lets build out the audit needs to return per-piece results, not just a single number. Something like a list of (piece, gross, net, role) records plus the squad-level totals and keep the overlay design as a design refernce

#| export
My question is do we need a FoodAuditEntry as opposed pandas dataframe, or simple numpy tables. It feels like alot of this is across multiple Entries

Can you built it? do we need role, because isn't that determined by net?

In [ ]:
#| export
@patch
def food_audit(self: Squad, food_tiers: np.ndarray) -> pd.DataFrame:
    """Static food analysis — one row per alive piece."""
    rows = []
    for p in self.alive:
        tier = int(food_tiers[p.location]) if (p.location is not None and p.location >= 0) else 0
        gross = p.harvest_strength * tier
        net = gross - p.diet
        rows.append({
            'piece': p,
            'name': p.name,
            'type': p.piece_type.name,
            'diet': p.diet,
            'harvest_str': p.harvest_strength,
            'tier': tier,
            'gross': gross,
            'net': net,
            'food': p.food,
            'reserve_turns': p.food / p.diet if p.diet > 0 else float('inf'),
            'location': p.location,
        })
    df = pd.DataFrame(rows)
    df.attrs['demand'] = df.diet.sum()
    df.attrs['supply'] = df.loc[df.net > 0, 'net'].sum()
    df.attrs['gap'] = df.attrs['demand'] - df.attrs['supply'] - df.loc[df.net <= 0, 'gross'].sum()
    return df


So maybe we need a general purpose piece dataframe to overlay that would bin positive and negative values (if no bins are provided)

I think just colors. then we could also have PieceArrowOverlay that would try to connect sources and sinks with arrows

from matplotlib.colors import to_hex, to_rgb

def _lerp_color(c1, c2, t):
    """Blend two colors. t=0 → c1, t=1 → c2."""
    a, b = np.array(to_rgb(c1)), np.array(to_rgb(c2))
    return to_hex(np.clip(a + (b - a) * t, 0, 1))




from matplotlib.colors import to_hex, to_rgb

def _lerp_color(c1, c2, t):
    """Blend two colors. t=0 → c1, t=1 → c2."""
    a, b = np.array(to_rgb(c1)), np.array(to_rgb(c2))
    return to_hex(np.clip(a + (b - a) * t, 0, 1))

def PieceDataOverlay(df: pd.DataFrame, column: str = 'net',
                     bins: int = 5,
                     pos_color: str = '#27ae60', neg_color: str = '#e74c3c',
                     zero_color: str = '#eeeeee',
                     opacity: float = 0.45, **kw) -> OverlaySpec:
    """Color-fill each piece's hex by a DataFrame column value.
    
    Auto-builds a diverging color ramp around zero.
    Expects df to have a 'location' column (hex index).
    """
    def render(ctx) -> str:
        grid = ctx.grid
        vals = df[column].values.astype(float)
        locs = df['location'].values.astype(int)

        vmin, vmax = vals.min(), vals.max()
        # Symmetric range so zero sits at center
        bound = max(abs(vmin), abs(vmax), 0.01)

        parts = []
        for val, loc in zip(vals, locs):
            if loc < 0 or loc >= len(grid.hexes):
                continue
            # Normalize to -1..+1
            t = np.clip(val / bound, -1, 1)
            if t >= 0:
                color = _lerp_color(zero_color, pos_color, t)
            else:
                color = _lerp_color(zero_color, neg_color, -t)

            h = grid.hexes[loc]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            parts.append(
                f'<polygon points="{pts}" fill="{color}" '
                f'opacity="{opacity:.2f}" stroke="{color}" '
                f'stroke-width="1.5" stroke-opacity="0.6"/>')
        return '\n'.join(parts)

    return OverlaySpec("piece_data", render, priority=72)


def PieceArrowOverlay(df: pd.DataFrame, column: str = 'net',
                      color: str = '#555555',
                      max_width: float = 4.0, min_width: float = 1.0,
                      opacity: float = 0.6,
                      head_size: float = 8.0, **kw) -> OverlaySpec:
    """Draw arrows from sources (column > 0) to nearest sinks (column < 0).
    
    Each sink connects to its nearest source. Arrow width ∝ sink demand.
    """
    def render(ctx) -> str:
        grid = ctx.grid

        sources = df[df[column] > 0]
        sinks = df[df[column] < 0]

        if sources.empty or sinks.empty:
            return ''

        # Marker definition for arrowhead
        marker_id = "food_arrow_head"
        marker_svg = (
            f'<marker id="{marker_id}" viewBox="0 0 10 10" '
            f'refX="9" refY="5" markerWidth="{head_size:.0f}" '
            f'markerHeight="{head_size:.0f}" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{color}" '
            f'opacity="{opacity:.2f}"/></marker>')
        ctx.builder.add_definition(SVGDef("", marker_id, marker_svg, raw=True))

        # Pre-compute source centers
        src_locs = sources['location'].values.astype(int)
        src_centers = []
        for loc in src_locs:
            if 0 <= loc < len(grid.hexes):
                c = grid.hexes[loc].center
                src_centers.append((loc, c.x, c.y))

        if not src_centers:
            return ''

        max_demand = max(abs(sinks[column].min()), 0.01)

        parts = []
        for _, row in sinks.iterrows():
            sink_loc = int(row['location'])
            if sink_loc < 0 or sink_loc >= len(grid.hexes):
                continue
            sc = grid.hexes[sink_loc].center

            # Find nearest source by pixel distance
            best_dist, best_src = float('inf'), None
            for s_loc, sx, sy in src_centers:
                d = math.hypot(sx - sc.x, sy - sc.y)
                if d < best_dist:
                    best_dist, best_src = d, (sx, sy)

            if best_src is None:
                continue

            # Width proportional to demand magnitude
            t = min(abs(row[column]) / max_demand, 1.0)
            w = min_width + (max_width - min_width) * t

            sx, sy = best_src
            parts.append(
                f'<line x1="{sx:.1f}" y1="{sy:.1f}" '
                f'x2="{sc.x:.1f}" y2="{sc.y:.1f}" '
                f'stroke="{color}" stroke-width="{w:.1f}" '
                f'opacity="{opacity:.2f}" '
                f'marker-end="url(#{marker_id})"/>')

        return '\n'.join(parts)

    return OverlaySpec("piece_arrows", render, priority=73)


In [ ]:
def PieceArrowOverlay(df: pd.DataFrame, column: str = 'net',
                      color: str = '#555555',
                      max_width: float = 4.0, min_width: float = 1.0,
                      opacity: float = 0.6,
                      head_size: float = 8.0, **kw) -> OverlaySpec:
    """Draw arrows from sources (column > 0) to nearest sinks (column < 0).
    
    Each sink connects to its nearest source. Arrow width ∝ sink demand.
    """
    def render(ctx) -> str:
        grid = ctx.grid
        c2f = getattr(ctx, 'c2f', None)

        sources = df[df[column] > 0]
        sinks = df[df[column] < 0]

        if sources.empty or sinks.empty:
            return ''

        # Marker definition for arrowhead
        marker_id = "food_arrow_head"
        marker_svg = (
            f'<marker id="{marker_id}" viewBox="0 0 10 10" '
            f'refX="9" refY="5" markerWidth="{head_size:.0f}" '
            f'markerHeight="{head_size:.0f}" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{color}" '
            f'opacity="{opacity:.2f}"/></marker>')
        ctx.builder.add_definition(SVGDef("", marker_id, marker_svg, raw=True))

        # Pre-compute source centers (mapped for zoom)
        src_locs = sources['location'].values.astype(int)
        src_centers = []
        for loc in src_locs:
            render_idx = _map_point(loc, c2f) if c2f is not None else loc
            if 0 <= render_idx < len(grid.hexes):
                c = grid.hexes[render_idx].center
                src_centers.append((render_idx, c.x, c.y))

        if not src_centers:
            return ''

        max_demand = max(abs(sinks[column].min()), 0.01)

        parts = []
        for _, row in sinks.iterrows():
            sink_loc = int(row['location'])
            render_idx = _map_point(sink_loc, c2f) if c2f is not None else sink_loc
            if render_idx < 0 or render_idx >= len(grid.hexes):
                continue
            sc = grid.hexes[render_idx].center

            # Find nearest source by pixel distance
            best_dist, best_src = float('inf'), None
            for s_loc, sx, sy in src_centers:
                d = math.hypot(sx - sc.x, sy - sc.y)
                if d < best_dist:
                    best_dist, best_src = d, (sx, sy)

            if best_src is None:
                continue

            # Width proportional to demand magnitude
            t = min(abs(row[column]) / max_demand, 1.0)
            w = min_width + (max_width - min_width) * t

            sx, sy = best_src
            parts.append(
                f'<line x1="{sx:.1f}" y1="{sy:.1f}" '
                f'x2="{sc.x:.1f}" y2="{sc.y:.1f}" '
                f'stroke="{color}" stroke-width="{w:.1f}" '
                f'opacity="{opacity:.2f}" '
                f'marker-end="url(#{marker_id})"/>')

        return '\n'.join(parts)

    return OverlaySpec("piece_arrows", render, priority=73)


In [ ]:
def PieceDataOverlay(df: pd.DataFrame, column: str = 'net',
                     pos_color: str = '#27ae60', neg_color: str = '#e74c3c',
                     zero_color: str = '#eeeeee',
                     opacity: float = 0.45, **kw) -> OverlaySpec:
    """Color-fill each piece's hex by a DataFrame column value."""
    def render(ctx) -> str:
        grid = ctx.grid
        c2f = getattr(ctx, 'c2f', None)
        vals = df[column].values.astype(float)
        locs = df['location'].values.astype(int)

        bound = max(abs(vals.min()), abs(vals.max()), 0.01)

        parts = []
        for val, loc in zip(vals, locs):
            # Map coarse → fine for zoomed views
            render_idx = _map_point(loc, c2f) if c2f is not None else loc
            if render_idx < 0 or render_idx >= len(grid.hexes):
                continue

            t = np.clip(val / bound, -1, 1)
            if t >= 0:
                color = StyleCSS.lerp_color(zero_color, pos_color, t)
            else:
                color = StyleCSS.lerp_color(zero_color, neg_color, -t)

            h = grid.hexes[render_idx]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            parts.append(
                f'<polygon points="{pts}" fill="{color}" '
                f'opacity="{opacity:.2f}" stroke="{color}" '
                f'stroke-width="1.5" stroke-opacity="0.6"/>')
        return '\n'.join(parts)

    return OverlaySpec("piece_data", render, priority=72)


def SquadSymbolOverlay(squads, size='large', font='Cinzel',
                       show_label=True,
                       ring_color: str = '#333333', ring_width: float = 2.5,
                       ring_fill: str = 'rgba(255,255,255,0.75)',
                       sim_grid: HexGrid = None, **kw) -> OverlaySpec:
    """Draw each squad's animal symbol at its centroid with a name label beneath.

    Args:
        squads:     list of Squad objects
        size:       animal_svg size — 'board', 'list', or 'large'
        font:       font family for labels
        show_label: whether to render the text label
        ring_color: outline color of the backing circle
        ring_width: stroke width of the backing circle
        ring_fill:  fill color/opacity of the backing circle
        sim_grid:   original grid for piece location lookups (needed in zoomed views)

    Context used:
        ctx.grid / ctx.builder  — render grid
        ctx.board               — coarse grid for centroid calc (optional)
        ctx.c2f                 — coarse→fine mapping for zoomed views (optional)
    """

    def render(ctx) -> str:
        grid = ctx.grid
        c2f  = getattr(ctx, 'c2f', None)
        board = getattr(ctx, 'board', None)
        sg = sim_grid or (board.terrain.hexGrid if board else grid)

        ctx.builder.add_font(font)

        parts = []
        for i, squad in enumerate(squads):
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            if not pieces:
                continue

            center_idx, _ = pieces_center(pieces, sg)
            if center_idx < 0:
                continue

            render_idx = _map_point(center_idx, c2f) if c2f is not None else center_idx
            if render_idx < 0 or render_idx >= len(grid.hexes):
                continue

            center = grid.hexes[render_idx].center
            flag   = squad.flag
            if flag is None:
                continue

            # ── Background ring ──
            r = grid.radius if hasattr(grid, 'radius') else 20
            ring_r = r * 0.65
            parts.append(
                f'<circle cx="{center.x:.1f}" cy="{center.y:.1f}" r="{ring_r:.1f}" '
                f'fill="{ring_fill}" stroke="{ring_color}" '
                f'stroke-width="{ring_width}"/>')

            # ── Animal symbol ──
            sid = f"sq_{squad.id}_{i}"
            parts.append(flag.animal_svg(size=size, center=center, piece_id=sid))

            # ── Label — pinned below ring edge, scales with zoom ──
            if show_label:
                label = squad.name or squad.animal or ""
                if not label:
                    continue
                lbl_style = flag.labelStyle(f"sql_{squad.id}_{i}")
                ctx.builder.add_style(lbl_style)
                lbl_y = center.y + ring_r + 8
                parts.append(
                    f'<text x="{center.x}" y="{lbl_y+ring_r:.1f}" '
                    f'text-anchor="middle" font-size="13" font-weight="bold" '
                    f"font-family=\"'{font}', serif\" "
                    f'dominant-baseline="hanging" '
                    f'stroke="white" stroke-width="3" paint-order="stroke" '
                    f'class="{lbl_style.name}">{label}</text>'
                )

        return '\n'.join(parts)

    return OverlaySpec("squad_symbols", render, priority=55)


def SquadDietOverlay(squads, food_tiers: np.ndarray,
                     sim_grid: HexGrid = None,
                     font='Cinzel', bar_width=60, bar_height=6,
                     offset_y=38, **kw) -> OverlaySpec:
    """Food budget badge at each squad's centroid."""
    def render(ctx) -> str:
        grid = ctx.grid
        c2f = getattr(ctx, 'c2f', None)
        board = getattr(ctx, 'board', None)

        # Use explicit sim_grid, or board's grid, or rendering grid
        sg = sim_grid or (board.terrain.hexGrid if board else grid)

        ctx.builder.add_font(font)
        parts = []

        # ── Background ring ──
        r = grid.radius if hasattr(grid, 'radius') else 20
        ring_r = r * 0.65


        for i, squad in enumerate(squads):
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            if not pieces:
                continue

            center_idx, _ = pieces_center(pieces, sg)
            if center_idx < 0:
                continue

            render_idx = _map_point(center_idx, c2f) if c2f is not None else center_idx
            if render_idx < 0 or render_idx >= len(grid.hexes):
                continue

            cx = grid.hexes[render_idx].center.x
            cy = grid.hexes[render_idx].center.y + offset_y

            audit = squad.food_audit(food_tiers)
            demand = audit.attrs['demand']
            supply = audit.attrs['supply']
            gap = audit.attrs['gap']

            if demand > 0:
                supply_frac = min(supply / demand, 1.0)
            else:
                supply_frac = 1.0

            sw = bar_width * supply_frac
            bar_x = cx - bar_width / 2
            bar_y = cy

            parts.append(
                f'<rect x="{bar_x:.1f}" y="{bar_y+ring_r:.1f}" '
                f'width="{bar_width}" height="{bar_height}" '
                f'rx="2" fill="#e74c3c" opacity="0.7"/>')
            if sw > 0:
                parts.append(
                    f'<rect x="{bar_x:.1f}" y="{bar_y+ring_r:.1f}" '
                    f'width="{sw:.1f}" height="{bar_height}" '
                    f'rx="2" fill="#27ae60" opacity="0.8"/>')
            parts.append(
                f'<rect x="{bar_x:.1f}" y="{bar_y+ring_r:.1f}" '
                f'width="{bar_width}" height="{bar_height}" '
                f'rx="2" fill="none" stroke="#333" stroke-width="0.8"/>')

            net_val = -gap
            sign = "+" if gap <= 0 else ""
            color = "#27ae60" if gap <= 0 else "#e74c3c"
            warn = "" if gap <= 0 else " ⚠"

            parts.append(
                f'<text x="{cx:.1f}" y="{cy + bar_height + 11:.1f}" '
                f'text-anchor="middle" font-size="9" '
                f"font-family=\"'{font}', sans-serif\" "
                f'fill="{color}" font-weight="bold">'
                f'🌾 {sign}{net_val:.1f}/turn{warn}</text>')

        return '\n'.join(parts)

    return OverlaySpec("squad_diet", render, priority=76)


In [ ]:
def SquadSymbolOverlay(squads, size='large', font='Cinzel',
                       show_label=True,
                       ring_color: str = '#333333', ring_width: float = 2.5,
                       ring_fill: str = 'rgba(255,255,255,0.75)',
                       sim_grid: HexGrid = None, **kw) -> OverlaySpec:
    """Draw each squad's animal symbol at its centroid with a name label beneath."""

    def render(ctx) -> str:
        grid  = ctx.grid
        c2f   = getattr(ctx, 'c2f', None)
        board = getattr(ctx, 'board', None)
        sg    = sim_grid or (board.terrain.hexGrid if board else grid)

        r      = grid.radius if hasattr(grid, 'radius') else 20
        ring_r = r * 0.65

        ctx.builder.add_font(font)
        parts = []

        for i, squad in enumerate(squads):
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            if not pieces:
                continue

            center_idx, _ = pieces_center(pieces, sg)
            if center_idx < 0:
                continue

            render_idx = _map_point(center_idx, c2f) if c2f is not None else center_idx
            if render_idx < 0 or render_idx >= len(grid.hexes):
                continue

            center = grid.hexes[render_idx].center
            flag   = squad.flag
            if flag is None:
                continue

            # ── Background ring ──
            parts.append(
                f'<circle cx="{center.x:.1f}" cy="{center.y:.1f}" r="{ring_r:.1f}" '
                f'fill="{ring_fill}" stroke="{ring_color}" '
                f'stroke-width="{ring_width}"/>')

            # ── Animal symbol ──
            sid = f"sq_{squad.id}_{i}"
            parts.append(flag.animal_svg(size=size, center=center, piece_id=sid))

            # ── Label — sits just below bottom of ring ──
            if show_label:
                label = squad.name or squad.animal or ""
                if not label:
                    continue
                lbl_style = flag.labelStyle(f"sql_{squad.id}_{i}")
                ctx.builder.add_style(lbl_style)
                lbl_y = center.y + ring_r + 10
                parts.append(
                    f'<text x="{center.x:.1f}" y="{lbl_y:.1f}" '
                    f'text-anchor="middle" font-size="13" font-weight="bold" '
                    f"font-family=\"'{font}', serif\" "
                    f'dominant-baseline="hanging" '
                    f'stroke="white" stroke-width="3" paint-order="stroke" '
                    f'class="{lbl_style.name}">{label}</text>')

        return '\n'.join(parts)

    return OverlaySpec("squad_symbols", render, priority=20)


def SquadDietOverlay(squads, food_tiers: np.ndarray,
                     sim_grid: HexGrid = None,
                     font='Cinzel', bar_width=60, bar_height=6, **kw) -> OverlaySpec:
    """Food budget bar + net/turn label beneath each squad's centroid."""

    def render(ctx) -> str:
        grid  = ctx.grid
        c2f   = getattr(ctx, 'c2f', None)
        board = getattr(ctx, 'board', None)
        sg    = sim_grid or (board.terrain.hexGrid if board else grid)

        r      = grid.radius if hasattr(grid, 'radius') else 20
        ring_r = r * 0.65

        ctx.builder.add_font(font)
        parts = []

        for i, squad in enumerate(squads):
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            if not pieces:
                continue

            center_idx, _ = pieces_center(pieces, sg)
            if center_idx < 0:
                continue

            render_idx = _map_point(center_idx, c2f) if c2f is not None else center_idx
            if render_idx < 0 or render_idx >= len(grid.hexes):
                continue

            cx = grid.hexes[render_idx].center.x
            cy = grid.hexes[render_idx].center.y

            # Anchor everything below the ring + label (label ~14px tall)
            bar_y  = cy + ring_r + 26
            text_y = bar_y + bar_height + 6

            audit  = squad.food_audit(food_tiers)
            demand = audit.attrs['demand']
            supply = audit.attrs['supply']
            gap    = audit.attrs['gap']

            supply_frac = min(supply / demand, 1.0) if demand > 0 else 1.0
            sw = bar_width * supply_frac
            bx = cx - bar_width / 2

            # Background (red = unmet demand)
            parts.append(
                f'<rect x="{bx:.1f}" y="{bar_y:.1f}" '
                f'width="{bar_width}" height="{bar_height}" '
                f'rx="2" fill="#e74c3c" opacity="0.7"/>')
            # Green fill (supply fraction)
            if sw > 0:
                parts.append(
                    f'<rect x="{bx:.1f}" y="{bar_y:.1f}" '
                    f'width="{sw:.1f}" height="{bar_height}" '
                    f'rx="2" fill="#27ae60" opacity="0.8"/>')
            # Border
            parts.append(
                f'<rect x="{bx:.1f}" y="{bar_y:.1f}" '
                f'width="{bar_width}" height="{bar_height}" '
                f'rx="2" fill="none" stroke="#333" stroke-width="0.8"/>')

            # Net/turn text
            net_val = -gap
            sign  = "+" if gap <= 0 else ""
            color = "#27ae60" if gap <= 0 else "#e74c3c"
            warn  = "" if gap <= 0 else " ⚠"
            parts.append(
                f'<text x="{cx:.1f}" y="{text_y:.1f}" '
                f'text-anchor="middle" font-size="9" '
                f"font-family=\"'{font}', sans-serif\" "
                f'dominant-baseline="hanging" '
                f'stroke="white" stroke-width="2.5" paint-order="stroke" '
                f'fill="{color}" font-weight="bold">'
                f'🌾 {sign}{net_val:.1f}/turn{warn}</text>')

        return '\n'.join(parts)

    return OverlaySpec("squad_diet", render, priority=76)


In [ ]:
??FoodProfile

In [ ]:
clearTerr()
myGrid.adjustRadius(50)


In [ ]:
food_tiers = fy.tiers
audit = queenGuard.food_audit(food_tiers)
myGrid.adjustRadius(45)
clearTerr()

TerrainDisplay(
    CreamOverlay(),
    FoodOverlay(color="#cf35b0ff", n_tiers=6),
    PieceDataOverlay(audit, 'net'),
    PieceArrowOverlay(audit, 'net'),
    #SquadSymbolOverlay([queenGuard], sim_grid=myGrid),
    SquadDietOverlay([queenGuard], food_tiers, sim_grid=myGrid),
    #RiverOverlay(),
    terrain=myTerr,
    basins=basins,
    region=queenRegion,
    debug=True
)


In [ ]:
??SquadSymbolOverlay

In [ ]:
# Zoomed
food_tiers = fy.tiers
audit = queenGuard.food_audit(food_tiers)

ctx = OverlayContext.from_terrain(myTerr)


zoomed = myTerr.zoom(queenRegion, padding=2)
ctx = OverlayContext(
    terrain=zoomed, grid=zoomed.hexGrid, builder=zoomed.hexGrid.builder,
    extras=dict(c2f=zoomed.c2f)
)

ctx.test(SquadSymbolOverlay([queenGuard],  sim_grid=myGrid))


How did size 

I am trying to put the label and diet in SquadDietOverlay and SquadSymbolOverlay underneath (larger y) the animal logo I tried adding the radius. what should I use?

Can you write these functions?

whsat should SquadSymbolOverlay look like with your fix?

The squad symbol should be much bigger. At this size it is hard to see especially since the colors are very similar.

But then it should be above the label

So which should be refactored? Can you do it?

@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue
        
        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#333"
        g = DiagramGlyphs(color=color, size=self.grid.radius * 0.45)
        svg += g.facing_bar(c.x, c.y, last.facing)
    
    return svg


@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()


#| export
@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#fff"
        
        # Dark outline pass first, then bright bar on top
        g_outline = DiagramGlyphs(color="#222222",
                                  size=self.grid.radius * 0.55,
                                  stroke_width=4.5, opacity=0.9)
        g_bar = DiagramGlyphs(color="#FFFFFF",
                              size=self.grid.radius * 0.55,
                              stroke_width=2.5, opacity=1.0)
        
        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


## Visiion and facing

#| export
@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()



In [ ]:
#show(NotStr(board.render_with_facing(num_turns=8)))

I can't really see the bars. Are they under? should we use a different color?

??DiagramGlyphs.facing_bar

#| export
@patch
def facing_bar(self: DiagramGlyphs, cx, cy, facing: int,
               length=None, offset=None, stroke=None,
               stroke_width=None, opacity=None) -> str:
    """Short flat bar on the 'front' side of the piece — football blocking style."""
    import math
    length       = length       or self.size * 1.2          # wider bar
    offset       = offset       or self.size * 1.1          # pushed toward hex edge
    stroke       = stroke       or self.color
    stroke_width = stroke_width or max(2, self.size * 0.15)
    opacity      = opacity if opacity is not None else self.opacity

    # SW=0,W=1,NW=2,NE=3,E=4,SE=5 → pixel angles 120,180,240,300,0,60
    angle = math.radians(60 * facing + 120)

    bx = cx + offset * math.cos(angle)
    by = cy + offset * math.sin(angle)

    perp = angle + math.pi / 2
    half = length / 2
    x1, y1 = bx + half * math.cos(perp), by + half * math.sin(perp)
    x2, y2 = bx - half * math.cos(perp), by - half * math.sin(perp)

    return (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
            f'x2="{x2:.1f}" y2="{y2:.1f}" '
            f'stroke="{stroke}" stroke-width="{stroke_width:.1f}" '
            f'stroke-linecap="round" opacity="{opacity}"/>\n')


@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    r = self.grid.radius
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center

        # Size drives offset/length via the multipliers above
        g_outline = DiagramGlyphs(color="#111111", size=r * 0.7,
                                  stroke_width=5.0, opacity=0.85)
        g_bar     = DiagramGlyphs(color="#FFFFFF",  size=r * 0.7,
                                  stroke_width=2.8, opacity=1.0)

        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


In [ ]:
#show(NotStr(board.render_with_facing(num_turns=8)))

So I think we are ready for our sightOverlay. we want to see which hexes are covered by a list of pieces. perhaps the overlay could be a dotted one similar to food, but with the kingdom color

In [ ]:
??Piece.hexes_in_sight

#| export
@patch
def hexes_in_sight(self: Piece, grid: HexGrid, elevations: np.ndarray,
                   elevation_mult: float = 0.005,
                   facing_only: bool = False) -> list[int]:
    """All hex indices visible from this piece's location."""
    if self.location is None or self.location < 0:
        return []
    
    elev = max(0, elevations[self.location])
    effective_sight = int(self.sight + elev * elevation_mult)
    
    if facing_only:
        facing_dir = HexPosition.directions()[self.facing % 6]
        hex_positions = field_of_view(HexPosition.origin(), facing_dir, effective_sight)
        return [
            idx for hp in hex_positions
            if (idx := grid.hexposition_to_index(hp, self.location)) >= 0
        ]
    else:
        return grid.indices_in_range(self.location, effective_sight)





@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.22,
                  facing_only: bool = False,
                  allowed_rotations: int = 1) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(1.5, r * 0.10)
    spacing = max(5.0, r * 0.32)

    # Group by kingdom color
    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = piece.flag.primary if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only,
            allowed_rotations=allowed_rotations
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        # Register dot pattern as a def
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.6" '
                f'stroke-dasharray="3,2" stroke-opacity="0.4"/>\n'
            )
    return svg


@patch
def render_with_sight(self: PieceBoard, pieces: list[Piece],
                      num_turns: int = 8,
                      facing_only: bool = False) -> str:
    """Board with sight overlay + movement plans + facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("sight",  self.sight_overlay(pieces, facing_only=facing_only))
    self.grid.builder.adjust("pieces", self._pieces_overlay())
    self.grid.builder.adjust("plan",   self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing", self._facing_overlay(num_turns))

    xs = [h.center.x for h in self.grid.hexes]
    ys = [h.center.y for h in self.grid.hexes]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()


 think we need to work on field of view which seems to go wider that 120. in ring 1 it is fine. but in ring 2 it should go straing out from its edges but it bends

These wound up too suttble. I wonder if darkPrimary or baseComp or perhaps larger dots makes more sense.

#| export
@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.35,          # up from 0.22
                  facing_only: bool = False,
                  allowed_rotations: int = 1,
                  color_attr: str = "darkPrimary"  # "primary" | "darkPrimary" | "baseComp"
                  ) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(2.5, r * 0.18)   # up from 0.10
    spacing = max(5.0, r * 0.32)

    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = getattr(piece.flag, color_attr, "#888") if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.8" '
                f'stroke-dasharray="3,2" stroke-opacity="0.5"/>\n'
            )
    return svg


#| export
@patch
def render_with_sight(self: PieceBoard, pieces: list[Piece],
                      num_turns: int = 8,
                      facing_only: bool = False) -> str:
    """Board with sight overlay + movement plans + facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("sight",  self.sight_overlay(pieces, facing_only=facing_only))
    self.grid.builder.adjust("pieces", self._pieces_overlay())
    self.grid.builder.adjust("plan",   self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing", self._facing_overlay(num_turns))

    xs = [h.center.x for h in self.grid.hexes]
    ys = [h.center.y for h in self.grid.hexes]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()

Lets do a demo with one piece so I can see the cone

# darkPrimary — recommended
show(NotStr(board.render_with_sight(feeders + [queen], num_turns=8)))

In [ ]:
??HexPosition.field_of_view

#| export
def field_of_view(origin: HexPosition, facing: HexPosition, max_distance: int) -> list[HexPosition]:
    """Get all hexes within a 120° field of view (±60° from facing) up to max_distance.
    
    Walks the cone arc per ring — no per-hex tests needed.
    """
    left_edge = facing.rotate(1)     # start of arc (left side)
    step1 = facing.rotate(-1)        # walk from left toward center
    step2 = facing.rotate(-2)        # walk from center toward right
    
    results = []
    for r in range(1, max_distance + 1):
        current = origin + r * left_edge
        for _ in range(r):
            results.append(current)
            current = current + step1
        for _ in range(r):
            results.append(current)
            current = current + step2
        results.append(current)      # final right-edge hex
    return results


In [ ]:
??Squad.supply_line

In [ ]:
#| export
@patch
def supply_line(feeders: Squad, path: list[int], 
                grid: HexGrid, elevations: np.ndarray,
                food_tiers: np.ndarray,
                min_tier: int = 2,
                max_offset: int = 2,
                countries: np.ndarray = None,
                elevation_mult: float = 0.005
                ) -> dict[str, InstructionList]:
    """Station feeders on fertile tiles along a march path.
    
    Args:
        path: Hex indices of the march route
        feeders: Squad of available pieces to assign as suppliers
        grid: HexGrid
        elevations: Elevation array
        food_tiers: Per-hex food tier array
        min_tier: Minimum food tier for a candidate station
        max_offset: Max ring distance from path for stations
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList (path + rotate + harvest/give loop)
    """
    pieces = feeders.alive
    if not pieces:
        return {}
    
    path_set = set(path)
    
    # --- Phase 1: Generate scored candidate stations ---
    # candidates[hex_idx] = (facing, tier, covered_path_hexes)
    candidates = {}
    
    for path_hex in path:
        for ring in range(1, max_offset + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, path_hex)
                if idx < 0 or idx in grid.invalidRegion:
                    continue
                if elevations[idx] <= 0:
                    continue
                if idx in path_set:
                    continue
                if int(food_tiers[idx]) < min_tier:
                    continue
                if countries is not None and countries[idx] < 0:
                    continue
                
                facing = _facing_toward(grid, idx, path_hex)
                facing_dir = HexPosition.directions()[facing]
                
                elev = max(0, elevations[idx])
                eff_sight = 3 + elev * elevation_mult
                fov = set(HexPosition.origin().field_of_view( facing_dir, int(eff_sight)))
                
                covered = {ph for ph in path
                           if grid.index_to_hexposition(ph, idx) in fov}
                
                if not covered:
                    continue
                
                tier = int(food_tiers[idx])
                if idx not in candidates or len(covered) > len(candidates[idx][2]):
                    candidates[idx] = (facing, tier, covered)
    
    if not candidates:
        return {}
    
    # --- Phase 2: Greedy set cover → pick station positions ---
    uncovered = set(path)
    selected = []      # [(hex_idx, facing, tier), ...]
    remaining = dict(candidates)
    
    while uncovered and remaining:
        best_idx = max(remaining, key=lambda i: (
            len(remaining[i][2] & uncovered) * remaining[i][1]
        ))
        facing, tier, covered = remaining.pop(best_idx)
        newly_covered = covered & uncovered
        if not newly_covered:
            continue
        selected.append((best_idx, facing, tier))
        uncovered -= newly_covered
    
    if not selected:
        return {}
    
    # --- Phase 3: Hungarian matching (single pathfind per pair) ---
    INF = 1e9
    n_pieces = len(pieces)
    n_stations = len(selected)
    cost = np.full((n_pieces, n_stations), INF)
    paths: dict[tuple[int, int], TroopPath] = {}
    
    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        for j, (pos_idx, _, _) in enumerate(selected):
            tp = piece.pathfind(pos_idx, grid, elevations, countries)
            if not tp:
                continue
            paths[(i, j)] = tp
            cost[i, j] = tp.cost
    
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # --- Phase 4: Build instructions from cached paths ---
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue
        
        piece = pieces[i]
        tp = paths[(i, j)]
        _, facing, _ = selected[j]
        
        rules = tp.to_rules(grid, piece.facing)
        
        # Rotate to face the path
        end_facing = _facing_toward(grid, tp[-2], tp[-1])
        diff = (facing - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        # Station action loop
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return assignments


# --- Board setup ---
board = PieceBoard.hilly(rings=8, radius=18, seed=42)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

# Fake food tiers from elevation
food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if e <= 0:      food_tiers[i] = 0
    elif e < 80:    food_tiers[i] = 6
    elif e < 140:   food_tiers[i] = 5
    elif e < 200:   food_tiers[i] = 3
    elif e < 280:   food_tiers[i] = 1
    else:           food_tiers[i] = 0

# --- Find start and target on the NORTH side (above river) ---
def find_land(offsets):
    for hp in offsets:
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            return idx
    return None

start_idx  = find_land([HexPosition(-5, -3, 8), HexPosition(-4, -3, 7), HexPosition(-3, -2, 5)])
target_idx = find_land([HexPosition(5, -3, -2), HexPosition(4, -3, -1), HexPosition(3, -2, -1)])

print(f"Start: hex {start_idx} (elev={elevs[start_idx]:.0f})")
print(f"Target: hex {target_idx} (elev={elevs[target_idx]:.0f})")

# --- Queen ---
queen = board.add_piece(start_idx, PieceType.QUEEN, facing=4, country_id=1,
                        instructions=InstructionList([]))
queen.food = 3.0
march_path = queen.pathfind(target_idx, grid, elevs)
queen.instructions = queen.pathfind_to_rules(target_idx, grid, elevs)
print(f"March path: {len(march_path)} hexes, {len(queen.instructions.rules)} instructions")

# --- Scatter feeders on the NORTH side ---
feeder_offsets = [
    (-4, -1), (-3, -3), (-1, -2), (0, -3), (1, -1),
    (2, -3), (3, -1), (-2, -1), (4, -2), (-1, -4)
]
feeders = []
for dq, dr in feeder_offsets:
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    p = board.add_piece(idx, PieceType.PAWN, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 5.0
    p.sight = 3
    feeders.append(p)

print(f"{len(feeders)} feeders available")

# --- Wrap in Squad and run supply line ---
feeder_squad = Squad(id=1, name="Supply Corps", pieces=feeders)

orders = supply_line(march_path.hexes, feeder_squad, grid, elevs, food_tiers,
                     min_tier=3, max_offset=2)

print(f"\nAssigned {len(orders)} feeders:")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name}: {len(p.instructions.rules)} instrs")

# --- Render ---
assigned = [p for p in feeders if p.id in orders]
show(NotStr(board.render_with_sight(
    assigned + [queen],
    num_turns=12,
    facing_only=True
)))


In [ ]:
#read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

actually ft stands for fast html not squad fitness

import matplotlib.pyplot as plt

#| export
def SquadVisionOverlay(squads, num_turns=5, facing_only=True,
                       opacity=0.30, color_attr="darkPrimary",
                       elevation_mult=0.005, **kw) -> OverlaySpec:
    """Vision cones + facing bars at each piece's simulated final position.

    Renders underneath SquadPlanOverlay (priority 65 < 70):
      - Dotted hex fill for every hex in each piece's FOV
      - White-on-dark facing bar showing which way each piece ends up looking

    Args:
        squads:         list of Squad objects
        num_turns:      turns to simulate forward
        facing_only:    True = 120° cone, False = full radius
        opacity:        fill opacity for vision hexes
        color_attr:     flag attribute for dot color ("darkPrimary", "primary", "comp")
        elevation_mult: sight bonus per elevation unit

    Context used:
        ctx.grid / ctx.builder  — render grid (may be zoomed)
        ctx.board               — coarse grid for simulation (optional)
        ctx.c2f                 — coarse→fine mapping for zoomed views (optional)
    """

    def render(ctx) -> str:
        grid = ctx.grid
        c2f  = getattr(ctx, 'c2f', None)

        board = getattr(ctx, 'board', None)
        if board is not None:
            sim_grid = board.terrain.hexGrid
            sim_elev = board.terrain.elevations
            sim_ctry = board.terrain.fields.get("country")
        else:
            sim_grid = grid
            sim_elev = ctx.terrain.elevations
            sim_ctry = ctx.terrain.fields.get("country")

        # ── Simulate to collect final positions ──
        color_to_hexes: dict[str, set[int]] = {}
        bars = []  # (render_hex_idx, facing, flag)

        for squad in squads:
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            for piece in pieces:
                if not piece.instructions or not piece.instructions.rules:
                    continue

                steps, _ = piece.plan_region(
                    sim_grid, sim_elev,
                    num_turns=num_turns, padding=0,
                    countries=sim_ctry,
                )
                if not steps:
                    continue

                last = steps[-1]
                color = (getattr(piece.flag, color_attr, "#888")
                         if piece.flag else "#888")

                # Vision from final position (computed on coarse/sim grid)
                elev = max(0, sim_elev[last.hex_idx])
                eff_sight = int(piece.sight + elev * elevation_mult)

                if facing_only:
                    facing_dir = HexPosition.directions()[last.facing % 6]
                    fov = HexPosition.origin().field_of_view(facing_dir, eff_sight)
                    visible = [idx for hp in fov
                               if (idx := sim_grid.hexposition_to_index(hp, last.hex_idx)) >= 0]
                else:
                    visible = sim_grid.indices_in_range(last.hex_idx, eff_sight)

                # Map coarse → fine for zoomed views
                if c2f is not None:
                    visible = [fi for v in visible if (fi := _map_point(v, c2f)) >= 0]
                    render_idx = _map_point(last.hex_idx, c2f)
                else:
                    render_idx = last.hex_idx

                color_to_hexes.setdefault(color, set()).update(visible)

                if 0 <= render_idx < len(grid.hexes):
                    bars.append((render_idx, last.facing, piece.flag))

        # ── Render vision hex fill ──
        parts = []
        r = grid.radius if hasattr(grid, 'radius') else 20
        dot_r   = max(2.5, r * 0.18)
        spacing = max(5.0, r * 0.32)

        for color, hex_indices in color_to_hexes.items():
            pat_id = f"vision_{color.replace('#', '')}"
            pat_svg = (
                f'<pattern id="{pat_id}" x="0" y="0" '
                f'width="{spacing:.1f}" height="{spacing:.1f}" '
                f'patternUnits="userSpaceOnUse">'
                f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
                f'r="{dot_r:.1f}" fill="{color}"/>'
                f'</pattern>')
            ctx.builder.add_definition(SVGDef("", pat_id, pat_svg, raw=True))

            for idx in sorted(hex_indices):
                if idx < 0 or idx >= len(grid.hexes):
                    continue
                h = grid.hexes[idx]
                pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
                parts.append(
                    f'<polygon points="{pts}" '
                    f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                    f'stroke="{color}" stroke-width="0.6" '
                    f'stroke-dasharray="3,2" stroke-opacity="0.4"/>')

        # ── Render facing bars ──
        for render_idx, facing, flag in bars:
            c = grid.hexes[render_idx].center
            g_out = DiagramGlyphs(color="#111111", size=r * 0.7,
                                  stroke_width=5.0, opacity=0.85)
            g_bar = DiagramGlyphs(color="#FFFFFF", size=r * 0.7,
                                  stroke_width=2.8, opacity=1.0)
            parts.append(g_out.facing_bar(c.x, c.y, facing))
            parts.append(g_bar.facing_bar(c.x, c.y, facing))

        return '\n'.join(parts)

    return OverlaySpec("squad_vision", render, priority=65)


Turns versus moves.
turns are synced across all pieces. different pieces will have different levels of moves. a queen might have 8 moves while a pawn might have 2.
Each turn can be thought of as having m number of moves (which is the largest move_strength of any alive piece)

some instructions don't have move cost (ie rotate) so they don't count

a turn would be

| turn  | move | Queen | Pawn |
|-------|------|-------|------|
| 0     | 0    | 1 - pause    | 1 - harvest    |
| 0     | 1    | 2  - pause   | 1 - share   |
| 0     | 2    | 3  - move   | 0    |
| 0     | 3    | 4  - move   | 0    |
| 0     | 4    | 5  - attack   | 0    |
| 0     | 5    | 6   - attack   | 0    |
| 0     | 6    | 7  - move (back)   | 0    |
| 0     | 7    | 1   - move (back)  | 0    |
| 1     | 0    | 8     | 2    |

This would let the queen go forward two place attack and retreat in time so she gets fed the next round.




In [ ]:
class SimEventType(Enum):
    MOVED     = "moved"
    BLOCKED   = "blocked"
    ROTATED   = "rotated"
    HARVESTED = "harvested"
    GAVE      = "gave"
    ATE       = "ate"
    STARVED   = "starved"
    DIED      = "died"

_FREE = {Instruction.ROT_L, Instruction.ROT_R}
_COSTS_ONE = {Instruction.PAUSE, Instruction.DEFEND,
              Instruction.HARVEST, Instruction.GIVE, Instruction.SETTLE}

@dataclass
class FoodSimulator:
    """Turn-based simulator: move → gather → share → eat."""
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    pieces: list                           # list[Piece]
    countries: np.ndarray = None
    elevation_mult: float = 0.005

    _rows: list = field(default_factory=list)
    turn: int = 0
    snapshots: list = field(default_factory=list)

    # ── helpers ──────────────────────────────────────────────

    @property
    def alive(self):
        return [p for p in self.pieces if p.health > 0]

    @property
    def occupied(self):
        return {p.location for p in self.alive
                if p.location is not None and p.location >= 0}

    @property
    def df(self) -> pd.DataFrame:
        return pd.DataFrame(self._rows)

    def _log(self, piece, etype: SimEventType, *,
             src_hex=None, dst_hex=None,
             dst_piece_id=None, amount=0.0):
        loc = piece.location if piece.location is not None else -1
        self._rows.append({
            'turn':         self.turn,
            'event':        etype.value,
            'piece_id':     piece.id,
            'piece_name':   piece.name,
            'src_hex':      src_hex if src_hex is not None else loc,
            'dst_hex':      dst_hex if dst_hex is not None else loc,
            'dst_piece_id': dst_piece_id,
            'amount':       amount,
        })

    def _sort_move(self, pieces):
        """Rank → oldest → healthiest → random."""
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99),
            p.birth_year,
            -p.health,
            random.random()))

    def _sort_share(self, pieces):
        """Rank → oldest → least food → random."""
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99),
            p.birth_year,
            p.food,
            random.random()))

    # ── move phase ───────────────────────────────────────────

    def _move_one(self, piece, occupied):
        il = piece.instructions
        if not il.rules:
            return []

        executed = []
        budget = float(piece.move_strength)

        while budget > 0:
            if il.cursor >= len(il.rules):
                if il.patrol:
                    il.cursor = 0
                else:
                    break

            instr = Instruction(il.rules[il.cursor])

            # ── free rotations ──
            if instr in _FREE:
                piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                                else (piece.facing + 1)) % 6
                executed.append(instr)
                self._log(piece, SimEventType.ROTATED)
                il.cursor += 1
                continue

            # ── cost-1 non-movement ──
            if instr in _COSTS_ONE:
                budget -= 1.0
                executed.append(instr)
                il.cursor += 1
                continue

            # ── FORWARD ──
            auto_rots = 0
            saved_fac = piece.facing
            moved = False

            while auto_rots <= 5:
                d = HexPosition.directions()[piece.facing % 6]
                nbr = self.grid.hexposition_to_index(d, piece.location)

                passable = (0 <= nbr < len(self.elevations)
                            and nbr not in self.grid.invalidRegion
                            and self.elevations[nbr] >= 1)
                if self.countries is not None and passable:
                    passable = self.countries[nbr] >= 0

                if not passable:
                    piece.facing = (piece.facing - 1) % 6
                    auto_rots += 1
                    continue

                # Occupied → freeze cursor, end turn
                if nbr in occupied and nbr != piece.location:
                    self._log(piece, SimEventType.BLOCKED, dst_hex=nbr)
                    moved = True
                    budget = 0
                    break

                # Can afford?
                cost = _move_cost(self.elevations, piece.location, nbr)
                if cost > budget:
                    budget = 0
                    moved = True
                    break

                # Move
                src = piece.location
                occupied.discard(src)
                piece.location = nbr
                occupied.add(nbr)
                budget -= cost
                executed.append(instr)
                self._log(piece, SimEventType.MOVED,
                          src_hex=src, dst_hex=nbr, amount=cost)
                il.cursor += 1
                moved = True
                break

            if not moved:
                piece.facing = saved_fac
                self._log(piece, SimEventType.BLOCKED)
                budget = 0

        return executed

    def _move_phase(self):
        occupied = self.occupied
        order = self._sort_move(self.alive)
        executed = {}
        for piece in order:
            executed[piece.id] = self._move_one(piece, occupied)
        return executed

    # ── gather phase ─────────────────────────────────────────

    def _gather_phase(self, executed):
        for piece in self.alive:
            if Instruction.HARVEST in executed.get(piece.id, []):
                loc = piece.location
                if loc is None or loc < 0:
                    continue
                tier = int(self.food_tiers[loc])
                gained = piece.harvest_strength * tier
                old = piece.food
                piece.food = min(piece.food + gained, piece.food_capacity)
                self._log(piece, SimEventType.HARVESTED,
                          amount=piece.food - old)

    # ── share phase ──────────────────────────────────────────

    def _share_phase(self, executed):
        givers = [p for p in self.alive
                  if Instruction.GIVE in executed.get(p.id, [])]

        for giver in self._sort_share(givers):
            reserve = FOOD_RESERVE_TURNS * giver.diet
            surplus = max(0.0, giver.food - reserve)
            if surplus <= 0:
                continue

            visible = set(giver.hexes_in_sight(
                self.grid, self.elevations,
                elevation_mult=self.elevation_mult,
                facing_only=True))

            receivers = sorted(
                [p for p in self.alive
                 if p is not giver
                 and p.owner_id == giver.owner_id
                 and p.food < p.food_capacity
                 and p.location in visible],
                key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

            for recv in receivers:
                if surplus <= 0:
                    break
                amount = min(surplus, recv.food_capacity - recv.food)
                recv.food += amount
                giver.food -= amount
                surplus -= amount
                self._log(giver, SimEventType.GAVE,
                          dst_piece_id=recv.id, amount=amount)

    # ── eat phase ────────────────────────────────────────────

    def _eat_phase(self, executed):
        for piece in list(self.pieces):
            if piece.health <= 0:
                continue
            if piece.id not in executed or not executed[piece.id]:
                continue

            shortfall = max(0.0, piece.diet - piece.food)
            piece.food = max(0.0, piece.food - piece.diet)
            self._log(piece, SimEventType.ATE, amount=piece.diet)

            if shortfall > 0:
                damage = shortfall * STARVE_MULT
                piece.health -= damage
                self._log(piece, SimEventType.STARVED, amount=damage)

            if piece.health <= 0:
                piece.health = 0
                self._log(piece, SimEventType.DIED)

    # ── snapshot ─────────────────────────────────────────────

    def _snapshot(self):
        alive = self.alive
        self.snapshots.append({
            'turn': self.turn,
            'alive': len(alive),
            'total': len(self.pieces),
            'avg_health': np.mean([p.health for p in alive]) if alive else 0,
            'min_health': min((p.health for p in alive), default=0),
            'avg_food': np.mean([p.food for p in alive]) if alive else 0,
            'avg_turns_left': np.mean([p.food / max(p.diet, 0.01)
                                       for p in alive]) if alive else 0,
        })

    # ── turn loop ────────────────────────────────────────────

    def step(self):
        executed = self._move_phase()
        self._gather_phase(executed)
        self._share_phase(executed)
        self._eat_phase(executed)
        self._snapshot()
        self.turn += 1

    def run(self, num_turns=20):
        for _ in range(num_turns):
            if not self.alive:
                self._snapshot()
                break
            self.step()
        return self

    def summary(self):
        a = self.alive
        print(f"Turn {self.turn}: {len(a)}/{len(self.pieces)} alive")
        if a:
            print(f"  Health: avg={np.mean([p.health for p in a]):.0f} "
                  f"min={min(p.health for p in a):.0f}")
            print(f"  Food:   avg={np.mean([p.food for p in a]):.1f} "
                  f"turns={np.mean([p.food/max(p.diet,.01) for p in a]):.1f}")


In [ ]:
@patch
def _eat_phase(self:FoodSimulator, executed):
    for piece in list(self.pieces):
        if piece.health <= 0:
            continue
        if piece.id not in executed or not executed[piece.id]:
            continue
            
        shortfall = max(0.0, piece.diet - piece.food)
        piece.food = max(0.0, piece.food - piece.diet)
        self._log(piece, SimEventType.ATE, amount=piece.diet)

        if shortfall > 0:
            damage = shortfall * STARVE_MULT
            piece.health -= damage
            self._log(piece, SimEventType.STARVED, amount=damage)

        if piece.health <= 0:
            piece.health = 0
            self._log(piece, SimEventType.DIED)


In [ ]:
@patch
def step(self:FoodSimulator):
    executed = self._move_phase()
    self._gather_phase(executed)
    self._share_phase(executed)
    self._eat_phase(executed)      # ← now gated
    self._snapshot()
    self.turn += 1

In [ ]:
@patch
def _snapshot(self: FoodSimulator):
    alive = self.alive
    self.snapshots.append({
        'turn': self.turn,
        'alive': len(alive),
        'total': len(self.pieces),
        'avg_health': np.mean([p.health for p in alive]) if alive else 0,
        'min_health': min((p.health for p in alive), default=0),
        'avg_food': np.mean([p.food for p in alive]) if alive else 0,
        'avg_turns_left': np.mean([p.food / max(p.diet, 0.01)
                                   for p in alive]) if alive else 0,
        'piece_states': {
            p.id: {'location': p.location, 'food': p.food,
                    'food_capacity': p.food_capacity, 'health': p.health}
            for p in self.pieces
        },
    })


In [ ]:
@patch
def __ft__(self: FoodSimulator):
    """Line charts: alive count, health, food turns remaining."""
    if not self.snapshots:
        return P("No turns simulated yet", cls="text-sm opacity-50")

    df = pd.DataFrame(self.snapshots)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3))

    # Alive
    axes[0].plot(df['turn'], df['alive'], 'o-', color='#2ecc71', lw=2, ms=4)
    axes[0].axhline(df['total'].iloc[0], color='#bbb', ls='--', lw=1)
    axes[0].set_title('Alive', fontweight='bold', fontsize=10)
    axes[0].set_xlabel('Turn'); axes[0].set_ylim(bottom=0)
    axes[0].fill_between(df['turn'], df['alive'], alpha=0.15, color='#2ecc71')

    # Health
    axes[1].plot(df['turn'], df['avg_health'], 'o-', color='#e74c3c', lw=2, ms=4, label='avg')
    axes[1].plot(df['turn'], df['min_health'], 's--', color='#c0392b', lw=1, ms=3, alpha=0.6, label='min')
    axes[1].set_title('Health', fontweight='bold', fontsize=10)
    axes[1].set_xlabel('Turn'); axes[1].legend(fontsize=8); axes[1].set_ylim(bottom=0)

    # Food turns remaining
    axes[2].plot(df['turn'], df['avg_turns_left'], 'o-', color='#f39c12', lw=2, ms=4)
    axes[2].axhline(FOOD_RESERVE_TURNS, color='#e67e22', ls=':', lw=1, label=f'reserve ({FOOD_RESERVE_TURNS})')
    axes[2].set_title('Food (turns left)', fontweight='bold', fontsize=10)
    axes[2].set_xlabel('Turn'); axes[2].legend(fontsize=8); axes[2].set_ylim(bottom=0)

    for ax in axes:
        ax.grid(axis='y', alpha=0.3, lw=0.5)
        ax.tick_params(labelsize=8)

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Simulated {self.turn} turns — {len(self.alive)}/{len(self.pieces)} alive",
          cls="text-xs opacity-60 text-center"),
        cls="space-y-1")

In [ ]:
show(sim)

So I think we need to refactor our food simular on this turn versus move idea.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great insight. The current sim treats a turn as "each piece burns its whole budget, then we do food." The move-slot model interleaves execution so fast pieces and slow pieces share the same timeline within a turn. That's what makes the "queen rushes out, attacks, retreats, gets fed" pattern work.

Here's the refactor plan:

**Current flow:**
```
turn → piece A burns all budget → piece B burns all budget → harvest → give → eat
```

**New flow:**
```
turn → [move_slot 0: each piece executes 1 instruction]
     → [move_slot 1: each piece executes 1 instruction]
     → ...
     → [move_slot m-1: ...]
     → eat phase
```

Key changes:

1. **`_tick_one(piece, budget)`** — execute a single cost-bearing instruction (consuming free rotations along the way). Returns `(instruction, cost_spent)` or `(None, 0)`.

2. **HARVEST and GIVE take immediate effect** during the move slot they execute — no separate gather/share phase. This means a pawn that HARVESTs at slot 0 and GIVEs at slot 1 delivers food *within* the same turn, before the eat phase.

3. **`step()`** computes `max_moves = max(p.move_strength for p in alive)`, then loops `max_moves` times. Each slot, every piece with remaining budget gets one tick.

4. **EAT stays at end of turn** — once per turn, every piece that acted pays its diet.

```python
class FoodSimulator:
    # ... existing fields ...

    def _tick_one(self, piece, budget, occupied):
        """Execute one cost-bearing instruction. Returns (instr, cost)."""
        il = piece.instructions
        if not il.rules:
            return None, 0.0

        while True:
            if il.cursor >= len(il.rules):
                if il.patrol: il.cursor = 0
                else: return None, 0.0

            instr = Instruction(il.rules[il.cursor])

            # Free rotations — chain through without spending budget
            if instr in _FREE:
                piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                                else (piece.facing + 1)) % 6
                self._log(piece, SimEventType.ROTATED)
                il.cursor += 1
                continue

            # Cost-1 actions (PAUSE, DEFEND, HARVEST, GIVE, SETTLE)
            if instr in _COSTS_ONE:
                if budget < 1.0:
                    return None, 0.0
                il.cursor += 1
                # Immediate effect
                if instr == Instruction.HARVEST:
                    self._do_harvest(piece)
                elif instr == Instruction.GIVE:
                    self._do_give(piece)
                self._log_action(piece, instr)
                return instr, 1.0

            # FORWARD — same wall-hugging logic as before
            cost = self._try_forward(piece, occupied)
            if cost > 0:
                return Instruction.FORWARD, cost
            return None, 0.0

    def step(self):
        alive = self.alive
        if not alive: return

        max_moves = max(p.move_strength for p in alive)
        budgets = {p.id: float(p.move_strength) for p in alive}
        executed = {p.id: [] for p in alive}
        occupied = self.occupied

        for slot in range(max_moves):
            for piece in self._sort_move(alive):
                if budgets[piece.id] <= 0:
                    continue
                instr, cost = self._tick_one(piece, budgets[piece.id], occupied)
                if instr is not None:
                    budgets[piece.id] -= cost
                    executed[piece.id].append(instr)

        self._eat_phase(executed)
        self._snapshot()
        self.turn += 1
```

The `_do_harvest` and `_do_give` are just the immediate-effect versions of what's currently in `_gather_phase` and `_share_phase`, extracted into single-piece methods.

Does this shape look right to you? Two things to decide before I write the full version:

1. **Move ordering within a slot** — currently `_sort_move` (rank → oldest → healthiest). Same order every slot, or should it alternate/randomize?
2. **GIVE visibility** — should `_do_give` check the piece's facing cone at the *moment* of giving (mid-turn, after it may have moved/rotated), or at its position at start of turn?

Perfect — that's clean and intuitive. Within each slot the queen acts first, so she can move forward *and* the pawn she just left behind can GIVE to someone else in the same slot. Here's the full refactor:

```python
class FoodSimulator:
    """Turn-based simulator with per-slot interleaved execution."""
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    pieces: list                           # list[Piece]
    countries: np.ndarray = None
    elevation_mult: float = 0.005

    _rows: list = field(default_factory=list)
    turn: int = 0
    snapshots: list = field(default_factory=list)

    # ── helpers ──────────────────────────────────────────────

    @property
    def alive(self):
        return [p for p in self.pieces if p.health > 0]

    @property
    def occupied(self):
        return {p.location for p in self.alive
                if p.location is not None and p.location >= 0}

    @property
    def df(self) -> pd.DataFrame:
        return pd.DataFrame(self._rows)

    def _log(self, piece, etype: SimEventType, *,
             slot=None, src_hex=None, dst_hex=None,
             dst_piece_id=None, amount=0.0):
        loc = piece.location if piece.location is not None else -1
        self._rows.append({
            'turn':         self.turn,
            'slot':         slot,
            'event':        etype.value,
            'piece_id':     piece.id,
            'piece_name':   piece.name,
            'src_hex':      src_hex if src_hex is not None else loc,
            'dst_hex':      dst_hex if dst_hex is not None else loc,
            'dst_piece_id': dst_piece_id,
            'amount':       amount,
        })

    def _sort_move(self, pieces):
        """Rank → oldest → healthiest → deterministic tiebreak."""
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99),
            p.birth_year,
            -p.health,
            p.id))

    # ── immediate-effect actions ─────────────────────────────

    def _do_harvest(self, piece, slot=None):
        loc = piece.location
        if loc is None or loc < 0:
            return
        tier = int(self.food_tiers[loc])
        gained = piece.harvest_strength * tier
        old = piece.food
        piece.food = min(piece.food + gained, piece.food_capacity)
        self._log(piece, SimEventType.HARVESTED, slot=slot,
                  amount=piece.food - old)

    def _do_give(self, piece, slot=None):
        reserve = FOOD_RESERVE_TURNS * piece.diet
        surplus = max(0.0, piece.food - reserve)
        if surplus <= 0:
            return

        # Check cone at THIS moment (mid-turn position + facing)
        visible = set(piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=self.elevation_mult,
            facing_only=True))

        receivers = sorted(
            [p for p in self.alive
             if p is not piece
             and p.owner_id == piece.owner_id
             and p.food < p.food_capacity
             and p.location in visible],
            key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

        for recv in receivers:
            if surplus <= 0:
                break
            amount = min(surplus, recv.food_capacity - recv.food)
            recv.food += amount
            piece.food -= amount
            surplus -= amount
            self._log(piece, SimEventType.GAVE, slot=slot,
                      dst_piece_id=recv.id, amount=amount)

    # ── single-tick execution ────────────────────────────────

    def _try_forward(self, piece, occupied, slot=None):
        """Attempt FORWARD with wall-hugging. Returns cost spent (0 if blocked)."""
        auto_rots = 0
        saved_fac = piece.facing

        while auto_rots <= 5:
            d = HexPosition.directions()[piece.facing % 6]
            nbr = self.grid.hexposition_to_index(d, piece.location)

            passable = (0 <= nbr < len(self.elevations)
                        and nbr not in self.grid.invalidRegion
                        and self.elevations[nbr] >= 1)
            if self.countries is not None and passable:
                passable = self.countries[nbr] >= 0

            if not passable:
                piece.facing = (piece.facing - 1) % 6
                auto_rots += 1
                continue

            # Occupied → blocked, budget consumed but no move
            if nbr in occupied and nbr != piece.location:
                self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
                return 1.0  # costs budget to bump into someone

            cost = _move_cost(self.elevations, piece.location, nbr)
            src = piece.location
            occupied.discard(src)
            piece.location = nbr
            occupied.add(nbr)
            self._log(piece, SimEventType.MOVED, slot=slot,
                      src_hex=src, dst_hex=nbr, amount=cost)
            return cost

        # Totally walled in
        piece.facing = saved_fac
        self._log(piece, SimEventType.BLOCKED, slot=slot)
        return 0.0

    def _tick_one(self, piece, budget, occupied, slot=None):
        """Execute one cost-bearing instruction (chaining free rotations).
        Returns (instruction_or_None, cost_spent)."""
        il = piece.instructions
        if not il.rules:
            return None, 0.0

        while True:
            if il.cursor >= len(il.rules):
                if il.patrol:
                    il.cursor = 0
                else:
                    return None, 0.0

            instr = Instruction(il.rules[il.cursor])

            # Free rotations — chain without spending budget
            if instr in _FREE:
                piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                                else (piece.facing + 1)) % 6
                self._log(piece, SimEventType.ROTATED, slot=slot)
                il.cursor += 1
                continue

            # Cost-1 non-movement
            if instr in _COSTS_ONE:
                if budget < 1.0:
                    return None, 0.0
                il.cursor += 1
                if instr == Instruction.HARVEST:
                    self._do_harvest(piece, slot)
                elif instr == Instruction.GIVE:
                    self._do_give(piece, slot)
                else:
                    self._log(piece, SimEventType(instr.name.lower()
                              if instr.name.lower() in {e.value for e in SimEventType}
                              else 'moved'), slot=slot)
                return instr, 1.0

            # FORWARD
            if budget < 1.0:
                return None, 0.0
            cost = self._try_forward(piece, occupied, slot)
            if cost > 0:
                if budget >= cost:
                    il.cursor += 1
                    return Instruction.FORWARD, cost
                else:
                    return None, 0.0  # can't afford uphill
            return None, 0.0

    # ── eat phase (once per turn, end) ───────────────────────

    def _eat_phase(self, acted_ids: set):
        for piece in list(self.pieces):
            if piece.health <= 0:
                continue
            if piece.id not in acted_ids:
                continue

            shortfall = max(0.0, piece.diet - piece.food)
            piece.food = max(0.0, piece.food - piece.diet)
            self._log(piece, SimEventType.ATE, amount=piece.diet)

            if shortfall > 0:
                damage = shortfall * STARVE_MULT
                piece.health -= damage
                self._log(piece, SimEventType.STARVED, amount=damage)

            if piece.health <= 0:
                piece.health = 0
                self._log(piece, SimEventType.DIED)

    # ── snapshot ─────────────────────────────────────────────

    def _snapshot(self):
        alive = self.alive
        self.snapshots.append({
            'turn': self.turn,
            'alive': len(alive),
            'total': len(self.pieces),
            'avg_health': np.mean([p.health for p in alive]) if alive else 0,
            'min_health': min((p.health for p in alive), default=0),
            'avg_food': np.mean([p.food for p in alive]) if alive else 0,
            'avg_turns_left': np.mean([p.food / max(p.diet, 0.01)
                                       for p in alive]) if alive else 0,
            'piece_states': {
                p.id: {'location': p.location, 'food': p.food,
                        'food_capacity': p.food_capacity, 'health': p.health}
                for p in self.pieces
            },
        })

    # ── turn loop ────────────────────────────────────────────

    def step(self):
        alive = self.alive
        if not alive:
            return

        max_slots = max(p.move_strength for p in alive)
        budgets = {p.id: float(p.move_strength) for p in alive}
        acted = set()
        occupied = self.occupied
        order = self._sort_move(alive)  # same order every slot

        for slot in range(max_slots):
            for piece in order:
                if piece.health <= 0:
                    continue
                if budgets[piece.id] <= 0:
                    continue
                instr, cost = self._tick_one(
                    piece, budgets[piece.id], occupied, slot=slot)
                if instr is not None:
                    budgets[piece.id] -= cost
                    acted.add(piece.id)

        self._eat_phase(acted)
        self._snapshot()
        self.turn += 1

    def run(self, num_turns=20):
        for _ in range(num_turns):
            if not self.alive:
                self._snapshot()
                break
            self.step()
        return self

    def summary(self):
        a = self.alive
        print(f"Turn {self.turn}: {len(a)}/{len(self.pieces)} alive")
        if a:
            print(f"  Health: avg={np.mean([p.health for p in a]):.0f} "
                  f"min={min(p.health for p in a):.0f}")
            print(f"  Food:   avg={np.mean([p.food for p in a]):.1f} "
                  f"turns={np.mean([p.food/max(p.diet,.01) for p in a]):.1f}")
```

The key differences from the old version:

- **`step()` loops slots, not pieces** — slot 0 processes queen then pawn, slot 1 processes queen then pawn again, etc. Queen's move_strength=5 means she gets 5 ticks; pawn's move_strength=2 means the pawn is idle for slots 2–4.
- **HARVEST and GIVE fire immediately** inside `_tick_one` — so a pawn doing `[HARVEST, GIVE]` harvests at slot 0 and gives at slot 1, mid-turn. The queen might have already moved into the cone by then (or out of it).
- **`_do_give` checks the cone at the moment of execution** — after any rotations and movements that happened earlier in the same slot or prior slots.
- **EAT is once at turn end** — only pieces that executed at least one instruction pay their diet. The `acted` set tracks this.
- **Logs include `slot`** — so you can replay the exact interleaving sequence.

The `_try_forward` blocked-by-occupied case now costs 1 budget (you spend effort bumping into them) but doesn't advance the cursor — the piece will retry FORWARD next slot when the hex might be clear.

In [ ]:
class FoodSimulator:
    """Turn-based simulator with per-slot interleaved execution."""
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    pieces: list                           # list[Piece]
    countries: np.ndarray = None
    elevation_mult: float = 0.005

    _rows: list = field(default_factory=list)
    turn: int = 0
    snapshots: list = field(default_factory=list)

    # ── helpers ──────────────────────────────────────────────

    @property
    def alive(self):
        return [p for p in self.pieces if p.health > 0]

    @property
    def occupied(self):
        return {p.location for p in self.alive
                if p.location is not None and p.location >= 0}

    @property
    def df(self) -> pd.DataFrame:
        return pd.DataFrame(self._rows)

    def _log(self, piece, etype: SimEventType, *,
             slot=None, src_hex=None, dst_hex=None,
             dst_piece_id=None, amount=0.0):
        loc = piece.location if piece.location is not None else -1
        self._rows.append({
            'turn':         self.turn,
            'slot':         slot,
            'event':        etype.value,
            'piece_id':     piece.id,
            'piece_name':   piece.name,
            'src_hex':      src_hex if src_hex is not None else loc,
            'dst_hex':      dst_hex if dst_hex is not None else loc,
            'dst_piece_id': dst_piece_id,
            'amount':       amount,
        })

    def _sort_move(self, pieces):
        """Rank → oldest → healthiest → deterministic tiebreak."""
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99),
            p.birth_year,
            -p.health,
            p.id))

    # ── immediate-effect actions ─────────────────────────────

    def _do_harvest(self, piece, slot=None):
        loc = piece.location
        if loc is None or loc < 0:
            return
        tier = int(self.food_tiers[loc])
        gained = piece.harvest_strength * tier
        old = piece.food
        piece.food = min(piece.food + gained, piece.food_capacity)
        self._log(piece, SimEventType.HARVESTED, slot=slot,
                  amount=piece.food - old)

    def _do_give(self, piece, slot=None):
        reserve = FOOD_RESERVE_TURNS * piece.diet
        surplus = max(0.0, piece.food - reserve)
        if surplus <= 0:
            return

        # Check cone at THIS moment (mid-turn position + facing)
        visible = set(piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=self.elevation_mult,
            facing_only=True))

        receivers = sorted(
            [p for p in self.alive
             if p is not piece
             and p.owner_id == piece.owner_id
             and p.food < p.food_capacity
             and p.location in visible],
            key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

        for recv in receivers:
            if surplus <= 0:
                break
            amount = min(surplus, recv.food_capacity - recv.food)
            recv.food += amount
            piece.food -= amount
            surplus -= amount
            self._log(piece, SimEventType.GAVE, slot=slot,
                      dst_piece_id=recv.id, amount=amount)

    # ── single-tick execution ────────────────────────────────

    def _try_forward(self, piece, occupied, slot=None):
        """Attempt FORWARD with wall-hugging. Returns cost spent (0 if blocked)."""
        auto_rots = 0
        saved_fac = piece.facing

        while auto_rots <= 5:
            d = HexPosition.directions()[piece.facing % 6]
            nbr = self.grid.hexposition_to_index(d, piece.location)

            passable = (0 <= nbr < len(self.elevations)
                        and nbr not in self.grid.invalidRegion
                        and self.elevations[nbr] >= 1)
            if self.countries is not None and passable:
                passable = self.countries[nbr] >= 0

            if not passable:
                piece.facing = (piece.facing - 1) % 6
                auto_rots += 1
                continue

            # Occupied → blocked, budget consumed but no move
            if nbr in occupied and nbr != piece.location:
                self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
                return 1.0  # costs budget to bump into someone

            cost = _move_cost(self.elevations, piece.location, nbr)
            src = piece.location
            occupied.discard(src)
            piece.location = nbr
            occupied.add(nbr)
            self._log(piece, SimEventType.MOVED, slot=slot,
                      src_hex=src, dst_hex=nbr, amount=cost)
            return cost

        # Totally walled in
        piece.facing = saved_fac
        self._log(piece, SimEventType.BLOCKED, slot=slot)
        return 0.0

    def _tick_one(self, piece, budget, occupied, slot=None):
        """Execute one cost-bearing instruction (chaining free rotations).
        Returns (instruction_or_None, cost_spent)."""
        il = piece.instructions
        if not il.rules:
            return None, 0.0

        while True:
            if il.cursor >= len(il.rules):
                if il.patrol:
                    il.cursor = 0
                else:
                    return None, 0.0

            instr = Instruction(il.rules[il.cursor])

            # Free rotations — chain without spending budget
            if instr in _FREE:
                piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                                else (piece.facing + 1)) % 6
                self._log(piece, SimEventType.ROTATED, slot=slot)
                il.cursor += 1
                continue

            # Cost-1 non-movement
            if instr in _COSTS_ONE:
                if budget < 1.0:
                    return None, 0.0
                il.cursor += 1
                if instr == Instruction.HARVEST:
                    self._do_harvest(piece, slot)
                elif instr == Instruction.GIVE:
                    self._do_give(piece, slot)
                else:
                    self._log(piece, SimEventType(instr.name.lower()
                              if instr.name.lower() in {e.value for e in SimEventType}
                              else 'moved'), slot=slot)
                return instr, 1.0

            # FORWARD
            if budget < 1.0:
                return None, 0.0
            cost = self._try_forward(piece, occupied, slot)
            if cost > 0:
                if budget >= cost:
                    il.cursor += 1
                    return Instruction.FORWARD, cost
                else:
                    return None, 0.0  # can't afford uphill
            return None, 0.0

    # ── eat phase (once per turn, end) ───────────────────────

    def _eat_phase(self, acted_ids: set):
        for piece in list(self.pieces):
            if piece.health <= 0:
                continue
            if piece.id not in acted_ids:
                continue

            shortfall = max(0.0, piece.diet - piece.food)
            piece.food = max(0.0, piece.food - piece.diet)
            self._log(piece, SimEventType.ATE, amount=piece.diet)

            if shortfall > 0:
                damage = shortfall * STARVE_MULT
                piece.health -= damage
                self._log(piece, SimEventType.STARVED, amount=damage)

            if piece.health <= 0:
                piece.health = 0
                self._log(piece, SimEventType.DIED)

    # ── snapshot ─────────────────────────────────────────────

    def _snapshot(self):
        alive = self.alive
        self.snapshots.append({
            'turn': self.turn,
            'alive': len(alive),
            'total': len(self.pieces),
            'avg_health': np.mean([p.health for p in alive]) if alive else 0,
            'min_health': min((p.health for p in alive), default=0),
            'avg_food': np.mean([p.food for p in alive]) if alive else 0,
            'avg_turns_left': np.mean([p.food / max(p.diet, 0.01)
                                       for p in alive]) if alive else 0,
            'piece_states': {
                p.id: {'location': p.location, 'food': p.food,
                        'food_capacity': p.food_capacity, 'health': p.health}
                for p in self.pieces
            },
        })

    # ── turn loop ────────────────────────────────────────────

    def step(self):
        alive = self.alive
        if not alive:
            return

        max_slots = max(p.move_strength for p in alive)
        budgets = {p.id: float(p.move_strength) for p in alive}
        acted = set()
        occupied = self.occupied
        order = self._sort_move(alive)  # same order every slot

        for slot in range(max_slots):
            for piece in order:
                if piece.health <= 0:
                    continue
                if budgets[piece.id] <= 0:
                    continue
                instr, cost = self._tick_one(
                    piece, budgets[piece.id], occupied, slot=slot)
                if instr is not None:
                    budgets[piece.id] -= cost
                    acted.add(piece.id)

        self._eat_phase(acted)
        self._snapshot()
        self.turn += 1

    def run(self, num_turns=20):
        for _ in range(num_turns):
            if not self.alive:
                self._snapshot()
                break
            self.step()
        return self

    def summary(self):
        a = self.alive
        print(f"Turn {self.turn}: {len(a)}/{len(self.pieces)} alive")
        if a:
            print(f"  Health: avg={np.mean([p.health for p in a]):.0f} "
                  f"min={min(p.health for p in a):.0f}")
            print(f"  Food:   avg={np.mean([p.food for p in a]):.1f} "
                  f"turns={np.mean([p.food/max(p.diet,.01) for p in a]):.1f}")


In [ ]:
#| export
def SquadFoodTrailOverlay(sim: 'FoodSimulator', squad: Squad = None,
                          min_opacity: float = 0.06,
                          max_opacity: float = 0.65,
                          sim_grid: HexGrid = None,
                          **kw) -> OverlaySpec:
    """Temporal food trail — pip circles sized by food %, arrows for shares, X for deaths.

    Opacity fades forward: earliest turns are faintest, most recent are boldest.
    Uses DiagramGlyphs for consistent styling with the rest of the overlay system.
    """

    def render(ctx) -> str:
        grid  = ctx.grid
        c2f   = getattr(ctx, 'c2f', None)
        board = getattr(ctx, 'board', None)

        piece_ids = ({p.id for p in squad.pieces} if squad
                     else {p.id for p in sim.pieces})

        snaps = sim.snapshots
        if not snaps:
            return ''
        max_turn = max(s['turn'] for s in snaps) or 1

        # Event log
        df = sim.df
        gives  = df[(df.event == 'gave')  & (df.piece_id.isin(piece_ids))]
        deaths = df[(df.event == 'died')  & (df.piece_id.isin(piece_ids))]

        r = grid.radius if hasattr(grid, 'radius') else 20

        # Build one DiagramGlyphs per flag we encounter, register styles once
        flag_glyphs: dict = {}
        def _get_glyphs(flag):
            key = id(flag) if flag else 'default'
            if key not in flag_glyphs:
                g = DiagramGlyphs(flag or CountryFlag.seaborn("husl", 1)[0], size=r * 0.7)
                g.register_styles(ctx.builder)
                # Register pip styles for food coloring
                for pct in range(0, 101, 5):
                    f = pct / 100
                    color = StyleCSS.lerp_color('#e74c3c', '#27ae60', f)
                    ctx.builder.add_style(StyleCSS(
                        f"pip_{pct}", fill=color, stroke=color,
                        stroke_width=0.8, opacity=0.85))
                flag_glyphs[key] = g
            return flag_glyphs[key]

        # Map piece_id → flag for glyph lookup
        piece_flags = {}
        src_pieces = squad.pieces if squad else sim.pieces
        for p in src_pieces:
            piece_flags[p.id] = p.flag

        def _render_idx(loc):
            ri = _map_point(loc, c2f) if c2f else loc
            return ri if 0 <= ri < len(grid.hexes) else -1

        parts = []

        # ── Per-turn food pips + share arrows ──
        for snap in snaps:
            t = snap['turn']
            alpha = min_opacity + (max_opacity - min_opacity) * (t / max_turn)
            states = snap.get('piece_states', {})

            # Food pips
            for pid, st in states.items():
                if pid not in piece_ids or st['health'] <= 0:
                    continue
                ri = _render_idx(st['location'])
                if ri < 0:
                    continue

                c = grid.hexes[ri].center
                cap = max(st['food_capacity'], 0.01)
                frac = np.clip(st['food'] / cap, 0, 1)

                glyphs = _get_glyphs(piece_flags.get(pid))
                pip_svg = glyphs.food_pip(MapCord(c.x, c.y), float(frac))
                parts.append(f'<g opacity="{alpha:.2f}">{pip_svg}</g>')

            # Give arrows for this turn
            turn_gives = gives[gives.turn == t]
            for _, row in turn_gives.iterrows():
                g_st = states.get(row.piece_id)
                r_st = states.get(row.dst_piece_id)
                if not g_st or not r_st:
                    continue
                gi = _render_idx(g_st['location'])
                ri2 = _render_idx(r_st['location'])
                if gi < 0 or ri2 < 0 or gi == ri2:
                    continue

                gc = grid.hexes[gi].center
                rc = grid.hexes[ri2].center
                glyphs = _get_glyphs(piece_flags.get(row.piece_id))
                arrow_svg = glyphs.transfer_arrow(
                    MapCord(gc.x, gc.y), MapCord(rc.x, rc.y))
                parts.append(f'<g opacity="{alpha:.2f}">{arrow_svg}</g>')

        # ── Death X markers (always prominent) ──
        for _, row in deaths.iterrows():
            ri = _render_idx(int(row.src_hex))
            if ri < 0:
                continue
            c = grid.hexes[ri].center
            glyphs = _get_glyphs(piece_flags.get(row.piece_id))
            parts.append(glyphs.death_marker(MapCord(c.x, c.y)))

        return '\n'.join(parts)

    return OverlaySpec("food_trail", render, priority=68)


In [ ]:
# Reset & apply food profiles (now that FOOD_RESERVE_TURNS exists)
for p in queenGuard.pieces:
    p.diet, p.food_capacity = 1.0, 10.0  # reset so .apply detects defaults
    FoodProfile.apply(p)

# Find the queen
queen = [p for p in queenGuard.pieces if p.piece_type == PieceType.QUEEN][0]
queen.facing = 1  # West

# Queen marches west
queen.instructions = InstructionList(
    [Instruction.FORWARD.value] * 10, cursor=0, patrol=False)

# Pawns: harvest + give patrol (stationary food stations)
for p in queenGuard.pieces:
    if p.piece_type == PieceType.PAWN:
        facing_to_queen = _facing_toward(myGrid, p.location, queen.location)
        p.facing = facing_to_queen
        p.instructions = InstructionList(
            [Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.HARVEST.value, Instruction.GIVE.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value,Instruction.FORWARD.value, Instruction.FORWARD.value],
            cursor=0, patrol=True)

        #p.instructions = []
        for i in range(0, 30):
            p.instructions.rules.append(Instruction.FORWARD.value)

# Run simulation
sim = FoodSimulator(
    grid=myGrid,
    elevations=myTerr.elevations,
    food_tiers=food_tiers,
    pieces=list(queenGuard.pieces),  # copy list so sim owns it
    countries=myTerr.fields.get('country'),
)
sim.run(num_turns=18)
sim.summary()

# Render
clearTerr()
myGrid.adjustRadius(45)
region = HexRegion(hexes=set(myTerr.ring(myHome.location, 7)), hexGrid=myGrid)

TerrainDisplay(
    CreamOverlay(),
    #FoodOverlay(color="#8D6E63", n_tiers=6),
    SquadFoodTrailOverlay(sim, queenGuard, sim_grid=myGrid),
    #SquadPlanOverlay([queenGuard], num_turns=8),
    terrain=myTerr,
    basins=basins,
    region=queenRegion
)


In [ ]:
show(myQueen)

GroupedPieceList(queenGuard.pieces,grid=myGrid)

In [ ]:
show(queenGuard)

In [ ]:
GroupedPieceList(queenGuard.pieces)

In [ ]:
#| export
show(sim)

#| export
The queen doesn't seem to lose health?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good catch! The bug is in `_eat_phase`. Look at this gate:

```python
if piece.id not in executed or not executed[piece.id]:
    continue
```

The queen has 10 `FORWARD` instructions with `patrol=False`. With speed 5, she burns through them in ~2 turns. After that her cursor is past the end of the list, so `_move_one` returns an empty list `[]` — which is **falsy**. The eat phase then skips her entirely: no food consumed, no starvation, no health loss.

The fix: pieces should always eat if they're alive, regardless of whether they executed instructions that turn. The gate should only skip pieces that are truly inactive (e.g. dead or not yet placed). Something like:

```python
@patch
def _eat_phase(self: FoodSimulator, executed):
    for piece in list(self.pieces):
        if piece.health <= 0:
            continue
        # Only skip pieces that never appeared in the sim at all
        if piece.location is None or piece.location < 0:
            continue

        shortfall = max(0.0, piece.diet - piece.food)
        piece.food = max(0.0, piece.food - piece.diet)
        self._log(piece, SimEventType.ATE, amount=piece.diet)

        if shortfall > 0:
            damage = shortfall * STARVE_MULT
            piece.health -= damage
            self._log(piece, SimEventType.STARVED, amount=damage)

        if piece.health <= 0:
            piece.health = 0
            self._log(piece, SimEventType.DIED)
```

This removes the `executed` check and instead just skips dead or unplaced pieces. Every alive piece on the board eats every turn — which makes sense, since they're burning calories whether or not they moved.

So we have an issue between the number of turns and number of moves. we want the pieces to eat every turn. maybe we do feed before we starting going through the rules list on each move?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a much cleaner mental model. Eat at the top of every turn — it's a daily cost of being alive, not a consequence of acting. Then the rest of the turn is about replenishing.

```python
@patch
def step(self: FoodSimulator):
    self._eat_phase()              # everyone eats first, every turn
    executed = self._move_phase()
    self._gather_phase(executed)
    self._share_phase(executed)
    self._snapshot()
    self.turn += 1
```

And `_eat_phase` drops the `executed` arg entirely:

```python
@patch
def _eat_phase(self: FoodSimulator):
    for piece in list(self.pieces):
        if piece.health <= 0:
            continue
        if piece.location is None or piece.location < 0:
            continue

        shortfall = max(0.0, piece.diet - piece.food)
        piece.food = max(0.0, piece.food - piece.diet)
        self._log(piece, SimEventType.ATE, amount=piece.diet)

        if shortfall > 0:
            damage = shortfall * STARVE_MULT
            piece.health -= damage
            self._log(piece, SimEventType.STARVED, amount=damage)

        if piece.health <= 0:
            piece.health = 0
            self._log(piece, SimEventType.DIED)
```

Now every alive piece on the board pays their `diet` cost at the start of every turn, then gets a chance to harvest/share to refill for next turn. The queen will start losing health properly once her food runs out.

I am also thinking about an __ft__ on FoodYield that would do matplotlib to show distributions

In [ ]:
#| export
@patch
def _eat_phase(self:FoodSimulator, executed):
    for piece in list(self.pieces):
        if piece.health <= 0:
            continue
        # Inactive this turn → excluded entirely (no eating)
        if piece.id not in executed or not executed[piece.id]:
            continue
            
        shortfall = max(0.0, piece.diet - piece.food)
        piece.food = max(0.0, piece.food - piece.diet)
        self._log(piece, SimEventType.ATE, f"diet={piece.diet:.1f}", piece.diet)

        if shortfall > 0:
            damage = shortfall * STARVE_MULT
            piece.health -= damage
            self._log(piece, SimEventType.STARVED, f"dmg={damage:.1f}", damage)

        if piece.health <= 0:
            piece.health = 0
            self._log(piece, SimEventType.DIED)


In [ ]:
#| export
@patch
def step(self:FoodSimulator):
    executed = self._move_phase()
    self._gather_phase(executed)
    self._share_phase(executed)
    self._eat_phase(executed)      # ← now gated
    self._snapshot()
    self.turn += 1

I previously gave you the wrong __ft__ can you we rewrite this one? maybe with a plot showing how many turns they have left if they don't eat.

```python

@patch
def __ft__(self: Piece):
    """Narrow vertical layout for w-64 sidebar panel."""
    ptype = self.piece_type.name.title()
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # ── Flag-patterned piece icon (80px) ──
    icon_size = 80
    icon_el = P(self.piece_type.icon, cls="text-5xl")  # fallback: unicode glyph from PieceType.icon

    if self.flag:
        builder = SVGBuilder()
        builder.width = icon_size
        builder.height = icon_size
        center = MapCord(icon_size / 2, icon_size / 2)
        scale = (icon_size * 0.55) / 22.5
        svg = self.flag.draw_piece(
            self.piece_type, center, builder,
            scale=scale, size='large',
            piece_id=f"piece_icon_{self.id[:8]}",
            layer="piece",
        )
        if svg:
            icon_el = NotStr(builder.xml())

    # ── Name / type / birth ──
    display_name = self.name if self.name else ptype
    subtitle = ptype
    if self.name:
        subtitle = f"{ptype} • b.{self.birth_year}" if self.birth_year else ptype

    # ── Stats row ──
    stats = [("⚔", self.attack_strength, "Atk"),
             ("🏃", self.move_strength,   "Spd"),
             ("🌾", self.harvest_strength, "Harv"),
             ("👁", self.sight,            "Sight")]
    stat_row = DivFullySpaced(
        *[Div(
            P(f"{ic} {v}", cls="text-sm font-bold text-center"),
            P(label, cls="text-[10px] text-center opacity-50"),
          ) for ic, v, label in stats],
    )

    # ── Rules — compact badge strip ──
    rules_el = self.instructions.compact(self.id)

    return Div(
        DivCentered(icon_el, cls="pt-2"),
        DivCentered(
            Strong(display_name),
            P(subtitle, cls=TextPresets.muted_sm),
        ),
        P(f"Facing {facing_arrow}", cls="text-center text-xs opacity-60"),

        StatBar("Health", self.health, self.max_health),
        StatBar("Food",   self.food,   self.food_capacity),

        stat_row,

        Divider(),
        rules_el,

        Divider(),
        DivVStacked(
            Button("Edit Rules",
                   hx_get=f"/piece_rules/{self.id}",
                   hx_target="#right-panel", hx_swap="innerHTML",
                   cls=ButtonT.primary + " " + ButtonT.sm + " w-full"),
            Button("Show Plan",
                   hx_get=f"/piece_plan/{self.id}",
                   hx_target="#map", hx_swap="innerHTML",
                   cls=ButtonT.sm + " w-full"),
            cls="space-y-1",
        ),

        id=f"piece-card-{self.id[:8]}",
        cls="space-y-2 p-3",
    )


```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's the rewritten `__ft__` with the `compact` bug fix and a food-turns-remaining gauge:

```python
@patch
def __ft__(self: Piece):
    """Narrow vertical layout for w-64 sidebar panel."""
    ptype = self.piece_type.name.title()
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # ── Flag-patterned piece icon (80px) ──
    icon_size = 80
    icon_el = P(self.piece_type.icon, cls="text-5xl")

    if self.flag:
        builder = SVGBuilder()
        builder.width = icon_size
        builder.height = icon_size
        center = MapCord(icon_size / 2, icon_size / 2)
        scale = (icon_size * 0.55) / 22.5
        svg = self.flag.draw_piece(
            self.piece_type, center, builder,
            scale=scale, size='large',
            piece_id=f"piece_icon_{self.id[:8]}",
            layer="piece",
        )
        if svg:
            icon_el = NotStr(builder.xml())

    # ── Name / type / birth ──
    display_name = self.name if self.name else ptype
    subtitle = ptype
    if self.name:
        subtitle = f"{ptype} • b.{self.birth_year}" if self.birth_year else ptype

    # ── Stats row ──
    stats = [("⚔", self.attack_strength, "Atk"),
             ("🏃", self.move_strength,   "Spd"),
             ("🌾", self.harvest_strength, "Harv"),
             ("👁", self.sight,            "Sight")]
    stat_row = DivFullySpaced(
        *[Div(
            P(f"{ic} {v}", cls="text-sm font-bold text-center"),
            P(label, cls="text-[10px] text-center opacity-50"),
          ) for ic, v, label in stats],
    )

    # ── Food turns remaining gauge ──
    turns_left = self.food / self.diet if self.diet > 0 else float('inf')
    max_turns = self.food_capacity / self.diet if self.diet > 0 else 1
    reserve = FOOD_RESERVE_TURNS

    # Small inline SVG gauge
    w, h = 220, 32
    bar_y, bar_h = 4, 14
    frac = min(turns_left / max_turns, 1.0) if max_turns > 0 else 1.0
    reserve_frac = min(reserve / max_turns, 1.0) if max_turns > 0 else 0

    if turns_left >= reserve:
        bar_color = "#27ae60"
    elif turns_left >= reserve * 0.5:
        bar_color = "#f39c12"
    else:
        bar_color = "#e74c3c"

    gauge_svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" '
        f'viewBox="0 0 {w} {h}">'
        # Background track
        f'<rect x="0" y="{bar_y}" width="{w}" height="{bar_h}" rx="3" fill="#eee"/>'
        # Fill bar
        f'<rect x="0" y="{bar_y}" width="{w * frac:.1f}" height="{bar_h}" '
        f'rx="3" fill="{bar_color}" opacity="0.8"/>'
        # Reserve threshold tick
        f'<line x1="{w * reserve_frac:.1f}" y1="{bar_y - 1}" '
        f'x2="{w * reserve_frac:.1f}" y2="{bar_y + bar_h + 1}" '
        f'stroke="#e67e22" stroke-width="2" stroke-dasharray="2,1"/>'
        # Labels
        f'<text x="4" y="{bar_y + bar_h + 14}" font-size="9" fill="#555" '
        f'font-family="sans-serif">'
        f'🌾 {turns_left:.1f} turns left</text>'
        f'<text x="{w - 4}" y="{bar_y + bar_h + 14}" font-size="9" fill="#e67e22" '
        f'font-family="sans-serif" text-anchor="end">'
        f'reserve={reserve}</text>'
        f'</svg>'
    )

    # ── Rules — compact badge strip (fixed: pass flag then id) ──
    rules_el = self.instructions.compact(self.flag, self.id)

    return Div(
        DivCentered(icon_el, cls="pt-2"),
        DivCentered(
            Strong(display_name),
            P(subtitle, cls=TextPresets.muted_sm),
        ),
        P(f"Facing {facing_arrow}", cls="text-center text-xs opacity-60"),

        StatBar("Health", self.health, self.max_health),
        StatBar("Food",   self.food,   self.food_capacity),
        DivCentered(NotStr(gauge_svg)),

        stat_row,

        Divider(),
        rules_el,

        Divider(),
        DivVStacked(
            Button("Edit Rules",
                   hx_get=f"/piece_rules/{self.id}",
                   hx_target="#right-panel", hx_swap="innerHTML",
                   cls=ButtonT.primary + " " + ButtonT.sm + " w-full"),
            Button("Show Plan",
                   hx_get=f"/piece_plan/{self.id}",
                   hx_target="#map", hx_swap="innerHTML",
                   cls=ButtonT.sm + " w-full"),
            cls="space-y-1",
        ),

        id=f"piece-card-{self.id[:8]}",
        cls="space-y-2 p-3",
    )
```

Two changes from the original:

1. **Bug fix**: `self.instructions.compact(self.flag, self.id)` — was passing `self.id` as the `flag` arg, which caused the `'str' object has no attribute 'darkPrimary'` error.

2. **Food turns gauge**: A lightweight inline SVG bar sits right below the Food StatBar. It shows:
   - Green/yellow/red fill proportional to `food / diet` relative to max capacity turns
   - An orange dashed tick at the `FOOD_RESERVE_TURNS` threshold
   - Text labels: turns remaining (left) and reserve target (right)

No matplotlib overhead per card — just a tiny SVG that renders instantly. If you'd prefer a full matplotlib chart (e.g. projecting food depletion over N future turns as a line), let me know and I can swap it in.

Here's a style guide you can drop into a CRAFT.ipynb note so it's always in context:

```markdown
## SVG Style Guide — Use Existing Primitives

When generating SVG for this project, **never write raw inline SVG strings**. Use the existing class hierarchy instead.

### Core Rule
All SVG output flows through `SVGBuilder`. All styling flows through `StyleCSS`. All coordinates use `MapCord`. All paths use `MapPath`.

### Colors & Styles — use `StyleCSS`, never inline `fill=`/`stroke=`

```python
# ✗ BAD — inline colors
f'<rect ... fill="#27ae60" stroke="#333" stroke-width="2"/>'

# ✓ GOOD — StyleCSS class
track = StyleCSS("gauge_track", fill="#eee", stroke="none", stroke_width=0)
bar   = StyleCSS("gauge_fill",  fill="#27ae60", stroke="none", opacity=0.8)
builder.add_style(track)
builder.add_style(bar)
f'<rect ... class="{track.name}"/>'
```

For color interpolation between two colors, use `StyleCSS.lerp_color(c1, c2, t)`.

### Points & Coordinates — use `MapCord`, never bare `x, y` floats

```python
# ✗ BAD
cx, cy = 100, 200

# ✓ GOOD
center = MapCord(100, 200)
```

### Lines & Shapes — use `MapPath`, never raw `<line>` or `<path d=>`

```python
# ✗ BAD
f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="red"/>'

# ✓ GOOD
MapPath([start, end], style).drawPolygon()       # open line
MapPath(points, style).drawClosed()              # closed polygon
MapPath([start, end], style).with_arrowhead()    # directed arrow
MapPath(points, style).svg()                     # smooth spline
```

### Composing SVG — use `SVGBuilder` layers, never standalone `<svg>` tags

```python
# ✗ BAD — hand-rolled SVG document
f'<svg xmlns="..." width="{w}" height="{h}">...<rect .../></svg>'

# ✓ GOOD — builder with layers
canvas = SVGBuilder()
canvas.width, canvas.height = w, h
canvas.add_style(my_style)
canvas.adjust("background", f'<rect ... class="{my_style.name}"/>')
canvas.adjust("content", body_svg)
canvas.show()  # or canvas.xml()
```

### Themed Glyphs — use `DiagramGlyphs`, never hand-draw icons

`DiagramGlyphs` provides flag-themed SVG primitives. All take `MapCord` centers and optional `StyleCSS` overrides:

| Method | Use for |
|---|---|
| `glyphs.circle(center, r)` | Any circle |
| `glyphs.ring(center, r)` | Hollow ring marker |
| `glyphs.arc(center, r, start_deg, end_deg)` | Partial arc |
| `glyphs.shield(center)` | Defend/protect |
| `glyphs.house(center)` | Settlement |
| `glyphs.gift(center)` | Transfer/trade |
| `glyphs.sheaf(center)` | Harvest/food |
| `glyphs.pause(center)` | Wait/hold |
| `glyphs.rot_arrow(center, clockwise)` | Rotation |
| `glyphs.death_marker(center)` | Death/destruction |
| `glyphs.blocked_x(center)` | Blocked/impassable |
| `glyphs.food_pip(center, frac)` | Food level 0–1 (auto red→green) |
| `glyphs.transfer_arrow(start, end)` | Directed resource arrow |
| `glyphs.facing_bar(center, facing)` | Directional facing indicator |
| `glyphs.step_label(center, n)` | Numbered step badge |
| `glyphs.start_marker(center)` | Start position ring |

Always call `glyphs.register_styles(builder)` once to inject all CSS classes.

### Gauge/Bar Example — the right way

```python
canvas = SVGBuilder()
canvas.width, canvas.height = w, h

track_style = StyleCSS("gauge_track", fill="#eee", stroke="none")
fill_color  = StyleCSS.lerp_color('#e74c3c', '#27ae60', frac)
fill_style  = StyleCSS("gauge_fill", fill=fill_color, stroke="none", opacity=0.8)
tick_style  = StyleCSS("gauge_tick", stroke="#e67e22", fill="none",
                        stroke_width=2, stroke_dasharray="2,1")

for s in (track_style, fill_style, tick_style):
    canvas.add_style(s)

body  = f'<rect x="0" y="{bar_y}" width="{w}" height="{bar_h}" rx="3" class="{track_style.name}"/>'
body += f'<rect x="0" y="{bar_y}" width="{w*frac:.1f}" height="{bar_h}" rx="3" class="{fill_style.name}"/>'
body += MapPath([
    MapCord(w * reserve_frac, bar_y - 1),
    MapCord(w * reserve_frac, bar_y + bar_h + 1)
], tick_style).drawPolygon()

canvas.adjust("gauge", body)
```

### Quick Checklist
- [ ] No `fill=` or `stroke=` on elements — use `class=` pointing to a `StyleCSS`
- [ ] No `(x, y)` tuples — use `MapCord(x, y)`
- [ ] No `<line>` or `<polygon>` by hand — use `MapPath`
- [ ] No standalone `<svg>` wrapper — use `SVGBuilder`
- [ ] No hand-drawn icons — check `DiagramGlyphs` first
- [ ] Fonts added via `canvas.add_font('Cinzel')`, not inline `@import`
- [ ] Flag-themed colors via `flag.darkPrimary`, `flag.primary`, `flag.comp`, `flag.lightPrimary`
```

Drop this as a note in your root or project-level CRAFT.ipynb and it'll be in context for every dialog. The checklist at the bottom is deliberately terse so it works as a quick mental scan even in a large context window.

Can we extract the guage part out and make something using the style guide?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



Absolutely! Let's extract a reusable `FoodGauge` function that follows the style guide — `SVGBuilder`, `StyleCSS`, `MapCord`, `MapPath`:

```python
def FoodGauge(piece, w=220, h=32, bar_h=14, bar_y=4):
    """Inline SVG food-turns-remaining gauge for a Piece card."""
    turns_left = piece.food / piece.diet if piece.diet > 0 else float('inf')
    max_turns = piece.food_capacity / piece.diet if piece.diet > 0 else 1
    reserve = FOOD_RESERVE_TURNS

    frac = min(turns_left / max_turns, 1.0) if max_turns > 0 else 1.0
    reserve_frac = min(reserve / max_turns, 1.0) if max_turns > 0 else 0

    # ── Styles ──
    canvas = SVGBuilder()
    canvas.width, canvas.height = w, h

    track = StyleCSS("fg_track", fill="#eee", stroke="none", stroke_width=0)

    if turns_left >= reserve:
        bar_color = StyleCSS.lerp_color('#e74c3c', '#27ae60', min(frac, 1.0))
    elif turns_left >= reserve * 0.5:
        bar_color = '#f39c12'
    else:
        bar_color = '#e74c3c'

    fill = StyleCSS("fg_fill", fill=bar_color, stroke="none", opacity=0.8)
    tick = StyleCSS("fg_tick", stroke="#e67e22", fill="none",
                    stroke_width=2, stroke_dasharray="2,1")
    lbl  = StyleCSS("fg_lbl", fill="#555", stroke="none", stroke_width=0)
    rlbl = StyleCSS("fg_rlbl", fill="#e67e22", stroke="none", stroke_width=0)

    for s in (track, fill, tick, lbl, rlbl):
        canvas.add_style(s)

    # ── Track + fill bar ──
    body  = f'<rect x="0" y="{bar_y}" width="{w}" height="{bar_h}" rx="3" class="{track.name}"/>'
    body += f'<rect x="0" y="{bar_y}" width="{w * frac:.1f}" height="{bar_h}" rx="3" class="{fill.name}"/>'

    # ── Reserve tick ──
    body += MapPath([
        MapCord(w * reserve_frac, bar_y - 1),
        MapCord(w * reserve_frac, bar_y + bar_h + 1),
    ], tick).drawPolygon()

    # ── Labels ──
    txt_y = bar_y + bar_h + 14
    tl_str = f"{turns_left:.1f}" if turns_left < 1000 else "∞"
    body += (f'<text x="4" y="{txt_y}" font-size="9" '
             f'font-family="sans-serif" class="{lbl.name}">'
             f'🌾 {tl_str} turns left</text>')
    body += (f'<text x="{w - 4}" y="{txt_y}" font-size="9" '
             f'font-family="sans-serif" text-anchor="end" class="{rlbl.name}">'
             f'reserve={reserve}</text>')

    canvas.adjust("gauge", body)
    return NotStr(canvas.xml())
```

Then in `Piece.__ft__` you just swap the raw SVG block for:

```python
    # ── Food turns gauge ──
    gauge_el = FoodGauge(self)

    # ... later in the return Div:
    StatBar("Food", self.food, self.food_capacity),
    DivCentered(gauge_el),
```

Key things this follows from the style guide:
- **`SVGBuilder`** for the document, not a raw `<svg>` tag
- **`StyleCSS`** for all colors — no inline `fill=`/`stroke=` on elements
- **`StyleCSS.lerp_color`** for the green↔red gradient
- **`MapCord` + `MapPath`** for the reserve tick line
- Labels still use `<text>` (no `MapPath` equivalent for text), but styling is via `class=`

I switched guage to be an @patch. can you write the full piece.__ft__

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



```python
@patch
def __ft__(self: Piece):
    """Narrow vertical layout for w-64 sidebar panel."""
    ptype = self.piece_type.name.title()
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # ── Flag-patterned piece icon (80px) ──
    icon_size = 80
    icon_el = P(self.piece_type.icon, cls="text-5xl")

    if self.flag:
        builder = SVGBuilder()
        builder.width = icon_size
        builder.height = icon_size
        center = MapCord(icon_size / 2, icon_size / 2)
        scale = (icon_size * 0.55) / 22.5
        svg = self.flag.draw_piece(
            self.piece_type, center, builder,
            scale=scale, size='large',
            piece_id=f"piece_icon_{self.id[:8]}",
            layer="piece",
        )
        if svg:
            icon_el = NotStr(builder.xml())

    # ── Name / type / birth ──
    display_name = self.name if self.name else ptype
    subtitle = ptype
    if self.name:
        subtitle = f"{ptype} • b.{self.birth_year}" if self.birth_year else ptype

    # ── Stats row ──
    stats = [("⚔", self.attack_strength, "Atk"),
             ("🏃", self.move_strength,   "Spd"),
             ("🌾", self.harvest_strength, "Harv"),
             ("👁", self.sight,            "Sight")]
    stat_row = DivFullySpaced(
        *[Div(
            P(f"{ic} {v}", cls="text-sm font-bold text-center"),
            P(label, cls="text-[10px] text-center opacity-50"),
          ) for ic, v, label in stats],
    )

    # ── Rules — compact badge strip ──
    rules_el = self.instructions.compact(self.flag, self.id)

    return Div(
        DivCentered(icon_el, cls="pt-2"),
        DivCentered(
            Strong(display_name),
            P(subtitle, cls=TextPresets.muted_sm),
        ),
        P(f"Facing {facing_arrow}", cls="text-center text-xs opacity-60"),

        StatBar("Health", self.health, self.max_health),
        StatBar("Food",   self.food,   self.food_capacity),
        DivCentered(self.food_gauge()),

        stat_row,

        Divider(),
        rules_el,

        Divider(),
        DivVStacked(
            Button("Edit Rules",
                   hx_get=f"/piece_rules/{self.id}",
                   hx_target="#right-panel", hx_swap="innerHTML",
                   cls=ButtonT.primary + " " + ButtonT.sm + " w-full"),
            Button("Show Plan",
                   hx_get=f"/piece_plan/{self.id}",
                   hx_target="#map", hx_swap="innerHTML",
                   cls=ButtonT.sm + " w-full"),
            cls="space-y-1",
        ),

        id=f"piece-card-{self.id[:8]}",
        cls="space-y-2 p-3",
    )
```

The only real changes from the original:
1. **`self.instructions.compact(self.flag, self.id)`** — passes `flag` first, fixing the `darkPrimary` crash
2. **`DivCentered(self.food_gauge())`** — calls your new `@patch` method, slotted between the Food StatBar and the stats row

sim = FoodSimulator(grid, elevs, food_tiers, all_pieces)
sim.run(30)
show(sim)        # instant line charts
sim.summary()    # text stats


## Recording

I think we want the food simulator to generate a pandas dataframe of the actions.

I was thinking we would need a source and dest column for both pieces and hexes on these action as well as keeping track of the round

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good instinct. Right now `SimEvent` conflates source and destination into the `detail` string, which is fine for logging but messy for analysis.

For a clean DataFrame you'd probably want something like:

| column | notes |
|---|---|
| `turn` | already have |
| `piece_id` / `piece_name` | the *acting* piece |
| `event` | event type enum |
| `src_hex` / `dst_hex` | for MOVED; for stationary actions both are the same hex |
| `src_piece_id` / `dst_piece_id` | for GAVE/RECEIVED; otherwise same piece |
| `amount` | food delta |

Then a property like `sim.to_df()` that builds it from `self.events`.

A couple of questions before writing anything:

1. Do you want to **replace** `SimEvent` with the expanded fields, or keep it as-is and just build the DataFrame from it in `to_df()`?
2. Should `src_piece_id` / `dst_piece_id` store the piece ID, the name, or both?

Wouldn't the dataframe be a great deal stronger than the simevent? I don't think we will need both

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Agreed — `SimEvent` is basically just a typed row. If you're going to analyse everything as a DataFrame anyway, the dataclass is just an extra layer with no real benefit.

The typical pattern would be: accumulate a `list[dict]` during the simulation, then expose it lazily:

```python
self._rows: list[dict] = []
# ...and at the end:
@property
def df(self) -> pd.DataFrame:
    return pd.DataFrame(self._rows)
```

One thing worth deciding: do you want to keep the `events_for_turn` / `events_for_piece` convenience methods, or just let the caller use `.query()` / boolean indexing on the DataFrame directly?

for the food audit we appended rows

```python
@patch
def food_audit(self: Squad, food_tiers: np.ndarray) -> pd.DataFrame:
    """Static food analysis — one row per alive piece."""
    rows = []
    for p in self.alive:
        tier = int(food_tiers[p.location]) if (p.location is not None and p.location >= 0) else 0
        gross = p.harvest_strength * tier
        net = gross - p.diet
        rows.append({
            'piece': p,
            'name': p.name,
            'type': p.piece_type.name,
            'diet': p.diet,
            'harvest_str': p.harvest_strength,
            'tier': tier,
            'gross': gross,
            'net': net,
            'food': p.food,
            'reserve_turns': p.food / p.diet if p.diet > 0 else float('inf'),
            'location': p.location,
        })
    df = pd.DataFrame(rows)
    df.attrs['demand'] = df.diet.sum()
    df.attrs['supply'] = df.loc[df.net > 0, 'net'].sum()
    df.attrs['gap'] = df.attrs['demand'] - df.attrs['supply'] - df.loc[df.net <= 0, 'gross'].sum()
    return df
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly the same pattern works perfectly for the simulator. Instead of `self._log(...)` building `SimEvent` objects, it just appends a dict to `self._rows`.

The columns would be the same shape as `food_audit` plus the dynamic fields:

```python
self._rows.append({
    'turn': self.turn,
    'event': etype.value,
    'piece_id': piece.id,
    'piece_name': piece.name,
    'src_hex': ...,
    'dst_hex': ...,
    'src_piece_id': ...,
    'dst_piece_id': ...,
    'amount': amount,
})
```

Then `sim.df` is just `pd.DataFrame(self._rows)`.

The main design question: for events like GAVE/RECEIVED, both pieces are involved. Do you want **one row per event** (GAVE row has both `src_piece_id` and `dst_piece_id`), or **two rows** (a GAVE row and a mirrored RECEIVED row for the recipient)?

#| export
I think just one event. If they each give other that would be 2 events.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Clean and simple. So the full column set would be something like:

| column | populated by |
|---|---|
| `turn` | all events |
| `event` | all events |
| `piece_id` / `piece_name` | the *acting* piece |
| `src_hex` / `dst_hex` | MOVED; same value for stationary |
| `dst_piece_id` | GAVE only (the recipient) |
| `amount` | HARVESTED, GAVE, ATE, STARVED |

And `None` / `NaN` for unused columns on a given row — which pandas handles naturally.

Shall I write the updated `FoodSimulator` with `_rows` replacing `SimEvent`?

Please rewrite the simulaor

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
class SimEventType(Enum):
    MOVED     = "moved"
    BLOCKED   = "blocked"
    ROTATED   = "rotated"
    HARVESTED = "harvested"
    GAVE      = "gave"
    ATE       = "ate"
    STARVED   = "starved"
    DIED      = "died"

_FREE = {Instruction.ROT_L, Instruction.ROT_R}
_COSTS_ONE = {Instruction.PAUSE, Instruction.DEFEND,
              Instruction.HARVEST, Instruction.GIVE, Instruction.SETTLE}

@dataclass
class FoodSimulator:
    """Turn-based simulator: move → gather → share → eat."""
    grid: HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    pieces: list                           # list[Piece]
    countries: np.ndarray = None
    elevation_mult: float = 0.005

    _rows: list = field(default_factory=list)
    turn: int = 0
    snapshots: list = field(default_factory=list)

    # ── helpers ──────────────────────────────────────────────

    @property
    def alive(self):
        return [p for p in self.pieces if p.health > 0]

    @property
    def occupied(self):
        return {p.location for p in self.alive
                if p.location is not None and p.location >= 0}

    @property
    def df(self) -> pd.DataFrame:
        return pd.DataFrame(self._rows)

    def _log(self, piece, etype: SimEventType, *,
             src_hex=None, dst_hex=None,
             dst_piece_id=None, amount=0.0):
        loc = piece.location if piece.location is not None else -1
        self._rows.append({
            'turn':         self.turn,
            'event':        etype.value,
            'piece_id':     piece.id,
            'piece_name':   piece.name,
            'src_hex':      src_hex if src_hex is not None else loc,
            'dst_hex':      dst_hex if dst_hex is not None else loc,
            'dst_piece_id': dst_piece_id,
            'amount':       amount,
        })

    def _sort_move(self, pieces):
        """Rank → oldest → healthiest → random."""
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99),
            p.birth_year,
            -p.health,
            random.random()))

    def _sort_share(self, pieces):
        """Rank → oldest → least food → random."""
        return sorted(pieces, key=lambda p: (
            RANK_ORDER.get(p.piece_type, 99),
            p.birth_year,
            p.food,
            random.random()))

    # ── move phase ───────────────────────────────────────────

    def _move_one(self, piece, occupied):
        il = piece.instructions
        if not il.rules:
            return []

        executed = []
        budget = float(piece.move_strength)

        while budget > 0:
            if il.cursor >= len(il.rules):
                if il.patrol:
                    il.cursor = 0
                else:
                    break

            instr = Instruction(il.rules[il.cursor])

            # ── free rotations ──
            if instr in _FREE:
                piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                                else (piece.facing + 1)) % 6
                executed.append(instr)
                self._log(piece, SimEventType.ROTATED)
                il.cursor += 1
                continue

            # ── cost-1 non-movement ──
            if instr in _COSTS_ONE:
                budget -= 1.0
                executed.append(instr)
                il.cursor += 1
                continue

            # ── FORWARD ──
            auto_rots = 0
            saved_fac = piece.facing
            moved = False

            while auto_rots <= 5:
                d = HexPosition.directions()[piece.facing % 6]
                nbr = self.grid.hexposition_to_index(d, piece.location)

                passable = (0 <= nbr < len(self.elevations)
                            and nbr not in self.grid.invalidRegion
                            and self.elevations[nbr] >= 1)
                if self.countries is not None and passable:
                    passable = self.countries[nbr] >= 0

                if not passable:
                    piece.facing = (piece.facing - 1) % 6
                    auto_rots += 1
                    continue

                # Occupied → freeze cursor, end turn
                if nbr in occupied and nbr != piece.location:
                    self._log(piece, SimEventType.BLOCKED, dst_hex=nbr)
                    moved = True
                    budget = 0
                    break

                # Can afford?
                cost = _move_cost(self.elevations, piece.location, nbr)
                if cost > budget:
                    budget = 0
                    moved = True
                    break

                # Move
                src = piece.location
                occupied.discard(src)
                piece.location = nbr
                occupied.add(nbr)
                budget -= cost
                executed.append(instr)
                self._log(piece, SimEventType.MOVED,
                          src_hex=src, dst_hex=nbr, amount=cost)
                il.cursor += 1
                moved = True
                break

            if not moved:
                piece.facing = saved_fac
                self._log(piece, SimEventType.BLOCKED)
                budget = 0

        return executed

    def _move_phase(self):
        occupied = self.occupied
        order = self._sort_move(self.alive)
        executed = {}
        for piece in order:
            executed[piece.id] = self._move_one(piece, occupied)
        return executed

    # ── gather phase ─────────────────────────────────────────

    def _gather_phase(self, executed):
        for piece in self.alive:
            if Instruction.HARVEST in executed.get(piece.id, []):
                loc = piece.location
                if loc is None or loc < 0:
                    continue
                tier = int(self.food_tiers[loc])
                gained = piece.harvest_strength * tier
                old = piece.food
                piece.food = min(piece.food + gained, piece.food_capacity)
                self._log(piece, SimEventType.HARVESTED,
                          amount=piece.food - old)

    # ── share phase ──────────────────────────────────────────

    def _share_phase(self, executed):
        givers = [p for p in self.alive
                  if Instruction.GIVE in executed.get(p.id, [])]

        for giver in self._sort_share(givers):
            reserve = FOOD_RESERVE_TURNS * giver.diet
            surplus = max(0.0, giver.food - reserve)
            if surplus <= 0:
                continue

            visible = set(giver.hexes_in_sight(
                self.grid, self.elevations,
                elevation_mult=self.elevation_mult,
                facing_only=True))

            receivers = sorted(
                [p for p in self.alive
                 if p is not giver
                 and p.owner_id == giver.owner_id
                 and p.food < p.food_capacity
                 and p.location in visible],
                key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

            for recv in receivers:
                if surplus <= 0:
                    break
                amount = min(surplus, recv.food_capacity - recv.food)
                recv.food += amount
                giver.food -= amount
                surplus -= amount
                self._log(giver, SimEventType.GAVE,
                          dst_piece_id=recv.id, amount=amount)

    # ── eat phase ────────────────────────────────────────────

    def _eat_phase(self, executed):
        for piece in list(self.pieces):
            if piece.health <= 0:
                continue
            if piece.id not in executed or not executed[piece.id]:
                continue

            shortfall = max(0.0, piece.diet - piece.food)
            piece.food = max(0.0, piece.food - piece.diet)
            self._log(piece, SimEventType.ATE, amount=piece.diet)

            if shortfall > 0:
                damage = shortfall * STARVE_MULT
                piece.health -= damage
                self._log(piece, SimEventType.STARVED, amount=damage)

            if piece.health <= 0:
                piece.health = 0
                self._log(piece, SimEventType.DIED)

    # ── snapshot ─────────────────────────────────────────────

    def _snapshot(self):
        alive = self.alive
        self.snapshots.append({
            'turn': self.turn,
            'alive': len(alive),
            'total': len(self.pieces),
            'avg_health': np.mean([p.health for p in alive]) if alive else 0,
            'min_health': min((p.health for p in alive), default=0),
            'avg_food': np.mean([p.food for p in alive]) if alive else 0,
            'avg_turns_left': np.mean([p.food / max(p.diet, 0.01)
                                       for p in alive]) if alive else 0,
        })

    # ── turn loop ────────────────────────────────────────────

    def step(self):
        executed = self._move_phase()
        self._gather_phase(executed)
        self._share_phase(executed)
        self._eat_phase(executed)
        self._snapshot()
        self.turn += 1

    def run(self, num_turns=20):
        for _ in range(num_turns):
            if not self.alive:
                self._snapshot()
                break
            self.step()
        return self

    def summary(self):
        a = self.alive
        print(f"Turn {self.turn}: {len(a)}/{len(self.pieces)} alive")
        if a:
            print(f"  Health: avg={np.mean([p.health for p in a]):.0f} "
                  f"min={min(p.health for p in a):.0f}")
            print(f"  Food:   avg={np.mean([p.food for p in a]):.1f} "
                  f"turns={np.mean([p.food/max(p.diet,.01) for p in a]):.1f}")
```

Key changes:

- **Dropped `SimEvent` dataclass** — `_log` now appends a flat dict directly
- **Columns**: `turn`, `event`, `piece_id`, `piece_name`, `src_hex`, `dst_hex`, `dst_piece_id`, `amount` — `dst_piece_id` is only non-null for GAVE events
- **`sim.df`** property returns the full DataFrame lazily
- **Merged all patches** (`_eat_phase`, `step`) inline
- **Dropped `RECEIVED`** — each GIVE is one row, the acting piece is the giver, the recipient is `dst_piece_id`
- **Dropped `events_for_turn` / `events_for_piece`** — just use `sim.df.query("turn == 5")` etc.

I need to fix some box plots: I am getting /tmp/ipykernel_1840/3142545611.py:74: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=type_labels, patch_artist=True,

In [ ]:
#| export
_FREE = {Instruction.ROT_L, Instruction.ROT_R}
_COSTS_ONE = {Instruction.HARVEST, Instruction.GIVE, Instruction.PAUSE}


In [ ]:
# ── Board ──────────────────────────────────────────────────────────────────
board = PieceBoard.hilly(rings=8, radius=18, seed=7)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

# Simple food tiers: low ground = fertile, high = barren
food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if   e <= 0:    food_tiers[i] = 0   # ocean
    elif e < 60:    food_tiers[i] = 7   # coastal plains  ← pawns go here
    elif e < 120:   food_tiers[i] = 5
    elif e < 180:   food_tiers[i] = 3
    elif e < 260:   food_tiers[i] = 1
    else:           food_tiers[i] = 0   # peaks

def find_land(origin, offsets):
    for hp in offsets:
        idx = grid.hexposition_to_index(hp, origin)
        if idx >= 0 and idx not in grid.invalidRegion and elevs[idx] > 0:
            return idx
    return None

# ── Queen — high ground, far from food, starts hungry ──────────────────────
queen_idx = find_land(mid, [HexPosition(q, r, -q-r)
                             for q in range(-2, 3) for r in range(-2, 3)
                             if elevs[grid.hexposition_to_index(
                                 HexPosition(q,r,-q-r), mid)] > 200])

if queen_idx is None:   # fallback
    queen_idx = find_land(mid, [HexPosition(0,0,0)])

queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0,
                        country_id=1,
                        instructions=InstructionList(
                            [Instruction.PAUSE.value], patrol=True))
FoodProfile.apply(queen)
queen.food = queen.diet * 4.5   # only 1.5 turns of food — very hungry

print(f"Queen @ hex {queen_idx}  elev={elevs[queen_idx]:.0f}  "
      f"food={queen.food:.1f}/{queen.food_capacity}  diet={queen.diet}")

# ── Pawns — fertile low tiles, harvest+give loop ────────────────────────────
pawn_offsets = [
    HexPosition(-5, 3, 2), HexPosition(-4, 4, 0), HexPosition(-6, 4, 2),
    HexPosition(-5, 5,-0), HexPosition(-3, 3, 0),
]
pawns = []
for hp in pawn_offsets:
    idx = find_land(mid, [hp, hp + HexPosition(1,0,-1), hp + HexPosition(0,1,-1)])
    if idx is None or food_tiers[idx] < 4:
        continue
    p = board.add_piece(idx, PieceType.PAWN,
                        facing=random.randint(0,5), country_id=1,
                        instructions=InstructionList(
                            [Instruction.HARVEST.value,
                             Instruction.GIVE.value], patrol=True))
    FoodProfile.apply(p)
    p.food = p.diet * FOOD_RESERVE_TURNS
    pawns.append(p)

print(f"{len(pawns)} pawns on food tiers: "
      f"{[food_tiers[p.location] for p in pawns]}")

# ── Bishops — run routes between pawn cluster and queen ─────────────────────
# Place two bishops midway between the pawn cluster centroid and the queen
pawn_positions = [grid.index_to_hexposition(p.location) for p in pawns]
pc_q = int(np.mean([p.q for p in pawn_positions]))
pc_r = int(np.mean([p.r for p in pawn_positions]))

queen_pos = grid.index_to_hexposition(queen_idx)

# Two waypoints: 1/3 and 2/3 along the pawn→queen vector
bishops = []
for frac, label in [(0.35, "near"), (0.65, "far")]:
    bq = int(pc_q + frac * (queen_pos.q - pc_q))
    br = int(pc_r + frac * (queen_pos.r - pc_r))
    bs = -bq - br
    b_idx = find_land(mid, [HexPosition(bq, br, bs),
                             HexPosition(bq+1, br, bs-1),
                             HexPosition(bq, br+1, bs-1),
                             HexPosition(bq-1, br, bs+1)])
    if b_idx is None:
        continue

    # Bishop path: pawn cluster → queen → back (supply run)
    pawn_anchor = pawns[0].location
    path_out = board.pieces[0].pathfind.__func__(  # use a dummy pathfind
        type('_', (), {'location': b_idx, 'pathfind': lambda s,t,g,e,c=None:
            None})(), queen_idx, grid, elevs) \
        if False else None

    # Simpler: give bishop a HARVEST+GIVE patrol so it passes food along
    b = board.add_piece(b_idx, PieceType.BISHOP,
                        facing=_facing_toward(grid, b_idx, queen_idx),
                        country_id=1,
                        instructions=InstructionList(
                            [Instruction.GIVE.value,
                             Instruction.FORWARD.value,
                             Instruction.GIVE.value,
                             Instruction.FORWARD.value], patrol=True))
    FoodProfile.apply(b)
    b.food = b.diet * FOOD_RESERVE_TURNS
    bishops.append(b)
    print(f"  Bishop ({label}) @ hex {b_idx}  elev={elevs[b_idx]:.0f}  "
          f"capacity={b.food_capacity}")

# ── Pathfind bishops toward queen properly ──────────────────────────────────
for b in bishops:
    path = b.pathfind(queen_idx, grid, elevs)
    if path and len(path) > 1:
        rules = (path.to_rules(grid, b.facing)
                 + [Instruction.GIVE.value]
                 + list(reversed(path.to_rules(grid, b.facing)))
                 + [Instruction.GIVE.value])
        b.instructions = InstructionList(rules, cursor=0, patrol=True)
        print(f"  Bishop route: {len(path)} hexes, "
              f"{len(b.instructions.rules)} instructions")

# ── Squads for display ───────────────────────────────────────────────────────
royal_guard = Squad(id=1, name="Royal Guard",
                    pieces=[queen] + bishops)
supply_corps = Squad(id=2, name="Supply Corps",
                     pieces=pawns)

all_pieces = [queen] + bishops + pawns

# ── Run simulator ────────────────────────────────────────────────────────────
# ── Before state ─────────────────────────────────────────────────────────────
print("=== BEFORE ===")
sim = FoodSimulator(
    grid=grid,
    elevations=elevs,
    food_tiers=food_tiers,
    pieces=all_pieces,
)
sim.summary()

#show(royal_guard)
#show(supply_corps)
#show(NotStr(board.render_with_facing(num_turns=12)))

sim.run(num_turns=25)

print("\nAfter:")
#sim.summary()
print(f"\nQueen food: {queen.food:.1f}  health: {queen.health:.0f}")
print(f"Events for queen: {len(sim.events_for_piece(queen.id))}")

if False:
    
    show(royal_guard)
    show(supply_corps)
    show(NotStr(board.render_with_sight(
        [queen] + bishops + pawns,
        num_turns=12,
        facing_only=True
    )))



# Sample queen's event log
for ev in sim.events_for_piece(queen.id)[:12]:
    print(f"  T{ev.turn:02d} {ev.event.value:<10} {ev.detail}  "
          f"{'±'+str(round(ev.amount,1)) if ev.amount else ''}")


We lose a pawn?

can you show the inital board with the pieces plans so I can see how it is going?

So the queen is on the opposite side of the river outside of where the bishops can see. So she can't be fed

In [ ]:
# ── Board ──────────────────────────────────────────────────────────────────
board = PieceBoard.hilly(rings=8, radius=18, seed=7)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if   e <= 0:    food_tiers[i] = 0
    elif e < 60:    food_tiers[i] = 7
    elif e < 120:   food_tiers[i] = 5
    elif e < 180:   food_tiers[i] = 3
    elif e < 260:   food_tiers[i] = 1
    else:           food_tiers[i] = 0

# ── Step 1: Find a fertile cluster for pawns ───────────────────────────────
# Pick the best food hex near the center as our anchor
best_food = sorted(
    [i for i in range(len(elevs))
     if elevs[i] > 0 and i not in grid.invalidRegion and food_tiers[i] >= 5],
    key=lambda i: abs(grid.index_to_hexposition(i, mid))
)[:20]  # 20 closest fertile hexes

pawn_anchor = best_food[0]
print(f"Pawn anchor: hex {pawn_anchor}  tier={food_tiers[pawn_anchor]}  elev={elevs[pawn_anchor]:.0f}")

# ── Step 2: Find queen spot REACHABLE from pawn anchor, on higher ground ────
# Dijkstra outward from pawn_anchor, pick first hex with elev > 150
dummy = Piece(location=pawn_anchor)
queen_idx = None
for min_elev in [180, 150, 120, 80]:
    # Search outward for elevated reachable hex
    pq = [(0.0, pawn_anchor)]
    visited = set()
    while pq:
        cost, cur = heapq.heappop(pq)
        if cur in visited: continue
        visited.add(cur)
        if cur != pawn_anchor and elevs[cur] >= min_elev and food_tiers[cur] <= 2:
            queen_idx = cur
            break
        for nb in grid.neighborsOf(cur):
            if nb in visited or nb in grid.invalidRegion or elevs[nb] <= 0: continue
            heapq.heappush(pq, (cost + _move_cost(elevs, cur, nb), nb))
    if queen_idx: break

print(f"Queen: hex {queen_idx}  elev={elevs[queen_idx]:.0f}  tier={food_tiers[queen_idx]}")

# Verify path exists
test_path = dummy.pathfind(queen_idx, grid, elevs)
print(f"Path pawn→queen: {len(test_path)} hexes, cost={test_path.cost:.1f}")
assert test_path, "No path — pick different spots!"

# ── Step 3: Place pieces ──────────────────────────────────────────────────
queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
FoodProfile.apply(queen)
queen.food = queen.diet * 4.5

# Pawns on best food hexes near anchor
pawns = []
for idx in best_food[:5]:
    p = board.add_piece(idx, PieceType.PAWN,
                        facing=_facing_toward(grid, idx, queen_idx),
                        country_id=1,
                        instructions=InstructionList(
                            [Instruction.HARVEST.value, Instruction.GIVE.value],
                            patrol=True))
    FoodProfile.apply(p)
    pawns.append(p)

# Bishops — start near pawns, loaded with food, route to queen and back
bishops = []
for i, start_near in enumerate(best_food[5:7]):
    b = board.add_piece(start_near, PieceType.BISHOP,
                        facing=_facing_toward(grid, start_near, queen_idx),
                        country_id=1,
                        instructions=InstructionList([]))
    FoodProfile.apply(b)
    b.food = b.food_capacity  # fully loaded courier

    # Route: bishop → queen → back to start
    path_out = b.pathfind(queen_idx, grid, elevs)
    path_back = Piece(location=queen_idx).pathfind(start_near, grid, elevs)

    if path_out and path_back:
        rules = (path_out.to_rules(grid, b.facing)
                 + [Instruction.GIVE.value]
                 + path_back.to_rules(grid, b.facing)
                 + [Instruction.GIVE.value])
        b.instructions = InstructionList(rules, cursor=0, patrol=True)
        print(f"Bishop {i}: hex {start_near} → queen ({len(path_out)} hops) "
              f"→ back ({len(path_back)} hops), {len(rules)} instructions")
    bishops.append(b)

all_pieces = [queen] + bishops + pawns

# ── Show the plan ──────────────────────────────────────────────────────────
if False:
    show(NotStr(board.render_with_sight(
        all_pieces, num_turns=12, facing_only=True)))


# ── Run simulator ────────────────────────────────────────────────────────────
# ── Before state ─────────────────────────────────────────────────────────────
print("=== BEFORE ===")
sim = FoodSimulator(
    grid=grid,
    elevations=elevs,
    food_tiers=food_tiers,
    pieces=all_pieces,
)
sim.summary()

#show(royal_guard)
#show(supply_corps)
#show(NotStr(board.render_with_facing(num_turns=12)))

sim.run(num_turns=25)

print("\nAfter:")
#sim.summary()
print(f"\nQueen food: {queen.food:.1f}  health: {queen.health:.0f}")
print(f"Events for queen: {len(sim.events_for_piece(queen.id))}")

if False:
    
    show(royal_guard)
    show(supply_corps)
    show(NotStr(board.render_with_sight(
        [queen] + bishops + pawns,
        num_turns=12,
        facing_only=True
    )))



# Sample queen's event log
for ev in sim.events_for_piece(queen.id)[:12]:
    print(f"  T{ev.turn:02d} {ev.event.value:<10} {ev.detail}  "
          f"{'±'+str(round(ev.amount,1)) if ev.amount else ''}")

The queen lives?

lets push the queen further away. She needs to drop a few pounds

# ── Board ──────────────────────────────────────────────────────────────────
board = PieceBoard.hilly(rings=8, radius=18, seed=7)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if   e <= 0:    food_tiers[i] = 0
    elif e < 60:    food_tiers[i] = 7
    elif e < 120:   food_tiers[i] = 5
    elif e < 180:   food_tiers[i] = 3
    elif e < 260:   food_tiers[i] = 1
    else:           food_tiers[i] = 0

# ── Step 1: Fertile cluster for pawns ──────────────────────────────────────
best_food = sorted(
    [i for i in range(len(elevs))
     if elevs[i] > 0 and i not in grid.invalidRegion and food_tiers[i] >= 5],
    key=lambda i: abs(grid.index_to_hexposition(i, mid))
)[:20]

pawn_anchor = best_food[0]
print(f"Pawn anchor: hex {pawn_anchor}  tier={food_tiers[pawn_anchor]}  elev={elevs[pawn_anchor]:.0f}")

# ── Step 2: Queen far away — minimum 6 hops from pawn anchor ───────────────
MIN_HOPS = 6  # ← the diet plan

dummy = Piece(location=pawn_anchor)
queen_idx = None
candidates = []

pq = [(0.0, pawn_anchor)]
visited = {}
parent = {pawn_anchor: None}

while pq:
    cost, cur = heapq.heappop(pq)
    if cur in visited: continue
    visited[cur] = cost
    
    # Count hops via parent chain
    hops = 0
    node = cur
    while parent.get(node) is not None:
        hops += 1
        node = parent[node]
    
    # Candidate: far enough, elevated, low food (she's on a hill, not a farm)
    if hops >= MIN_HOPS and elevs[cur] >= 120 and food_tiers[cur] <= 2:
        candidates.append((cur, hops, cost, elevs[cur]))
    
    for nb in grid.neighborsOf(cur):
        if nb in visited or nb in grid.invalidRegion or elevs[nb] <= 0: continue
        new_cost = cost + _move_cost(elevs, cur, nb)
        if nb not in parent or new_cost < visited.get(nb, float('inf')):
            parent[nb] = cur
            heapq.heappush(pq, (new_cost, nb))

# Pick the highest elevation among far candidates
if candidates:
    candidates.sort(key=lambda c: (-c[1], -c[3]))  # most hops, then highest
    queen_idx, q_hops, q_cost, q_elev = candidates[0]

print(f"Queen: hex {queen_idx}  elev={q_elev:.0f}  tier={food_tiers[queen_idx]}  "
      f"hops={q_hops}  path_cost={q_cost:.1f}")

test_path = dummy.pathfind(queen_idx, grid, elevs)
print(f"Verified path: {len(test_path)} hexes, cost={test_path.cost:.1f}")

# ── Step 3: Place pieces ──────────────────────────────────────────────────
queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
FoodProfile.apply(queen)
queen.food = queen.diet * 4  # modest reserves

# Pawns
pawns = []
for idx in best_food[:5]:
    p = board.add_piece(idx, PieceType.PAWN,
                        facing=_facing_toward(grid, idx, queen_idx),
                        country_id=1,
                        instructions=InstructionList(
                            [Instruction.HARVEST.value, Instruction.GIVE.value],
                            patrol=True))
    FoodProfile.apply(p)
    pawns.append(p)

# Bishops — 3 this time for the longer route
bishops = []
for i, start_near in enumerate(best_food[5:8]):
    b = board.add_piece(start_near, PieceType.BISHOP,
                        facing=_facing_toward(grid, start_near, queen_idx),
                        country_id=1,
                        instructions=InstructionList([]))
    FoodProfile.apply(b)
    b.food = b.food_capacity  # fully loaded

    path_out = b.pathfind(queen_idx, grid, elevs)
    path_back = Piece(location=queen_idx).pathfind(start_near, grid, elevs)

    if path_out and path_back:
        rules = (path_out.to_rules(grid, b.facing)
                 + [Instruction.GIVE.value]
                 + path_back.to_rules(grid, b.facing)
                 + [Instruction.GIVE.value])
        b.instructions = InstructionList(rules, cursor=0, patrol=True)
        print(f"  Bishop {i}: {len(path_out)} hops out, "
              f"{len(path_back)} back, {len(rules)} instructions")
    bishops.append(b)

all_pieces = [queen] + bishops + pawns

# ── Render plan ──────────────────────────────────────────────────────────
show(NotStr(board.render_with_sight(
    all_pieces, num_turns=15, facing_only=True)))

# ── Simulate ─────────────────────────────────────────────────────────────
sim = FoodSimulator(grid=grid, elevations=elevs,
                    food_tiers=food_tiers, pieces=all_pieces)

print(f"\n=== BEFORE === queen food={queen.food:.1f} diet={queen.diet}")
sim.run(num_turns=30)
sim.summary()

print(f"\nQueen: food={queen.food:.1f}  health={queen.health:.0f}")
show(sim)

# Queen's story
print("\nQueen's log:")
for ev in sim.events_for_piece(queen.id):
    if ev.event in (SimEventType.RECEIVED, SimEventType.STARVED, SimEventType.DIED):
        print(f"  T{ev.turn:02d} {ev.event.value:<10} {ev.detail}  ±{ev.amount:.1f}")


She dies

## Supply Planner

In [ ]:
#| export
@dataclass
class SupplyPlanner:
    terrain: Terrain
    settlement: int
    profiles: dict = field(default_factory=lambda: FOOD_PROFILES)

    @property
    def grid(self): return self.terrain.hexGrid
    @property
    def elevations(self): return self.terrain.elevations
    @property
    def food_tiers(self):
        return self.terrain.fields.get('food_yield',
            np.zeros(len(self.terrain.elevations), dtype=int))


    @property
    def bishop(self): return self.profiles[PieceType.BISHOP]
    @property
    def pawn(self): return self.profiles[PieceType.PAWN]
    @property
    def queen(self): return self.profiles[PieceType.QUEEN]

    # ── Core math ────────────────────────────────────────────

    def travel_turns(self, path_cost, move_strength=None):
        """How many turns to traverse a path of given cost."""
        ms = move_strength or 3  # bishop default
        return math.ceil(path_cost / ms)

    def net_delivery(self, path_cost, move_strength=None):
        """Food a bishop delivers after eating its own share on the round trip."""
        tt = self.travel_turns(path_cost, move_strength)
        eaten = 2 * tt * self.bishop.diet
        return max(0.0, self.bishop.food_capacity - eaten)

    def throughput(self, path_cost, n_bishops=1, move_strength=None):
        """Food delivered per turn with N bishops on this route."""
        tt = self.travel_turns(path_cost, move_strength)
        nd = self.net_delivery(path_cost, move_strength)
        if nd <= 0 or tt <= 0:
            return 0.0
        round_trip = 2 * tt
        return n_bishops * nd / round_trip

    def bishops_needed(self, path_cost, consumer_diet=None, move_strength=None):
        """Minimum bishops to sustain a consumer (default: queen) at this distance."""
        diet = consumer_diet or self.queen.diet
        tp1 = self.throughput(path_cost, n_bishops=1, move_strength=move_strength)
        if tp1 <= 0:
            return float('inf')  # unreachable
        return math.ceil(diet / tp1)

    def max_range(self, move_strength=None):
        """Maximum one-way path cost where a bishop can still deliver food."""
        ms = move_strength or 3
        max_tt = self.bishop.food_capacity / (2 * self.bishop.diet)
        return int(max_tt * ms)

    def startup_food(self, path_cost, move_strength=None):
        """How much food the consumer needs to survive until first delivery."""
        tt = self.travel_turns(path_cost, move_strength)
        return self.queen.diet * (tt + 1)  # +1 buffer turn

    # ── Harvest rate at settlement ───────────────────────────

    def harvest_rate(self, n_pawns=5):
        """Food per turn from N pawns at the settlement."""
        # Average tier of nearby fertile hexes
        nbs = [self.settlement] + list(self.grid.neighborsOf(self.settlement))
        tiers = [self.food_tiers[n] for n in nbs
                 if 0 <= n < len(self.food_tiers) and self.food_tiers[n] > 0]
        avg_tier = np.mean(tiers) if tiers else 0
        return n_pawns * self.pawn.harvest_strength * avg_tier

    def can_sustain(self, n_pawns, n_bishops, path_cost, n_consumers=1,
                    consumer_diet=None, move_strength=None):
        """Can this setup sustain N consumers at the given distance?"""
        diet = consumer_diet or self.queen.diet
        total_demand = n_consumers * diet + n_bishops * self.bishop.diet
        supply = self.harvest_rate(n_pawns)
        delivery = self.throughput(path_cost, n_bishops, move_strength)
        return supply >= total_demand and delivery >= n_consumers * diet

    # ── Reachability map ─────────────────────────────────────

    def supply_range_map(self, move_strength=None):
        """Dijkstra from settlement → dict of {hex: (path_cost, bishops_needed)}."""
        max_cost = self.max_range(move_strength)
        pq = [(0.0, self.settlement)]
        visited = {}

        while pq:
            cost, cur = heapq.heappop(pq)
            if cur in visited:
                continue
            visited[cur] = (cost, self.bishops_needed(cost, move_strength=move_strength))

            for nb in self.grid.neighborsOf(cur):
                if nb in self.grid.invalidRegion or self.elevations[nb] <= 0:
                    continue
                new_cost = cost + _move_cost(self.elevations, cur, nb)
                if new_cost <= max_cost and nb not in visited:
                    heapq.heappush(pq, (new_cost, nb))

        return visited

    def summary(self, n_pawns=5, n_bishops=2):
        """Print a quick supply line summary."""
        mr = self.max_range()
        hr = self.harvest_rate(n_pawns)
        print(f"Settlement hex {self.settlement}  "
              f"tier={self.food_tiers[self.settlement]}")
        print(f"  Max bishop range: {mr} path cost "
              f"({self.travel_turns(mr)} turns one-way)")
        print(f"  Harvest rate: {hr:.1f} food/turn ({n_pawns} pawns)")
        print(f"  Bishop capacity: {self.bishop.food_capacity}, "
              f"diet: {self.bishop.diet}/turn")
        print()
        for cost in range(2, mr + 1, 2):
            nd = self.net_delivery(cost)
            tp = self.throughput(cost, n_bishops)
            bn = self.bishops_needed(cost)
            sf = self.startup_food(cost)
            print(f"  cost={cost:2d}  net={nd:5.1f}  "
                  f"throughput={tp:.2f}/turn  "
                  f"bishops_needed={bn}  startup_food={sf:.0f}")


In [ ]:
#| export
@patch
def plan_route(self: SupplyPlanner, target, board=None):
    """Analyze a supply route to a target hex. Returns dict of metrics."""
    dummy = Piece(location=self.settlement)
    path = dummy.pathfind(target, self.grid, self.elevations)
    if not path:
        return None
    cost = path.cost
    return {
        'target': target,
        'path': path,
        'cost': cost,
        'travel_turns': self.travel_turns(cost),
        'net_delivery': self.net_delivery(cost),
        'bishops_needed': self.bishops_needed(cost),
        'startup_food': self.startup_food(cost),
        'throughput_per_bishop': self.throughput(cost, 1),
        'reachable': self.net_delivery(cost) > 0,
    }

@patch
def find_target(self: SupplyPlanner, min_elev=120, max_tier=2,
                min_cost=4.0, max_cost=None):
    """Find a reachable target hex: elevated, low food, within supply range."""
    max_cost = max_cost or self.max_range()
    pq = [(0.0, self.settlement)]
    visited = {}
    candidates = []

    while pq:
        cost, cur = heapq.heappop(pq)
        if cur in visited: continue
        visited[cur] = cost

        if (cur != self.settlement
            and self.elevations[cur] >= min_elev
            and self.food_tiers[cur] <= max_tier
            and cost >= min_cost):
            candidates.append((cur, cost, self.elevations[cur]))

        for nb in self.grid.neighborsOf(cur):
            if nb in self.grid.invalidRegion or self.elevations[nb] <= 0: continue
            new_cost = cost + _move_cost(self.elevations, cur, nb)
            if new_cost <= max_cost and nb not in visited:
                heapq.heappush(pq, (new_cost, nb))

    if not candidates:
        return None
    # Highest elevation among reachable candidates
    candidates.sort(key=lambda c: (-c[2], c[1]))
    idx, cost, elev = candidates[0]
    return idx

@patch
def create_supply_line(self: SupplyPlanner, target, board,
                       n_bishops=None, country_id=1):
    """Place and configure bishops on a supply route. Returns list of Pieces."""
    route = self.plan_route(target)
    if not route or not route['reachable']:
        print(f"⚠ Target hex {target} is unreachable from settlement {self.settlement}")
        return []

    n_bishops = n_bishops or route['bishops_needed']
    path_out = route['path']

    # Find start hexes near settlement for each bishop
    # Spread them across fertile hexes so they don't stack
    nearby = sorted(
        [i for i in [self.settlement] + list(self.grid.neighborsOf(self.settlement))
         if i not in self.grid.invalidRegion and self.elevations[i] > 0],
        key=lambda i: -self.food_tiers[i]  # prefer fertile start hexes
    )

    bishops = []
    occupied = {p.location for p in board.pieces if p.location is not None}

    for i in range(n_bishops):
        # Pick a start hex that isn't already occupied
        start = None
        for candidate in nearby:
            if candidate not in occupied:
                start = candidate
                break
        if start is None:
            start = nearby[i % len(nearby)]  # fallback: stack if needed

        b = board.add_piece(
            start, PieceType.BISHOP,
            facing=_facing_toward(self.grid, start, target),
            country_id=country_id,
            instructions=InstructionList([]))
        FoodProfile.apply(b)
        b.food = b.food_capacity  # fully loaded courier

        # Build route: outbound → GIVE → return → GIVE (pick up from pawns)
        path_back = Piece(location=target).pathfind(start, self.grid, self.elevations)
        if path_out and path_back:
            rules = (path_out.to_rules(self.grid, b.facing)
                     + [Instruction.GIVE.value]
                     + path_back.to_rules(self.grid, b.facing)
                     + [Instruction.GIVE.value])
            b.instructions = InstructionList(rules, cursor=0, patrol=True)

        occupied.add(start)
        bishops.append(b)

    print(f"🚚 Supply line: {n_bishops} bishops, "
          f"cost={route['cost']:.1f}, "
          f"{route['travel_turns']} turns one-way, "
          f"net delivery={route['net_delivery']:.1f}/trip")
    return bishops

@patch
def provision_consumer(self: SupplyPlanner, piece, target=None, path_cost=None):
    """Set a consumer's starting food to survive until first delivery."""
    if path_cost is None and target is not None:
        route = self.plan_route(target)
        path_cost = route['cost'] if route else 0
    food = self.startup_food(path_cost or 0)
    piece.food = min(food, piece.food_capacity)
    print(f"🍖 {piece.name} provisioned with {piece.food:.1f} food "
          f"(need {food:.0f} to survive delivery wait)")


In [ ]:
#| export
@patch
def __ft__(self: SupplyPlanner):
    """Supply range heatmap from settlement + throughput curve."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

    # ── 1. Throughput vs distance curve ──────────────────────
    ax = axes[0]
    mr = self.max_range()
    costs = np.arange(1, mr + 1, 0.5)
    tp1 = [self.throughput(c, 1) for c in costs]
    tp2 = [self.throughput(c, 2) for c in costs]
    tp3 = [self.throughput(c, 3) for c in costs]

    ax.plot(costs, tp1, '-',  color='#3498db', lw=2, label='1 bishop')
    ax.plot(costs, tp2, '--', color='#2ecc71', lw=2, label='2 bishops')
    ax.plot(costs, tp3, ':',  color='#e67e22', lw=2, label='3 bishops')
    ax.axhline(self.queen.diet, color='#e74c3c', ls='-.', lw=1.5,
               alpha=0.7, label=f'queen diet ({self.queen.diet})')
    ax.axhline(self.profiles[PieceType.KNIGHT].diet, color='#9b59b6',
               ls='-.', lw=1, alpha=0.5, label=f'knight diet')
    ax.set_title('Throughput vs Distance', fontweight='bold', fontsize=10)
    ax.set_xlabel('Path cost')
    ax.set_ylabel('Food/turn delivered')
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(axis='both', alpha=0.3, lw=0.5)

    # ── 2. Bishops needed heatmap by (cost, consumer_diet) ───
    ax = axes[1]
    diets = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
    cost_range = list(range(2, mr + 1, 2))
    grid_data = []
    for d in diets:
        row = []
        for c in cost_range:
            bn = self.bishops_needed(c, consumer_diet=d)
            row.append(min(bn, 10))  # cap for display
        grid_data.append(row)

    im = ax.imshow(grid_data, aspect='auto', cmap='YlOrRd',
                   vmin=0, vmax=8, origin='lower')
    ax.set_xticks(range(len(cost_range)))
    ax.set_xticklabels(cost_range, fontsize=7)
    ax.set_yticks(range(len(diets)))
    ax.set_yticklabels([f'{d:.1f}' for d in diets], fontsize=7)
    ax.set_xlabel('Path cost')
    ax.set_ylabel('Consumer diet')
    ax.set_title('Bishops Needed', fontweight='bold', fontsize=10)

    # Annotate cells
    for i, d in enumerate(diets):
        for j, c in enumerate(cost_range):
            val = grid_data[i][j]
            txt = '∞' if val >= 10 else str(val)
            color = 'white' if val > 4 else 'black'
            ax.text(j, i, txt, ha='center', va='center',
                    fontsize=7, color=color, fontweight='bold')

    plt.colorbar(im, ax=ax, shrink=0.8, label='Bishops')

    hr = self.harvest_rate()
    fig.suptitle(
        f"Supply Planner — settlement hex {self.settlement}, "
        f"max range {mr}, harvest {hr:.0f}/turn",
        fontsize=10, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Bishop: cap={self.bishop.food_capacity} diet={self.bishop.diet} · "
          f"Pawn: harvest={self.pawn.harvest_strength}",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


I switched SupplyPlanner to use terain. does any of this need to be refactored?

better?

Should we have some linear program that optimizes which supply lines to create and who get them. we could have a max the colony kind of thing.

### ColonyOptimizer

In [ ]:
#| export
from scipy.optimize import milp, LinearConstraint, Bounds

@dataclass
class ColonyOptimizer:
    """Optimize supply line allocation across a colony."""
    planner: SupplyPlanner
    max_pawns: int = 8
    max_bishops: int = 6

    @dataclass
    class Consumer:
        piece: object           # Piece or just a name
        hex_idx: int
        diet: float
        priority: float = 1.0   # queen=10, knight=3, rook=2, etc.
        path_cost: float = 0.0
        reachable: bool = True

    def analyze_consumers(self, pieces):
        """Build Consumer list with route analysis."""
        consumers = []
        for p in pieces:
            route = self.planner.plan_route(p.location)
            if route is None:
                consumers.append(self.Consumer(
                    piece=p, hex_idx=p.location, diet=p.diet,
                    priority=_priority(p), path_cost=float('inf'),
                    reachable=False))
            else:
                consumers.append(self.Consumer(
                    piece=p, hex_idx=p.location, diet=p.diet,
                    priority=_priority(p), path_cost=route['cost'],
                    reachable=route['reachable']))
        return consumers

    def optimize(self, consumers):
        """
        Decide which consumers to feed and how many bishops per route.
        
        Returns dict with:
          - assignments: list of (consumer, n_bishops, throughput)
          - total_fed: number of consumers fed
          - total_priority: sum of priority of fed consumers
          - pawns_used, bishops_used
        """
        # Filter to reachable consumers
        reachable = [c for c in consumers if c.reachable and c.path_cost > 0]
        if not reachable:
            return self._empty_result(consumers)

        n = len(reachable)
        sp = self.planner

        # ── Variables ──────────────────────────────────────────
        # For each consumer i:
        #   b_i = bishops assigned (integer ≥ 0)
        #   f_i = binary: is this consumer fed?
        # Layout: [b_0, b_1, ..., b_n-1, f_0, f_1, ..., f_n-1]

        # Precompute per-route throughput per bishop
        tp1 = [sp.throughput(c.path_cost, 1) for c in reachable]

        # ── Objective: maximize Σ priority_i * f_i ─────────────
        # milp minimizes, so negate priorities
        c_obj = np.zeros(2 * n)
        for i, con in enumerate(reachable):
            c_obj[n + i] = -con.priority  # maximize priority

        # ── Variable bounds ────────────────────────────────────
        # b_i ∈ [0, max_bishops], f_i ∈ [0, 1]
        lb = np.zeros(2 * n)
        ub = np.concatenate([
            np.full(n, self.max_bishops),   # b_i upper
            np.ones(n)                       # f_i upper
        ])

        # ── Integrality: all integer ───────────────────────────
        integrality = np.ones(2 * n)  # 1 = integer

        # ── Constraints ────────────────────────────────────────
        A_rows = []
        lb_cons = []
        ub_cons = []

        # 1. Total bishops ≤ max_bishops
        row = np.zeros(2 * n)
        row[:n] = 1.0
        A_rows.append(row)
        lb_cons.append(0)
        ub_cons.append(self.max_bishops)

        # 2. For each consumer: tp1_i * b_i ≥ diet_i * f_i
        #    → tp1_i * b_i - diet_i * f_i ≥ 0
        for i, con in enumerate(reachable):
            row = np.zeros(2 * n)
            row[i] = tp1[i]           # bishop throughput
            row[n + i] = -con.diet    # diet requirement
            A_rows.append(row)
            lb_cons.append(0)
            ub_cons.append(np.inf)

        # 3. Total food demand ≤ harvest capacity
        #    Σ (diet_i * f_i) + Σ (bishop_diet * b_i) ≤ harvest_rate
        harvest = sp.harvest_rate(self.max_pawns)
        row = np.zeros(2 * n)
        for i, con in enumerate(reachable):
            row[i] = sp.bishop.diet       # bishop consumption
            row[n + i] = con.diet         # consumer consumption
        A_rows.append(row)
        lb_cons.append(0)
        ub_cons.append(harvest)

        # 4. f_i can only be 1 if at least 1 bishop assigned
        #    f_i ≤ b_i  →  -b_i + f_i ≤ 0
        for i in range(n):
            row = np.zeros(2 * n)
            row[i] = -1.0
            row[n + i] = 1.0
            A_rows.append(row)
            lb_cons.append(-np.inf)
            ub_cons.append(0)

        A = np.array(A_rows)
        constraints = LinearConstraint(A, lb_cons, ub_cons)
        bounds = Bounds(lb, ub)

        # ── Solve ──────────────────────────────────────────────
        result = milp(c_obj, constraints=constraints, bounds=bounds,
                      integrality=integrality)

        if not result.success:
            print(f"⚠ Optimization failed: {result.message}")
            return self._empty_result(consumers)

        x = np.round(result.x).astype(int)
        b_vals = x[:n]
        f_vals = x[n:]

        # ── Build result ───────────────────────────────────────
        assignments = []
        for i, con in enumerate(reachable):
            assignments.append({
                'consumer': con,
                'bishops': b_vals[i],
                'fed': bool(f_vals[i]),
                'throughput': tp1[i] * b_vals[i],
                'surplus': tp1[i] * b_vals[i] - con.diet if f_vals[i] else 0,
            })

        return {
            'assignments': assignments,
            'total_fed': int(f_vals.sum()),
            'total_priority': sum(a['consumer'].priority
                                  for a in assignments if a['fed']),
            'bishops_used': int(b_vals.sum()),
            'harvest_available': harvest,
            'harvest_consumed': sum(
                a['bishops'] * sp.bishop.diet + (a['consumer'].diet if a['fed'] else 0)
                for a in assignments),
        }

    def _empty_result(self, consumers):
        return {'assignments': [], 'total_fed': 0, 'total_priority': 0,
                'bishops_used': 0, 'harvest_available': 0, 'harvest_consumed': 0}

    def print_plan(self, result):
        """Pretty-print the optimization result."""
        print(f"═══ Colony Supply Plan ═══")
        print(f"  Fed: {result['total_fed']} consumers  "
              f"(priority score: {result['total_priority']:.0f})")
        print(f"  Bishops: {result['bishops_used']}/{self.max_bishops}")
        print(f"  Harvest: {result['harvest_consumed']:.1f}"
              f"/{result['harvest_available']:.1f} food/turn")
        print()
        for a in sorted(result['assignments'],
                        key=lambda x: -x['consumer'].priority):
            c = a['consumer']
            status = "✅" if a['fed'] else "❌"
            name = getattr(c.piece, 'name', str(c.hex_idx))
            print(f"  {status} {name:<16} "
                  f"pri={c.priority:<4.0f} "
                  f"diet={c.diet:<4.1f} "
                  f"cost={c.path_cost:<5.1f} "
                  f"bishops={a['bishops']} "
                  f"tp={a['throughput']:.2f}/turn")


def _priority(piece):
    """Default priority by piece type."""
    return {
        PieceType.KING: 20, PieceType.QUEEN: 10,
        PieceType.ROOK: 5, PieceType.KNIGHT: 3,
        PieceType.BISHOP: 2, PieceType.PAWN: 1,
    }.get(piece.piece_type, 1)


In [ ]:
#| export
@patch
def __ft__(self: ColonyOptimizer):
    """Visualize the last optimization result."""
    # We need a result to display — run a quick check
    if not hasattr(self, '_last_result') or not self._last_result:
        return P("No optimization run yet — call .optimize() first",
                 cls="text-sm opacity-50")

    result = self._last_result
    assignments = sorted(result['assignments'],
                         key=lambda a: -a['consumer'].priority)

    if not assignments:
        return P("No reachable consumers", cls="text-sm opacity-50")

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

    names = [getattr(a['consumer'].piece, 'name', f"hex {a['consumer'].hex_idx}")
             for a in assignments]
    n = len(names)
    x = np.arange(n)

    # ── 1. Bishops allocated per consumer ────────────────────
    ax = axes[0]
    bishops = [a['bishops'] for a in assignments]
    colors = ['#2ecc71' if a['fed'] else '#e74c3c' for a in assignments]
    ax.barh(x, bishops, color=colors, edgecolor='#555', linewidth=0.5)
    ax.set_yticks(x)
    ax.set_yticklabels(names, fontsize=7)
    ax.set_xlabel('Bishops')
    ax.set_title('Bishop Allocation', fontweight='bold', fontsize=10)
    ax.axvline(0, color='#333', lw=0.5)
    ax.grid(axis='x', alpha=0.3)

    # ── 2. Throughput vs diet ────────────────────────────────
    ax = axes[1]
    throughputs = [a['throughput'] for a in assignments]
    diets = [a['consumer'].diet for a in assignments]

    bars = ax.barh(x, throughputs, color='#3498db', alpha=0.7,
                   edgecolor='#555', linewidth=0.5, label='throughput')
    # Overlay diet as markers
    for i, (tp, d) in enumerate(zip(throughputs, diets)):
        ax.plot(d, i, 'D', color='#e74c3c', ms=6, zorder=5)
    ax.plot([], [], 'D', color='#e74c3c', ms=6, label='diet')
    ax.set_yticks(x)
    ax.set_yticklabels(names, fontsize=7)
    ax.set_xlabel('Food/turn')
    ax.set_title('Throughput vs Diet', fontweight='bold', fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(axis='x', alpha=0.3)

    # ── 3. Harvest budget pie ────────────────────────────────
    ax = axes[2]
    harvest = result['harvest_available']
    consumed = result['harvest_consumed']
    bishop_eat = sum(a['bishops'] * self.planner.bishop.diet
                     for a in assignments)
    consumer_eat = sum(a['consumer'].diet for a in assignments if a['fed'])
    spare = max(0, harvest - consumed)

    sizes = [consumer_eat, bishop_eat, spare]
    labels = [f'Consumers\n{consumer_eat:.1f}',
              f'Bishops\n{bishop_eat:.1f}',
              f'Spare\n{spare:.1f}']
    pie_colors = ['#e74c3c', '#3498db', '#95a5a6']

    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=pie_colors,
        autopct='%1.0f%%', startangle=90,
        textprops={'fontsize': 7})
    for t in autotexts:
        t.set_fontsize(7)
        t.set_fontweight('bold')
    ax.set_title('Harvest Budget', fontweight='bold', fontsize=10)

    fig.suptitle(
        f"Colony Plan — {result['total_fed']} fed, "
        f"{result['bishops_used']}/{self.max_bishops} bishops, "
        f"priority={result['total_priority']:.0f}",
        fontsize=10, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"✅ {result['total_fed']} fed · "
          f"❌ {len(assignments) - result['total_fed']} unfed · "
          f"🌾 {spare:.1f} food/turn spare",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


Can we have a demo using supply planner?

In [ ]:
#| export
def SupplyOverlay(settlement, field='bishops_needed', planner=None,
                  cmap_name='RdYlGn_r', opacity=0.35, **kw) -> OverlaySpec:
    """Heatmap of supply reachability from a settlement.

    field: 'bishops_needed' | 'net_delivery' | 'path_cost' | 'travel_turns'
    """

    def render(ctx):
        nonlocal planner
        if planner is None:
            planner = SupplyPlanner(ctx.terrain, settlement=settlement)

        srm = planner.supply_range_map()
        if not srm:
            return ""

        grid = ctx.grid

        # Compute field values
        field_vals = {}
        for idx, (cost, bn) in srm.items():
            if field == 'bishops_needed':
                field_vals[idx] = min(bn, 8)
            elif field == 'net_delivery':
                field_vals[idx] = planner.net_delivery(cost)
            elif field == 'path_cost':
                field_vals[idx] = cost
            elif field == 'travel_turns':
                field_vals[idx] = planner.travel_turns(cost)

        if not field_vals:
            return ""

        vals = list(field_vals.values())
        vmin, vmax = min(vals), max(vals)
        if field == 'net_delivery':
            cmap = plt.cm.get_cmap('RdYlGn')     # green=high delivery=good
        else:
            cmap = plt.cm.get_cmap(cmap_name)     # red=high cost=bad

        svg = ""
        for idx, val in field_vals.items():
            if idx < 0 or idx >= len(grid.hexes):
                continue
            t = (val - vmin) / (vmax - vmin) if vmax > vmin else 0.5
            rgba = cmap(t)
            color = f"rgb({int(rgba[0]*255)},{int(rgba[1]*255)},{int(rgba[2]*255)})"
            h = grid.hexes[idx]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            svg += (f'<polygon points="{pts}" fill="{color}" '
                    f'opacity="{opacity:.2f}" stroke="none"/>\n')

        # Settlement marker
        sc = grid.hexes[settlement].center
        r = grid.radius if hasattr(grid, 'radius') else 20
        svg += (f'<circle cx="{sc.x:.0f}" cy="{sc.y:.0f}" r="{r*0.6:.0f}" '
                f'fill="#FFD700" stroke="#333" stroke-width="2" opacity="0.9"/>\n')
        svg += (f'<text x="{sc.x:.0f}" y="{sc.y+4:.0f}" text-anchor="middle" '
                f'font-size="{r*0.6:.0f}" font-weight="bold" fill="#333">⛺</text>\n')

        return svg

    return OverlaySpec("supply_range", render, priority=42)


def ColonyPlanOverlay(planner, result, **kw) -> OverlaySpec:
    """Show colony optimization result: settlement, routes, and target markers.

    Args:
        planner: SupplyPlanner (provides settlement + pathfinding)
        result:  dict from ColonyOptimizer.optimize()
    """

    def render(ctx):
        grid = ctx.grid
        r = grid.radius if hasattr(grid, 'radius') else 20
        svg = ""

        settlement = planner.settlement
        sc = grid.hexes[settlement].center

        assignments = result.get('assignments', [])
        if not assignments:
            return ""

        for a in assignments:
            consumer = a['consumer']
            target_idx = consumer.hex_idx
            if target_idx < 0 or target_idx >= len(grid.hexes):
                continue

            tc = grid.hexes[target_idx].center
            fed = a['fed']
            n_bish = a['bishops']

            # ── Route line ──
            dummy = Piece(location=settlement)
            path = dummy.pathfind(target_idx, grid, planner.elevations)
            if path and len(path) > 1:
                route_color = "#2ecc71" if fed else "#e74c3c"
                points = []
                for hex_idx in path.hexes:
                    if 0 <= hex_idx < len(grid.hexes):
                        hc = grid.hexes[hex_idx].center
                        points.append(f"{hc.x:.0f},{hc.y:.0f}")
                if points:
                    svg += (f'<polyline points="{" ".join(points)}" '
                            f'fill="none" stroke="{route_color}" '
                            f'stroke-width="{max(2, r*0.12):.1f}" '
                            f'stroke-dasharray="6,4" '
                            f'stroke-opacity="0.7" '
                            f'marker-end="url(#arrowhead_{route_color[1:]})"/>\n')

            # ── Target marker ──
            marker_color = "#2ecc71" if fed else "#e74c3c"
            marker_r = r * 0.55
            svg += (f'<circle cx="{tc.x:.0f}" cy="{tc.y:.0f}" r="{marker_r:.0f}" '
                    f'fill="{marker_color}" fill-opacity="0.25" '
                    f'stroke="{marker_color}" stroke-width="2.5"/>\n')

            # ── Piece type icon ──
            ptype = getattr(consumer.piece, 'piece_type', None)
            icon = ptype.icon if ptype else "?"
            svg += (f'<text x="{tc.x:.0f}" y="{tc.y+r*0.15:.0f}" '
                    f'text-anchor="middle" font-size="{r*0.55:.0f}" '
                    f'fill="{marker_color}" font-weight="bold">{icon}</text>\n')

            # ── Bishop count label ──
            if n_bish > 0:
                lx = tc.x + marker_r * 0.9
                ly = tc.y - marker_r * 0.7
                svg += (f'<circle cx="{lx:.0f}" cy="{ly:.0f}" r="{r*0.28:.0f}" '
                        f'fill="#3498db" stroke="#fff" stroke-width="1.5"/>\n')
                svg += (f'<text x="{lx:.0f}" y="{ly+r*0.1:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.3:.0f}" '
                        f'fill="white" font-weight="bold">{n_bish}</text>\n')

        # ── Arrowhead defs ──
        for color_hex in ['2ecc71', 'e74c3c']:
            aid = f"arrowhead_{color_hex}"
            arrow_def = SVGDef("marker", aid,
                f'<path d="M0,0 L0,6 L6,3 z" fill="#{color_hex}"/>',
                markerWidth=6, markerHeight=6,
                refX=5, refY=3,
                orient="auto", markerUnits="strokeWidth")
            ctx.builder.add_definition(arrow_def)

        # ── Settlement marker (on top) ──
        svg += (f'<circle cx="{sc.x:.0f}" cy="{sc.y:.0f}" r="{r*0.65:.0f}" '
                f'fill="#FFD700" stroke="#333" stroke-width="2.5" opacity="0.9"/>\n')
        svg += (f'<text x="{sc.x:.0f}" y="{sc.y+r*0.18:.0f}" text-anchor="middle" '
                f'font-size="{r*0.6:.0f}" font-weight="bold" fill="#333">⛺</text>\n')

        return svg

    return OverlaySpec("colony_plan", render, priority=72)


maybe we should have an overlayo that show the planners targets in the TerrainDisplay

This is quite a bit we have built up and I think this would be a good git checkpoint. That gets rid of the ai conversations so I need to put it into a note so you can think better about where we are headed and what we have built so far. I think these supply planner and colony optimizer are good foundations for a supply chain system. The visualizations are also amazing. This feels very different to any game I have played, but it does have a nice simple set of rules at is heart that I want to keep. Can you give me a good write up?

So my question for game design is what to we want the player to decide. I do think keeping the micromanagement is fine (if not cool and unqiue), but I am think we give them a heat map of where they might want to expand towards and they pick a few, and then we optimize. (For the AI they would do the steps themselves.)

Yes lets build a heat map. do we need multiple to show the trade offs?

In [ ]:
from HexMagic.primitives import HexGrid
??HexGrid

In [ ]:
??Terrain

I think we should start switching this to terrain. We quite literally have heat maps for it.

In [ ]:
#| export
@patch
def compute_supply_fields(self: SupplyPlanner):
    """Populate terrain.fields with supply metrics from settlement."""
    range_map = self.supply_range_map()  # {hex: (path_cost, bishops_needed)}
    n = len(self.elevations)

    costs      = np.full(n, np.nan)
    bishops    = np.full(n, np.nan)
    delivery   = np.full(n, 0.0)
    max_diet   = np.full(n, 0.0)
    efficiency = np.full(n, 0.0)

    for idx, (cost, bn) in range_map.items():
        costs[idx]      = cost
        bishops[idx]    = min(bn, 10)
        delivery[idx]   = self.net_delivery(cost)
        max_diet[idx]   = self.throughput(cost, 1)
        efficiency[idx] = delivery[idx] / cost if cost > 0 else 0

    self.terrain.fields['supply_cost']      = costs
    self.terrain.fields['bishops_needed']   = bishops
    self.terrain.fields['net_delivery']     = delivery
    self.terrain.fields['max_diet']         = max_diet
    self.terrain.fields['supply_efficiency'] = efficiency

    print(f"📡 Supply fields computed from hex {self.settlement}, "
          f"{len(range_map)} reachable hexes, max range {self.max_range()}")


In [ ]:
#| export
@patch
def render_supply_overlay(self: SupplyPlanner,
                          field='bishops_needed',
                          layer_name='supply_heatmap'):
    """Render a supply field onto the terrain using its SVG pipeline."""
    if field not in self.terrain.fields:
        self.compute_supply_fields()

    data = self.terrain.fields[field]
    grid = self.grid

    # Color stops per field type
    stops = {
        'bishops_needed':   [(1, "#2ecc71"), (2, "#f1c40f"), (4, "#e67e22"), (6, "#e74c3c"), (10, "#1a1a2e")],
        'net_delivery':     [(0, "#e74c3c"), (5, "#f1c40f"), (10, "#2ecc71"), (20, "#27ae60")],
        'max_diet':         [(0, "#e74c3c"), (1, "#f39c12"), (2, "#f1c40f"), (3, "#2ecc71"), (5, "#27ae60")],
        'supply_efficiency':[(0, "#e74c3c"), (0.5, "#f1c40f"), (1, "#2ecc71"), (2, "#27ae60")],
        'supply_cost':      [(0, "#2ecc71"), (5, "#f1c40f"), (10, "#e67e22"), (20, "#e74c3c")],
    }

    color_stops = stops.get(field, stops['supply_cost'])

    def lerp_color(val):
        """Interpolate between color stops."""
        if np.isnan(val): return None
        for i in range(len(color_stops) - 1):
            v0, c0 = color_stops[i]
            v1, c1 = color_stops[i + 1]
            if val <= v1:
                t = (val - v0) / (v1 - v0) if v1 != v0 else 0
                t = max(0, min(1, t))
                r0, g0, b0 = int(c0[1:3], 16), int(c0[3:5], 16), int(c0[5:7], 16)
                r1, g1, b1 = int(c1[1:3], 16), int(c1[3:5], 16), int(c1[5:7], 16)
                r = int(r0 + (r1 - r0) * t)
                g = int(g0 + (g1 - g0) * t)
                b = int(b0 + (b1 - b0) * t)
                return f"#{r:02x}{g:02x}{b:02x}"
        return color_stops[-1][1]

    # Build SVG overlay using the same hex polygon approach as render_icon_temperature
    overlay = ""
    for i in range(len(data)):
        if self.elevations[i] <= 0: continue  # skip ocean
        color = lerp_color(data[i])
        if color is None: continue

        hex_obj = grid.hexes[i]
        points = " ".join([f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices()])
        overlay += f'\t<polygon points="{points}" fill="{color}" opacity="0.5"/>\n'

    # Settlement marker
    s_hex = grid.hexes[self.settlement]
    sc = s_hex.center   # ← attribute, not method
    overlay += (f'<circle cx="{sc.x}" cy="{sc.y}" r="{grid.radius * 0.6}" '
                f'fill="none" stroke="#f1c40f" stroke-width="2.5" opacity="0.9"/>\n')


    grid.builder.adjust(layer_name, overlay)
    return self.terrain


Can you build a demo using myTerr. pick a settlement at a good watershed and show the supply overlay.

Can we make a SuppyOverlay like the way we did FoodOverlay using our system?

In [ ]:
#| export
def SupplyOverlay(settlement: int,
                  field: str = 'bishops_needed',
                  planner: SupplyPlanner = None,
                  opacity: float = 0.5,
                  **kw) -> OverlaySpec:
    """Heatmap overlay showing supply range from a settlement.
    
    Fields:
      'bishops_needed'    — green (1) → red (6+) → black (unreachable)
      'net_delivery'      — red (0) → green (high delivery)
      'max_diet'          — red (0) → green (high sustainable diet)
      'supply_efficiency' — red (0) → green (high efficiency)
      'supply_cost'       — green (cheap) → red (expensive)
    """
    STOPS = {
        'bishops_needed':    [(1, "#2ecc71"), (2, "#f1c40f"), (4, "#e67e22"),
                              (6, "#e74c3c"), (10, "#1a1a2e")],
        'net_delivery':      [(0, "#e74c3c"), (5, "#f1c40f"),
                              (10, "#2ecc71"), (20, "#27ae60")],
        'max_diet':          [(0, "#e74c3c"), (1, "#f39c12"), (2, "#f1c40f"),
                              (3, "#2ecc71"), (5, "#27ae60")],
        'supply_efficiency': [(0, "#e74c3c"), (0.5, "#f1c40f"),
                              (1, "#2ecc71"), (2, "#27ae60")],
        'supply_cost':       [(0, "#2ecc71"), (5, "#f1c40f"),
                              (10, "#e67e22"), (20, "#e74c3c")],
    }

    def lerp_color(val, color_stops):
        if np.isnan(val): return None
        if val <= color_stops[0][0]: return color_stops[0][1]
        for i in range(len(color_stops) - 1):
            v0, c0 = color_stops[i]
            v1, c1 = color_stops[i + 1]
            if val <= v1:
                t = (val - v0) / (v1 - v0) if v1 != v0 else 0
                t = max(0, min(1, t))
                r0, g0, b0 = int(c0[1:3],16), int(c0[3:5],16), int(c0[5:7],16)
                r1, g1, b1 = int(c1[1:3],16), int(c1[3:5],16), int(c1[5:7],16)
                r = int(r0 + (r1-r0)*t)
                g = int(g0 + (g1-g0)*t)
                b = int(b0 + (b1-b0)*t)
                return f"#{r:02x}{g:02x}{b:02x}"
        return color_stops[-1][1]

    def render(ctx):
        # Reuse existing planner or create one
        sp = planner or SupplyPlanner(ctx.terrain, settlement)

        # Compute if needed
        if field not in ctx.terrain.fields:
            sp.compute_supply_fields()

        data = ctx.terrain.fields[field]
        color_stops = STOPS.get(field, STOPS['bishops_needed'])
        grid = ctx.grid

        overlay = ""
        for i in range(len(data)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            color = lerp_color(data[i], color_stops)
            if color is None:
                continue

            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += (f'\t<polygon points="{pts}" '
                        f'fill="{color}" opacity="{opacity}"/>\n')

        # Settlement marker
        sc = grid.hexes[settlement].center
        r = grid.radius
        overlay += (f'<circle cx="{sc.x}" cy="{sc.y}" r="{r*0.6}" '
                    f'fill="none" stroke="#f1c40f" stroke-width="2.5" '
                    f'opacity="0.9"/>\n')
        overlay += (f'<text x="{sc.x}" y="{sc.y+4}" text-anchor="middle" '
                    f'font-size="{r*0.5}" fill="#f1c40f" font-weight="bold" '
                    f'font-family="sans-serif">⛺</text>\n')

        return overlay

    return OverlaySpec(f"supply_{field}", render, priority=45)


# Compare different fields
TerrainDisplay(
    CreamOverlay(),
    SupplyOverlay(settlement=best_settlement, field='net_delivery'),
    terrain=myTerr,
)


# ── Pick a good settlement: fertile, low elevation, central-ish ──────────
grid = myTerr.hexGrid
elevs = myTerr.elevations
mid = grid.middle

# Get food tiers (or compute simple ones from elevation)
if 'food_yield' in myTerr.fields:
    food_tiers = myTerr.fields['food_yield']
else:
    food_tiers = np.zeros(len(elevs), dtype=int)
    for i, e in enumerate(elevs):
        if   e <= 0:    food_tiers[i] = 0
        elif e < 60:    food_tiers[i] = 7
        elif e < 120:   food_tiers[i] = 5
        elif e < 180:   food_tiers[i] = 3
        elif e < 260:   food_tiers[i] = 1
        else:           food_tiers[i] = 0
    myTerr.fields['food_yield'] = food_tiers

# Score each hex: high food tier + low elevation + close to center + many fertile neighbors
def settlement_score(idx):
    if idx in grid.invalidRegion or elevs[idx] <= 0:
        return -999
    tier = food_tiers[idx]
    if tier < 4:
        return -999
    # Count fertile neighbors
    nbs = grid.neighborsOf(idx)
    fertile_nbs = sum(1 for n in nbs
                      if n not in grid.invalidRegion
                      and elevs[n] > 0
                      and food_tiers[n] >= 3)
    # Distance from center (prefer central-ish)
    dist = abs(grid.index_to_hexposition(idx, mid))
    return tier * 3 + fertile_nbs * 2 - dist * 0.5

best_settlement = max(range(len(elevs)), key=settlement_score)
print(f"⛺ Settlement: hex {best_settlement}  "
      f"elev={elevs[best_settlement]:.0f}  "
      f"tier={food_tiers[best_settlement]}  "
      f"score={settlement_score(best_settlement):.1f}")

# ── Create planner and compute fields ────────────────────────────────────
planner = SupplyPlanner(terrain=myTerr, settlement=best_settlement)
planner.compute_supply_fields()
planner.summary()

# ── Render the base terrain + supply overlay ─────────────────────────────
myTerr.colorMap()
myTerr.hexGrid.update()

# Show bishops_needed heatmap
planner.render_supply_overlay('bishops_needed', layer_name='supply_heatmap')
show(NotStr(myTerr.hexGrid.builder.xml()))


Please make sure we are using the correct primtives. Hex Coordiantes postions can be tricky to understand. grid might have a different middle

### Battle Plans

In [ ]:
#| export
@dataclass
class BattlePlan:
    """An editable optimization result the player can tweak before committing."""
    optimizer: ColonyOptimizer
    assignments: list          # the optimizer's suggestion
    harvest_budget: float
    
    # Player overrides
    vetoed: set = field(default_factory=set)       # consumer hex_idx player said "no"
    boosted: dict = field(default_factory=dict)     # hex_idx → extra bishops
    locked: set = field(default_factory=set)        # hex_idx player said "must feed"
    
    def veto(self, hex_idx):
        """Player says: don't feed this one."""
        self.vetoed.add(hex_idx)
        return self._reoptimize()
    
    def lock(self, hex_idx):
        """Player says: this one MUST be fed, no matter what."""
        self.locked.add(hex_idx)
        return self._reoptimize()
    
    def boost(self, hex_idx, extra=1):
        """Player adds extra bishops to a route (insurance)."""
        self.boosted[hex_idx] = self.boosted.get(hex_idx, 0) + extra
        return self._reoptimize()
    
    def _reoptimize(self):
        """Re-run with player constraints applied."""
        # ... rerun optimizer with vetoed filtered out,
        #     locked as hard constraints, boosted as minimums
        return self
    
    def commit(self, board):
        """Execute the plan — place bishops, provision consumers."""
        placed = []
        for a in self.assignments:
            if not a['fed'] or a['consumer'].hex_idx in self.vetoed:
                continue
            bishops = self.optimizer.planner.create_supply_line(
                a['consumer'].hex_idx, board, n_bishops=a['bishops'])
            self.optimizer.planner.provision_consumer(
                a['consumer'].piece, target=a['consumer'].hex_idx)
            placed.extend(bishops)
        return placed


What can we do flesh this out. I do think we will comeback later when we add the other pieces - king is what must survive otherwise you loose. I am thinking of knights as fast pieces and rooks as slow tanks. but we will have to do the combat optimization part later as well (since I haven't yet built out that system). Any thoughts on this?

Can we do a similar BattlePlanOverlay as we did for SupplyOverlay. I know we would have to pass a battle plan in

In [ ]:
#| export
def BattlePlanOverlay(plan: BattlePlan,
                      show_routes: bool = True,
                      opacity: float = 0.6,
                      **kw) -> OverlaySpec:
    """Render a BattlePlan on the hex map: routes, consumer markers, bishop counts.
    
    Colors:
      ✅ fed + locked  → gold border
      ✅ fed           → green
      ❌ unfed         → red
      🚫 vetoed        → grey strikethrough
    """
    
    def render(ctx):
        grid = ctx.grid
        sp = plan.optimizer.planner
        settlement = sp.settlement
        sc = grid.hexes[settlement].center
        r = grid.radius
        
        overlay = ""
        
        for a in plan.assignments:
            c = a['consumer']
            hex_idx = c.hex_idx
            if hex_idx < 0 or hex_idx >= len(grid.hexes):
                continue
            
            tc = grid.hexes[hex_idx].center
            vetoed = hex_idx in plan.vetoed
            locked = hex_idx in plan.locked
            fed = a['fed'] and not vetoed
            
            # ── Route line from settlement to consumer ──────────
            if show_routes and not vetoed:
                # Try to get actual path for a nice hex-following line
                route = sp.plan_route(hex_idx)
                if route and route['path']:
                    path_pts = []
                    for pidx in route['path'].hexes:
                        pc = grid.hexes[pidx].center
                        path_pts.append(f"{pc.x:.0f},{pc.y:.0f}")
                    pts_str = " ".join(path_pts)
                    
                    line_color = "#2ecc71" if fed else "#e74c3c"
                    line_width = 2.5 if fed else 1.5
                    dash = "" if fed else 'stroke-dasharray="6,4"'
                    
                    overlay += (f'<polyline points="{pts_str}" '
                                f'fill="none" stroke="{line_color}" '
                                f'stroke-width="{line_width}" '
                                f'opacity="{opacity * 0.7}" '
                                f'{dash} stroke-linecap="round"/>\n')
                else:
                    # Fallback: straight line
                    line_color = "#2ecc71" if fed else "#e74c3c"
                    overlay += (f'<line x1="{sc.x:.0f}" y1="{sc.y:.0f}" '
                                f'x2="{tc.x:.0f}" y2="{tc.y:.0f}" '
                                f'stroke="{line_color}" stroke-width="1.5" '
                                f'opacity="{opacity * 0.5}" '
                                f'stroke-dasharray="4,3"/>\n')
            
            # ── Consumer hex highlight ──────────────────────────
            hex_obj = grid.hexes[hex_idx]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            
            if vetoed:
                fill_color = "#95a5a6"
                stroke_color = "#7f8c8d"
            elif locked:
                fill_color = "#f1c40f"
                stroke_color = "#f39c12"
            elif fed:
                fill_color = "#2ecc71"
                stroke_color = "#27ae60"
            else:
                fill_color = "#e74c3c"
                stroke_color = "#c0392b"
            
            overlay += (f'<polygon points="{pts}" '
                        f'fill="{fill_color}" opacity="{opacity * 0.4}" '
                        f'stroke="{stroke_color}" stroke-width="2" '
                        f'stroke-opacity="{opacity}"/>\n')
            
            # ── Piece icon + bishop count label ─────────────────
            piece_type = c.piece.piece_type if hasattr(c.piece, 'piece_type') else None
            icon = getattr(piece_type, 'icon', '●') if piece_type else '●'
            name = getattr(c.piece, 'name', '')[:6]
            
            # Icon
            overlay += (f'<text x="{tc.x:.0f}" y="{tc.y - r*0.15:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.6}" '
                        f'fill="{stroke_color}" opacity="0.9">'
                        f'{icon}</text>\n')
            
            # Bishop count badge
            if a['bishops'] > 0 and not vetoed:
                bx = tc.x + r * 0.5
                by = tc.y - r * 0.4
                badge_color = "#2ecc71" if fed else "#e74c3c"
                overlay += (f'<circle cx="{bx:.0f}" cy="{by:.0f}" '
                            f'r="{r*0.25}" fill="{badge_color}" '
                            f'stroke="white" stroke-width="1"/>\n')
                overlay += (f'<text x="{bx:.0f}" y="{by + r*0.08:.0f}" '
                            f'text-anchor="middle" font-size="{r*0.25}" '
                            f'fill="white" font-weight="bold" '
                            f'font-family="sans-serif">'
                            f'{a["bishops"]}</text>\n')
            
            # Status label below
            status = "🚫" if vetoed else ("🔒" if locked else ("✅" if fed else "❌"))
            overlay += (f'<text x="{tc.x:.0f}" y="{tc.y + r*0.55:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.3}" '
                        f'fill="#333" font-family="sans-serif" '
                        f'font-weight="bold">{status} {name}</text>\n')
            
            # Vetoed strikethrough
            if vetoed:
                overlay += (f'<line x1="{tc.x - r*0.6:.0f}" y1="{tc.y:.0f}" '
                            f'x2="{tc.x + r*0.6:.0f}" y2="{tc.y:.0f}" '
                            f'stroke="#e74c3c" stroke-width="2.5" '
                            f'opacity="0.8"/>\n')
        
        # ── Settlement marker ───────────────────────────────────
        overlay += (f'<circle cx="{sc.x}" cy="{sc.y}" r="{r*0.7}" '
                    f'fill="none" stroke="#f1c40f" stroke-width="3" '
                    f'opacity="0.9"/>\n')
        overlay += (f'<text x="{sc.x}" y="{sc.y+4}" text-anchor="middle" '
                    f'font-size="{r*0.5}" fill="#f1c40f" font-weight="bold" '
                    f'font-family="sans-serif">⛺</text>\n')
        
        # ── Summary legend ──────────────────────────────────────
        fed_count = sum(1 for a in plan.assignments
                        if a['fed'] and a['consumer'].hex_idx not in plan.vetoed)
        total = len(plan.assignments)
        bishops_used = sum(a['bishops'] for a in plan.assignments
                           if a['consumer'].hex_idx not in plan.vetoed)
        
        lx, ly = r * 0.5, r * 0.5
        overlay += (f'<rect x="{lx}" y="{ly}" width="{r*8}" height="{r*1.2}" '
                    f'rx="4" fill="white" opacity="0.85" '
                    f'stroke="#ccc" stroke-width="1"/>\n')
        overlay += (f'<text x="{lx + r*0.3}" y="{ly + r*0.8}" '
                    f'font-size="{r*0.35}" font-family="sans-serif" '
                    f'fill="#333" font-weight="bold">'
                    f'🗺 {fed_count}/{total} fed · '
                    f'⛪ {bishops_used} bishops · '
                    f'🌾 {plan.harvest_budget:.0f}/turn</text>\n')
        
        return overlay
    
    return OverlaySpec("battle_plan", render, priority=50)


Can we do a BattlePieceOverlay that would show the individual piece movements that are proposed?

def BattlePieceOverlay(plan: BattlePlan,
                       num_turns: int = 12,
                       show_bishops: bool = True,
                       show_pawns: bool = False,
                       show_consumers: bool = True,
                       opacity: float = 0.85,
                       **kw) -> OverlaySpec:
    """Render proposed piece movements from a BattlePlan.
    
    Shows simulated patrol routes for:
      - Bishops on supply runs (arrows + GIVE glyphs)
      - Consumers at their positions (PAUSE/DEFEND glyphs)
      - Optionally pawns at settlement (HARVEST/GIVE loops)
    """

    def render(ctx):
        grid = ctx.grid
        elevs = ctx.terrain.elevations
        countries = ctx.terrain.fields.get('country')
        r = grid.radius
        svg = ""

        # Collect all pieces from the plan
        pieces_to_show = []

        for a in plan.assignments:
            c = a['consumer']
            if c.hex_idx in plan.vetoed:
                continue

            # Consumer piece — show their orders (PAUSE, DEFEND, etc.)
            if show_consumers and hasattr(c, 'piece') and c.piece is not None:
                p = c.piece
                if p.location is not None and p.location >= 0:
                    pieces_to_show.append(('consumer', p, a['fed']))

        # Bishops created by the plan (if committed, they'll be on the board)
        # For pre-commit preview, we simulate from the plan's assignments
        if show_bishops:
            sp = plan.optimizer.planner
            for a in plan.assignments:
                c = a['consumer']
                if not a['fed'] or c.hex_idx in plan.vetoed:
                    continue
                if a['bishops'] <= 0:
                    continue

                # Create a phantom bishop for each assigned route
                route = sp.plan_route(c.hex_idx)
                if not route or not route['path']:
                    continue

                for bi in range(a['bishops']):
                    # Find a start hex near settlement
                    start = sp.settlement
                    nbs = list(grid.neighborsOf(start))
                    valid = [n for n in nbs
                             if n not in grid.invalidRegion and elevs[n] > 0]
                    b_start = valid[bi % len(valid)] if valid else start

                    # Build bishop patrol: out → GIVE → back → GIVE
                    path_out = route['path']
                    path_back_piece = Piece(location=c.hex_idx)
                    path_back = path_back_piece.pathfind(b_start, grid, elevs)

                    b_facing = _facing_toward(grid, b_start, c.hex_idx)
                    if path_out and path_back:
                        rules = (path_out.to_rules(grid, b_facing)
                                 + [Instruction.GIVE.value]
                                 + path_back.to_rules(grid, b_facing)
                                 + [Instruction.GIVE.value])
                    else:
                        rules = [Instruction.FORWARD.value,
                                 Instruction.GIVE.value]

                    phantom = Piece(
                        location=b_start,
                        piece_type=PieceType.BISHOP,
                        facing=b_facing,
                        instructions=InstructionList(rules, cursor=0, patrol=True),
                        owner_id=c.piece.owner_id if hasattr(c.piece, 'owner_id') else 1,
                        name=f"Supply Bishop {bi+1}",
                    )
                    # Copy flag from consumer if available
                    if hasattr(c.piece, 'flag'):
                        phantom.flag = c.piece.flag
                    FoodProfile.apply(phantom)
                    pieces_to_show.append(('bishop', phantom, True))

        # Pawns at settlement
        if show_pawns:
            sp = plan.optimizer.planner
            for pi in range(plan.optimizer.max_pawns):
                nbs = [sp.settlement] + list(grid.neighborsOf(sp.settlement))
                valid = [n for n in nbs
                         if n not in grid.invalidRegion and elevs[n] > 0]
                p_loc = valid[pi % len(valid)] if valid else sp.settlement

                phantom = Piece(
                    location=p_loc,
                    piece_type=PieceType.PAWN,
                    facing=random.randint(0, 5),
                    instructions=InstructionList(
                        [Instruction.HARVEST.value, Instruction.GIVE.value],
                        cursor=0, patrol=True),
                    name=f"Farmer {pi+1}",
                )
                FoodProfile.apply(phantom)
                pieces_to_show.append(('pawn', phantom, True))

        # ── Render each piece's plan ─────────────────────────────
        for role, piece, fed in pieces_to_show:
            if piece.location is None or piece.location < 0:
                continue
            if piece.location >= len(grid.hexes):
                continue

            color = "#3498db"  # bishop blue
            if role == 'consumer':
                color = "#2ecc71" if fed else "#e74c3c"
            elif role == 'pawn':
                color = "#8D6E63"

            # Use flag color if available
            if hasattr(piece, 'flag') and piece.flag:
                color = piece.flag.primary

            g = DiagramGlyphs(color=color, size=r * 0.4, opacity=opacity)

            # Simulate
            steps = piece.simulate(grid, elevs,
                                   num_turns=num_turns,
                                   countries=countries)
            if not steps and role == 'consumer':
                # Static consumer — just draw at position
                cx, cy = grid.hexes[piece.location].center.x, grid.hexes[piece.location].center.y
                svg += g.start_marker(cx, cy, r=r * 0.5)
                continue

            # Arrow style per role
            dash = f"{r*0.2:.0f},{r*0.1:.0f}" if role == 'bishop' else f"{r*0.15:.0f},{r*0.08:.0f}"
            arrow_style = StyleCSS(
                f"bp_{role}_{id(piece) % 9999}",
                stroke=color, stroke_width=max(1, r * 0.06),
                fill="none", opacity=str(opacity * 0.7),
                stroke_dasharray=dash,
            )
            grid.builder.add_style(arrow_style)

            # Start marker
            c = grid.hexes[piece.location].center
            svg += g.start_marker(c.x, c.y, r=r * 0.45)

            # Steps
            prev_idx = piece.location
            turn_ends = {}
            for i, s in enumerate(steps):
                turn_ends[s.turn] = i

            for i, step in enumerate(steps):
                idx = step.hex_idx
                if idx < 0 or idx >= len(grid.hexes):
                    prev_idx = idx
                    continue

                cx = grid.hexes[idx].center.x
                cy = grid.hexes[idx].center.y

                if step.instruction == Instruction.FORWARD and not step.blocked:
                    if 0 <= prev_idx < len(grid.hexes) and prev_idx != idx:
                        svg += grid.arrow(prev_idx, idx, style=arrow_style, factor=0.3)
                elif step.blocked:
                    svg += g.blocked_x(cx, cy)
                else:
                    svg += g.action_glyph(step.instruction, cx, cy)

                prev_idx = idx

            # Turn-end labels (merged ranges)
            for hex_idx, t0, t1 in _turn_end_labels(steps):
                if 0 <= hex_idx < len(grid.hexes):
                    cx = grid.hexes[hex_idx].center.x
                    cy = grid.hexes[hex_idx].center.y
                    comp = piece.flag.comp if hasattr(piece, 'flag') and piece.flag else "#555"
                    svg += _range_label(g, cx, cy, t0, t1, comp)

        return svg

    return OverlaySpec("battle_pieces", render, priority=55)


Can you build a nice demo using my terr to show a battle plan to secure a nearby peak and a fertile river valley?

In [ ]:
myTerr.hexGrid.builder.layers = []

In [ ]:
# ── Setup: ensure food tiers exist ──────────────────────────────────────
grid = myTerr.hexGrid
elevs = myTerr.elevations
mid = grid.middle

if 'food_yield' not in myTerr.fields:
    fy = FoodYield(myTerr, basins, n_tiers=8)
    fy.compute()

food_tiers = myTerr.fields['food_yield']

# ── 1. Find the best settlement: fertile, good neighbors ───────────────
def settlement_score(idx):
    if idx in grid.invalidRegion or elevs[idx] <= 0: return -999
    tier = int(food_tiers[idx])
    if tier < 4: return -999
    nbs = grid.neighborsOf(idx)
    fertile_nbs = sum(1 for n in nbs
                      if n not in grid.invalidRegion
                      and elevs[n] > 0 and food_tiers[n] >= 3)
    dist = abs(grid.index_to_hexposition(idx, mid))
    return tier * 3 + fertile_nbs * 2 - dist * 0.3

settlement = max(range(len(elevs)), key=settlement_score)
print(f"⛺ Settlement: hex {settlement}  "
      f"elev={elevs[settlement]:.0f}  tier={int(food_tiers[settlement])}")

# ── 2. Find a nearby peak (strategic high ground) ──────────────────────
planner = SupplyPlanner(terrain=myTerr, settlement=settlement)
planner.compute_supply_fields()
max_cost = planner.max_range()

# Dijkstra outward — find highest reachable peak within supply range
peak_target = None
peak_elev = 0
pq = [(0.0, settlement)]
visited = {}

while pq:
    cost, cur = heapq.heappop(pq)
    if cur in visited: continue
    visited[cur] = cost

    # Peak candidate: high elevation, reachable, within supply range
    if (cur != settlement
        and elevs[cur] >= 200
        and cost <= max_cost * 0.7  # not at the absolute edge of range
        and elevs[cur] > peak_elev):
        peak_target = cur
        peak_elev = elevs[cur]

    for nb in grid.neighborsOf(cur):
        if nb in visited or nb in grid.invalidRegion or elevs[nb] <= 0: continue
        new_cost = cost + _move_cost(elevs, cur, nb)
        if new_cost <= max_cost and nb not in visited:
            heapq.heappush(pq, (new_cost, nb))

print(f"🏔 Peak target: hex {peak_target}  "
      f"elev={elevs[peak_target]:.0f}  "
      f"tier={int(food_tiers[peak_target])}  "
      f"cost={visited[peak_target]:.1f}")

# ── 3. Find a fertile river valley (high food, near water) ─────────────
valley_target = None
valley_score = -999

for idx, cost in visited.items():
    if idx == settlement or cost > max_cost * 0.6:
        continue
    tier = int(food_tiers[idx])
    if tier < 5:
        continue
    # Bonus for being near water (low-elevation neighbors)
    nbs = grid.neighborsOf(idx)
    water_adj = sum(1 for n in nbs if elevs[n] <= 0)
    low_adj = sum(1 for n in nbs
                  if 0 < elevs[n] < 80 and food_tiers[n] >= 4)
    score = tier * 2 + water_adj * 3 + low_adj * 2 - cost * 0.5

    # Must be somewhat distant from settlement (not right next door)
    if cost < 3:
        continue

    if score > valley_score:
        valley_score = score
        valley_target = idx

print(f"🌾 Valley target: hex {valley_target}  "
      f"elev={elevs[valley_target]:.0f}  "
      f"tier={int(food_tiers[valley_target])}  "
      f"cost={visited[valley_target]:.1f}")

# ── 4. Create pieces at targets ─────────────────────────────────────────
# Knight on the peak (fast scout, holds high ground)
knight = Piece(
    location=peak_target,
    piece_type=PieceType.KNIGHT,
    facing=_facing_away(grid, peak_target, settlement),
    name="Sir Edmund",
    owner_id=1,
    instructions=InstructionList([Instruction.PAUSE.value], patrol=True),
)
FoodProfile.apply(knight)

# Rook in the valley (slow tank, holds the river crossing)
rook = Piece(
    location=valley_target,
    piece_type=PieceType.ROOK,
    facing=_facing_away(grid, valley_target, settlement),
    name="Fort Riverside",
    owner_id=1,
    instructions=InstructionList([Instruction.PAUSE.value], patrol=True),
)
FoodProfile.apply(rook)

# Queen midway between — projecting power
queen_target = planner.find_target(min_elev=80, max_tier=4,
                                    min_cost=2.0, max_cost=max_cost * 0.4)
if queen_target is None:
    queen_target = settlement  # fallback

queen = Piece(
    location=queen_target,
    piece_type=PieceType.QUEEN,
    facing=_facing_toward(grid, queen_target, peak_target),
    name="Her Majesty",
    owner_id=1,
    instructions=InstructionList([Instruction.PAUSE.value], patrol=True),
)
FoodProfile.apply(queen)

print(f"👑 Queen: hex {queen_target}  "
      f"elev={elevs[queen_target]:.0f}  "
      f"cost={visited.get(queen_target, 0):.1f}")

# ── 5. Run the optimizer ────────────────────────────────────────────────
opt = ColonyOptimizer(planner, max_pawns=6, max_bishops=8)
consumers = opt.analyze_consumers([queen, knight, rook])
result = opt.optimize(consumers)
opt._last_result = result  # for __ft__
opt.print_plan(result)

# ── 6. Build the BattlePlan ────────────────────────────────────────────
battle_plan = BattlePlan(
    optimizer=opt,
    assignments=result['assignments'],
    harvest_budget=result['harvest_available'],
)

# ── 7. Show everything ─────────────────────────────────────────────────
print("\n🗺 Rendering battle plan...")

# Supply range + battle plan + piece movements
TerrainDisplay(
    CreamOverlay(),
    #FoodOverlay(color="#8D6E63", n_tiers=6),
    #SupplyOverlay(settlement=settlement, planner=planner, opacity=0.3),
    #BattlePlanOverlay(battle_plan),
    BattlePieceOverlay(battle_plan, num_turns=10, show_pawns=True),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)


In [ ]:
show(myTerr.hexGrid.builder)

It didn't show any pieces

def BattlePieceOverlay(plan: BattlePlan,
                       num_turns: int = 12,
                       show_bishops: bool = True,
                       show_pawns: bool = False,
                       show_consumers: bool = True,
                       opacity: float = 0.85,
                       **kw) -> OverlaySpec:
    """Render piece positions and planned bishop routes from a BattlePlan."""

    def render(ctx):
        grid = ctx.grid
        elevs = ctx.terrain.elevations
        r = grid.radius
        svg = ""
        sp = plan.optimizer.planner

        # ── Consumer markers ────────────────────────────────────
        if show_consumers:
            for a in plan.assignments:
                c = a['consumer']
                if c.hex_idx in plan.vetoed:
                    continue
                if c.hex_idx < 0 or c.hex_idx >= len(grid.hexes):
                    continue

                tc = grid.hexes[c.hex_idx].center
                fed = a['fed']
                color = "#2ecc71" if fed else "#e74c3c"
                name = getattr(c.piece, 'name', '')[:8]
                ptype = c.piece.piece_type.name if hasattr(c.piece, 'piece_type') else '?'
                icon = getattr(c.piece.piece_type, 'icon', '●') if hasattr(c.piece, 'piece_type') else '●'

                # Piece icon
                svg += (f'<text x="{tc.x:.0f}" y="{tc.y + r*0.15:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.7}" '
                        f'fill="{color}" opacity="{opacity}">{icon}</text>\n')

                # Name label
                svg += (f'<text x="{tc.x:.0f}" y="{tc.y + r*0.65:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.28}" '
                        f'fill="#333" font-family="sans-serif" '
                        f'font-weight="bold">{name}</text>\n')

        # ── Bishop routes (as polylines along pathfind hexes) ───
        if show_bishops:
            bishop_colors = ["#3498db", "#2980b9", "#1abc9c",
                             "#16a085", "#8e44ad", "#2c3e50"]
            bi = 0
            for a in plan.assignments:
                c = a['consumer']
                if not a['fed'] or c.hex_idx in plan.vetoed:
                    continue
                if a['bishops'] <= 0:
                    continue

                route = sp.plan_route(c.hex_idx)
                if not route or not route['path']:
                    continue

                for _ in range(a['bishops']):
                    color = bishop_colors[bi % len(bishop_colors)]
                    bi += 1

                    # Outbound path
                    path_pts = []
                    for pidx in route['path'].hexes:
                        if 0 <= pidx < len(grid.hexes):
                            pc = grid.hexes[pidx].center
                            # Slight offset per bishop so parallel routes don't overlap
                            offset = (bi % 3 - 1) * r * 0.15
                            path_pts.append(f"{pc.x + offset:.0f},{pc.y + offset:.0f}")

                    if len(path_pts) >= 2:
                        pts_str = " ".join(path_pts)
                        svg += (f'<polyline points="{pts_str}" '
                                f'fill="none" stroke="{color}" '
                                f'stroke-width="{r * 0.08:.1f}" '
                                f'opacity="{opacity * 0.6}" '
                                f'stroke-dasharray="{r*0.3:.0f},{r*0.15:.0f}" '
                                f'stroke-linecap="round" '
                                f'marker-end="url(#arrow_{color[1:]})"/>\n')

                    # Bishop start marker (small circle at settlement)
                    sc = grid.hexes[sp.settlement].center
                    svg += (f'<circle cx="{sc.x + (bi%3-1)*r*0.3:.0f}" '
                            f'cy="{sc.y + (bi//3)*r*0.3:.0f}" '
                            f'r="{r*0.2}" fill="{color}" '
                            f'opacity="{opacity * 0.8}" '
                            f'stroke="white" stroke-width="1"/>\n')

                    # GIVE marker at destination
                    tc = grid.hexes[c.hex_idx].center
                    svg += (f'<text x="{tc.x + r*0.4:.0f}" '
                            f'y="{tc.y - r*0.3:.0f}" '
                            f'text-anchor="middle" font-size="{r*0.3}" '
                            f'fill="{color}" opacity="{opacity}">🌾</text>\n')

        # ── Pawn cluster at settlement ──────────────────────────
        if show_pawns:
            sc = grid.hexes[sp.settlement].center
            nbs = [sp.settlement] + list(grid.neighborsOf(sp.settlement))
            valid = [n for n in nbs
                     if n not in grid.invalidRegion and elevs[n] > 0]
            for pi in range(min(plan.optimizer.max_pawns, len(valid))):
                pc = grid.hexes[valid[pi]].center
                svg += (f'<text x="{pc.x:.0f}" y="{pc.y + r*0.15:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.5}" '
                        f'fill="#8D6E63" opacity="{opacity * 0.7}">♟</text>\n')
                svg += (f'<text x="{pc.x:.0f}" y="{pc.y + r*0.55:.0f}" '
                        f'text-anchor="middle" font-size="{r*0.2}" '
                        f'fill="#6D4C41" font-family="sans-serif">'
                        f'🌾⇄</text>\n')

        return svg

    return OverlaySpec("battle_pieces", render, priority=55)


In [ ]:
# Check what DiagramGlyphs can do
print([m for m in dir(DiagramGlyphs) if not m.startswith('_')])

# Check if Piece has simulate
print(hasattr(Piece, 'simulate'))

# Check if grid has arrow
print(hasattr(grid, 'arrow'))

# Check helpers
print('_turn_end_labels' in dir())
print('_range_label' in dir())


Thoughts?

Yes this version did draw.

In [ ]:
#| export
def BattlePieceOverlay(plan: BattlePlan,
                       num_turns: int = 12,
                       show_bishops: bool = True,
                       show_pawns: bool = False,
                       show_consumers: bool = True,
                       opacity: float = 0.85,
                       scale: float = 1.8,
                       **kw) -> OverlaySpec:
    """Render piece positions and planned bishop routes from a BattlePlan.
    
    Args:
        scale: Multiplier for all text/glyph sizes. Increase if text is too small.
    """

    def render(ctx):
        grid = ctx.grid
        elevs = ctx.terrain.elevations
        countries = ctx.terrain.fields.get('country')
        r = grid.radius
        s = r * scale        # scaled base unit
        svg = ""
        sp = plan.optimizer.planner

        pieces_to_show = []

        # ── Consumer pieces ─────────────────────────────────────
        if show_consumers:
            for a in plan.assignments:
                c = a['consumer']
                if c.hex_idx in plan.vetoed: continue
                if c.hex_idx < 0 or c.hex_idx >= len(grid.hexes): continue
                if hasattr(c, 'piece') and c.piece is not None:
                    pieces_to_show.append(('consumer', c.piece, a['fed']))

        # ── Phantom bishops from plan ───────────────────────────
        if show_bishops:
            bi = 0
            for a in plan.assignments:
                c = a['consumer']
                if not a['fed'] or c.hex_idx in plan.vetoed: continue
                if a['bishops'] <= 0: continue

                route = sp.plan_route(c.hex_idx)
                if not route or not route['path']: continue

                nbs = list(grid.neighborsOf(sp.settlement))
                valid = [n for n in nbs
                         if n not in grid.invalidRegion and elevs[n] > 0]

                for _ in range(a['bishops']):
                    b_start = valid[bi % len(valid)] if valid else sp.settlement
                    bi += 1
                    b_facing = _facing_toward(grid, b_start, c.hex_idx)

                    path_back = Piece(location=c.hex_idx).pathfind(
                        b_start, grid, elevs)

                    if route['path'] and path_back:
                        rules = (route['path'].to_rules(grid, b_facing)
                                 + [Instruction.GIVE.value]
                                 + path_back.to_rules(grid, b_facing)
                                 + [Instruction.GIVE.value])
                    else:
                        rules = [Instruction.FORWARD.value,
                                 Instruction.GIVE.value]

                    phantom = Piece(
                        location=b_start,
                        piece_type=PieceType.BISHOP,
                        facing=b_facing,
                        instructions=InstructionList(rules, cursor=0, patrol=True),
                        name=f"Bishop {bi}",
                    )
                    FoodProfile.apply(phantom)
                    pieces_to_show.append(('bishop', phantom, True))

        # ── Pawn phantoms at settlement ─────────────────────────
        if show_pawns:
            nbs = [sp.settlement] + list(grid.neighborsOf(sp.settlement))
            valid = [n for n in nbs
                     if n not in grid.invalidRegion and elevs[n] > 0]
            for pi in range(min(plan.optimizer.max_pawns, len(valid))):
                phantom = Piece(
                    location=valid[pi],
                    piece_type=PieceType.PAWN,
                    facing=random.randint(0, 5),
                    instructions=InstructionList(
                        [Instruction.HARVEST.value, Instruction.GIVE.value],
                        cursor=0, patrol=True),
                    name=f"Farmer {pi+1}",
                )
                FoodProfile.apply(phantom)
                pieces_to_show.append(('pawn', phantom, True))

        # ── Render each piece ───────────────────────────────────
        bishop_colors = ["#3498db", "#2980b9", "#1abc9c",
                         "#16a085", "#8e44ad", "#2c3e50"]
        b_idx = 0

        for role, piece, fed in pieces_to_show:
            if piece.location is None or piece.location < 0: continue
            if piece.location >= len(grid.hexes): continue

            # Pick color by role
            if role == 'bishop':
                color = bishop_colors[b_idx % len(bishop_colors)]
                b_idx += 1
            elif role == 'consumer':
                color = "#2ecc71" if fed else "#e74c3c"
            else:
                color = "#8D6E63"

            g = DiagramGlyphs(color=color, size=s * 0.4, opacity=opacity)

            # Start marker
            hc = grid.hexes[piece.location].center
            svg += g.start_marker(hc.x, hc.y, r=s * 0.45)

            # Simulate movement
            steps = piece.simulate(grid, elevs,
                                   num_turns=num_turns,
                                   countries=countries)

            if not steps:
                # Static piece — icon + name only
                name = getattr(piece, 'name', '')[:8]
                icon = getattr(piece.piece_type, 'icon', '●')
                svg += (f'<text x="{hc.x:.0f}" y="{hc.y + s*0.15:.0f}" '
                        f'text-anchor="middle" font-size="{s*0.7:.1f}" '
                        f'fill="{color}" opacity="{opacity}">{icon}</text>\n')
                svg += (f'<text x="{hc.x:.0f}" y="{hc.y + s*0.6:.0f}" '
                        f'text-anchor="middle" font-size="{s*0.28:.1f}" '
                        f'fill="#222" font-family="sans-serif" '
                        f'font-weight="bold">{name}</text>\n')
                continue

            # Draw simulated steps
            prev_idx = piece.location
            for step in steps:
                idx = step.hex_idx
                if idx < 0 or idx >= len(grid.hexes):
                    prev_idx = idx
                    continue

                sc = grid.hexes[idx].center

                if step.instruction == Instruction.FORWARD and not step.blocked:
                    if 0 <= prev_idx < len(grid.hexes) and prev_idx != idx:
                        p1 = grid.hexes[prev_idx].center
                        svg += (f'<line x1="{p1.x:.0f}" y1="{p1.y:.0f}" '
                                f'x2="{sc.x:.0f}" y2="{sc.y:.0f}" '
                                f'stroke="{color}" '
                                f'stroke-width="{s*0.07:.1f}" '
                                f'opacity="{opacity * 0.6}" '
                                f'stroke-dasharray="{s*0.2:.0f},{s*0.1:.0f}" '
                                f'stroke-linecap="round"/>\n')
                elif step.blocked:
                    svg += g.blocked_x(sc.x, sc.y)
                else:
                    svg += g.action_glyph(step.instruction, sc.x, sc.y)

                prev_idx = idx

            # Name label + turn at final position
            if steps:
                last = steps[-1]
                if 0 <= last.hex_idx < len(grid.hexes):
                    fc = grid.hexes[last.hex_idx].center
                    svg += g.step_label(fc.x, fc.y + s * 0.4,
                                        f"T{last.turn}")
                    # Name below turn label
                    name = getattr(piece, 'name', '')[:8]
                    svg += (f'<text x="{fc.x:.0f}" y="{fc.y + s*0.65:.0f}" '
                            f'text-anchor="middle" font-size="{s*0.28:.1f}" '
                            f'fill="#222" font-family="sans-serif" '
                            f'font-weight="bold">{name}</text>\n')

        return svg

    return OverlaySpec("battle_pieces", render, priority=55)


In [ ]:
myTerr.hexGrid.builder.layers = []
TerrainDisplay(
    CreamOverlay(),
    #SupplyOverlay(settlement=settlement, planner=planner, opacity=0.3),
    BattlePlanOverlay(battle_plan),
    BattlePieceOverlay(battle_plan, num_turns=10, show_pawns=True, scale=2.5),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)


fix please

plan_overlay

In [ ]:
??piece.pieces_in_sight